# Chapter 3 — Kinematics

*Fluid Mechanics, 5th edition — Pijush K. Kundu, Ira M. Cohen, David R. Dowling (2012).*
This notebook teaches every new idea of the chapter with plain words, step-by-step mathematics, small worked examples, tested Python from the `fluidpy` package, figures, animations and interactive explainers. It does not reproduce the book's text or figures; equations are cited by their numbers so you can follow along in your copy.

---

**The big idea in one paragraph**

Kinematics describes motion without asking what causes it. A fluid can be described by following each particle
(Lagrangian) or by watching fixed points (Eulerian); the material derivative
$\frac{DF}{Dt}=\frac{\partial F}{\partial t}+\mathbf u\cdot\nabla F$ *(Eq. 3.5)* is the bridge. Three kinds of flow line
picture what the flow does, and they differ whenever it is unsteady. Near any point the motion splits into a
deformation (the strain-rate tensor $\mathbf S$: stretching, shearing, swelling) and a rigid spin at half the vorticity.
The chapter ends with the Reynolds transport theorem, the rule for differentiating an integral over a moving volume —
the tool Chapter 4 uses to write every conservation law.

**Road map** — the new ideas of this chapter, in the order they build on each other:

1. §3.1 steady vs unsteady, 1-/2-/3-D flows, cylindrical and spherical coordinates
2. §3.2 Lagrangian and Eulerian descriptions (C01); the material derivative D/Dt (C02)
3. §3.3 streamlines (C03), path lines and streak lines (C04); the acceleration is the same for every steadily moving observer (C05)
4. §3.4 relative motion near a point (C06): stretching (C07), shearing (C08), volume change (C09), spin = ½ vorticity (C10), deformation + rotation (C11), principal strain axes (C12)
5. §3.5 shear flow, solid-body rotation and the irrotational vortex (C13); Rankine and Gaussian vortices (C14)
6. §3.6 Leibniz's rule and the Reynolds transport theorem (C15)

**What you should already know** (each is recapped where it is used):

- partial derivatives, the chain rule and first-order Taylor expansion (Ch. 1 primers P25, P49, P26)
- index notation, the velocity gradient G, the split G = S + ½R and the vorticity ω = ∇×u (Ch. 2 §2.1, §2.9–2.10)
- eigenvalues and principal axes of a symmetric tensor (Ch. 2 §2.11, P80)
- Gauss' and Stokes' theorems, circulation (Ch. 2 §2.12–2.13)
- scipy.integrate.solve_ivp and scipy.linalg.expm (Ch. 1 P31, Ch. 2 P79)

## 🎮 Interactive explainers in this chapter

| # | Explainer | What it makes clear |
|---|---|---|
| 1 | **Three lines through one point — why do they disagree?** | C03 C04: streamline, path line and streak line in an unsteady flow (Ex. 3.1) |
| 2 | **Why does the station warm while the air does not?** | C01 C02: DF/Dt = ∂F/∂t + u·∇F with a fixed probe and a drifting float |
| 3 | **Steady or not — does the acceleration care?** | C05: the local/advective split moves with the observer, the total does not |
| 4 | **What does each number in S measure?** | C06–C09: stretching, shearing and swelling measured on a moving element |
| 5 | **Can a straight flow make a fluid element spin?** | C10–C12: spin = ½ω for any pair of lines, du = S·dx + ½ω×dx, principal axes |
| 6 | **Going round in circles ≠ spinning** | C13 C14: solid-body, irrotational, Rankine and Gaussian vortices |
| 7 | **What changes inside a moving box?** | C15: Leibniz and the Reynolds transport theorem as a budget |

Each one opens right where its idea is taught and fills the window (no scrolling). Start with the **Walkthrough**, then **Explore** with the sliders and presets, read **Explain** (every number worked out with your settings, and what it means), step through the **Derivation** where there is one, read the **Equations** and the **Code**, and finish with **Check yourself**. On the web page they are full-window; in Colab and Jupyter they appear in the cell output (use ⤢ *Full screen* or *Open in new tab* for more room).

## ⚙️ Setup — run this cell first

It works in three places without changes:
* **Google Colab** — clones the public repository into `/content/fluidpy` (the `fluidpy` package, its tests and the
  interactive explainers) and installs the one package Colab lacks (`pint`). No GPU is needed.
* **Jupyter / VS Code on your computer** — finds the repository root (the folder with `book.yaml`) and imports from it.
* **The published web page** — already executed; nothing to run.

In [ ]:
# ── Setup: find (or fetch) the fluidpy repository, then load the house style ─────────────────────────────
import os, sys, subprocess, pathlib                     # standard library only, so this cell never fails on imports

IN_COLAB = "google.colab" in sys.modules                 # True when this notebook runs inside Google Colab
REPO_URL = "https://github.com/shammun/fluidpy.git"               # the public repository with the fluidpy package

if IN_COLAB:
    ROOT = pathlib.Path("/content/fluidpy")              # where the repository is cloned on Colab
    if not ROOT.exists():                                 # first run in this Colab session: clone it
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    else:                                                 # later runs: fetch the newest version
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)
else:
    ROOT = pathlib.Path.cwd().resolve()                   # start where Jupyter was launched …
    while not (ROOT / "book.yaml").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent                                # … and walk up until we find the repository root

os.chdir(ROOT)                                            # relative paths (outputs/, viz/) now resolve from the root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))                         # make `import fluidpy` work

from fluidpy.core.style import setup_notebook            # matplotlib + plotly style shared with the explainers
from fluidpy.core.embed import show_viz                  # shows an interactive explainer in the notebook
from fluidpy.core.anim import show_animation             # plays a matplotlib animation inline
FAST = setup_notebook()                                   # FAST=True (env FLUIDPY_FAST=1) shrinks grids for quick runs
print(f"IN_COLAB = {IN_COLAB} | repository root: {ROOT} | FAST = {FAST}")

In [ ]:
import numpy as np                                      # arrays (Ch. 1 primer P03)
import sympy as sp                                      # symbolic algebra (Ch. 1 primer P40)
import matplotlib.pyplot as plt                         # static figures (Ch. 1 primer P01)
import plotly.graph_objects as go                       # rotatable 3-D and slider figures (Ch. 1 primer P41)
from fluidpy import ch03_kinematics as ch03             # the tested chapter-3 module: every function cites its § and Eq.
from fluidpy.core.interact import slider_figure         # plotly figure with a slider that works on the web page (P17)
from fluidpy.core.anim import animate                   # matplotlib animations (P16); show_animation came with the setup
from fluidpy.core.style import COLORS, savefig          # the house palette and a helper that saves PNGs to outputs/ch03
from tools.convergence import observed_order            # slope of log(error) vs log(step): the observed order (P13)
import logging                                          # standard library: controls library log messages
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # hide harmless font-substitution notes


def recolor(fig, colors, dashes=None):                  # give plotly slider traces the notebook's colours by trace name
    for tr in fig.data:                                 # every trace of every slider step
        if tr.name in colors:                           # a name we assigned a colour to
            tr.line.color = colors[tr.name]             # same colour meaning as in the matplotlib figures
            tr.marker.color = colors[tr.name]           # markers too (for "markers" traces)
        if dashes and tr.name in dashes:                # optional dash pattern ("dash", "dot")
            tr.line.dash = dashes[tr.name]
    return fig                                          # the same figure, restyled


print(len([n for n in dir(ch03) if not n.startswith("_")]), "public names in fluidpy.ch03_kinematics")   # the toolbox this notebook calls

**What does the code above do?**

1. Numerical, symbolic and plotting libraries (all primed in Ch. 1).
2. `ch03` is the chapter module; it re-exports the Ch. 3 primitives in `fluidpy/core/` (kinematics, coordinates,
   vortices, transport) and the Ch. 2 tensor tools, so one name covers the whole chapter. Every function is tested in
   `tests/test_ch03.py`.
3. The house helpers for slider figures, animations, colours and the observed order of convergence;
   `recolor` only restyles plotly traces so that a colour always means the same thing (below).

**Notation and colours used in this notebook.**

| Symbol | Meaning | Unit |
|---|---|---|
| $\mathbf x=(x_1,x_2,x_3)=(x,y,z)$ | position | m |
| $t$ | time | s |
| $\mathbf u=(u_1,u_2,u_3)=(u,v,w)$ | velocity | m/s |
| $F$ | any field (a temperature, a velocity component …) | its own |
| $G_{ij}=\partial u_i/\partial x_j$ | velocity gradient (row = velocity component) | 1/s |
| $\mathbf S$, $\mathbf R=\mathbf G-\mathbf G^{\rm T}$ | strain-rate tensor; the book's rotation tensor (no ½) | 1/s |
| $\boldsymbol\omega=\nabla\times\mathbf u$ | vorticity; **the spin of a fluid element is ½ω** | 1/s |
| $\gamma=du_1/dx_2$ | shear rate ($=2S_{12}$; Ch. 2's Γ was $S_{12}$ itself) | 1/s |
| $\Gamma$ | circulation | m²/s |
| $\theta$ | plane polar angle (§3.5) · polar angle from $+z$ (spherical) · cone half-angle (Ex. 3.2) | rad |

**Colours** (one meaning each, also in the explainers): streamline **teal** · path line **orange** · streak line
**rose** · local rate $\partial/\partial t$ **blue** · advective rate **amber** · total $D/Dt$ **purple** · strain part
**teal** · rotation part **orange** · stretching **blue** / compressing **rose** · volume term **blue**, surface term
**orange**.

**Where this chapter is used later**

| Result here | Used in |
|---|---|
| $\frac{DF}{Dt}=\frac{\partial F}{\partial t}+\mathbf u\cdot\nabla F$ (3.5) | every conservation law of Ch. 4, the vorticity equation (Ch. 5), potential vorticity (Ch. 13) |
| path lines $d\mathbf r/dt=\mathbf u(\mathbf r,t)$ (3.8) | particle orbits under waves (Ch. 7), Lagrangian statistics (Ch. 12), trajectories (Ch. 13) |
| $du_i=S_{ij}dx_j+\tfrac12(\boldsymbol\omega\times d\mathbf x)_i$ (3.19) | the Newtonian stress law (Ch. 4 §4.5), vortex stretching (Ch. 5) |
| circulation $\Gamma=\oint\mathbf u\cdot d\mathbf s$ (3.18) | Kelvin's theorem (Ch. 5), lift (Ch. 6, 14) |
| $\omega'_z=\omega_z-2\Omega$ | absolute vs relative vorticity $f+\zeta$ (Ch. 4 §4.7, Ch. 13) |
| $\frac{d}{dt}\int_{V^*}F\,dV=\int_{V^*}\frac{\partial F}{\partial t}dV+\int_{A^*}F\,\mathbf b\cdot\mathbf n\,dA$ (3.35) | mass, momentum and energy equations (Ch. 4 §4.2–4.4), layer budgets (Ch. 13) |

*Climate hook:* the advective term $\mathbf u\cdot\nabla T$ is the "warm/cold advection" shaded on every weather map.

🔁 **Tools from earlier chapters used here without a new primer** (see `knowledge/primers.md`): f-strings `f"{x:.3f}"`
(Ch. 1 P04) · `lambda` and functions passed as arguments (P29) · `assert np.allclose(a, b)` — "the two agree to
rounding" (P15) · tuple unpacking `u, a = f(...)` (P14) · `show_viz` embeds an explainer (P18) · `animate` /
`show_animation` (P16) and `slider_figure` (P17) · the ordinary derivative as a slope (P19) and the definite integral
(P27) · `np.trapezoid` (P37) · matrix–vector products `G @ dx` (Ch. 2 P63) · `np.linalg.norm` (P67) · the quadratic form
$\mathbf n\cdot\mathbf A\cdot\mathbf n$ (P82; it vanishes for an antisymmetric A because $n_in_jA_{ij}=-n_jn_iA_{ji}$) ·
eigenvalues and eigenvectors (P80) · `np.arctan2` (P70) · midpoint sums for volume and surface integrals (P83).

**Small idioms, glossed once:** `zip(a, b)` walks two lists in step · `enumerate(a)` gives (index, item) pairs · `np.hypot(x, y)` $=\sqrt{x^2+y^2}$ · `np.ptp(a)` = max − min ("peak to peak") · `np.r_[a, b]` joins arrays end to end, `np.full(n, v)` is n copies of v · `np.ma.masked_invalid(a)` hides NaN entries from a plot · a result such as `ch03.acceleration(...)` is a *named tuple*: unpack it (`a, loc, adv = …`) or read a field (`res.local`) · `plt.contour(...).allsegs` returns the contour lines as lists of points · `sp.Function('f')` is an unknown function, `sp.Lambda(args, expr)` a function built from an expression · `sp.series(e, x, 0, n)` a Taylor series, `sp.limit(e, x, a)` a limit.

---

## 3.1 Introduction and Coordinate Systems

**What is this section about?** The words used to describe *any* flow before we ask what drives it: steady or
unsteady, one-, two- or three-dimensional, and the coordinate systems whose velocity components later chapters use
(Cartesian, plane polar, cylindrical, spherical).

📝 **Note.** **Kinematics** `N01` — describes how a fluid moves — positions, velocities, accelerations, stretching and spinning — without asking which
forces cause it. Forces enter in Ch. 4 (§4.4 onward, Newton's second law for a fluid).

📝 **Note.** **Steady and unsteady** `N02` — A flow is **steady** when nothing measured at a fixed point changes with time: $\partial(\cdot)/\partial t=0$ for every
field. Otherwise it is **unsteady**. Careful: steadiness depends on the observer — the air flowing past a parked car
seen from the car is steady; the same kind of flow seen from the pavement as a car drives by is not. C05 turns this into
a theorem.

$$ \frac{\partial(\cdot)}{\partial t}=0\quad\text{(steady)}  $$

> 📎 **Primer — scipy.integrate.quad and dblquad.** `P87` · Adaptive numerical integration: `quad(f, a, b)` returns $\int_a^b f\,dx$ and an error estimate; `dblquad(f, a, b,
gfun, hfun)` does a double integral, the **inner variable first** in `f`'s arguments (its limits may depend on the outer
one). We use them for section averages here, for the cone of Ex. 3.2 and for exact checks of the Reynolds transport
theorem in §3.6.

In [ ]:
from scipy.integrate import quad, dblquad                     # adaptive 1-D and 2-D quadrature
val, err = quad(lambda r: 2*r, 0.0, 1.0)                      # ∫₀¹ 2r dr = 1
print(val, err)                                               # 1.0 and a tiny error estimate
area, _ = dblquad(lambda r, th: r, 0.0, 2*np.pi, 0.0, 1.0)    # ∫∫ r dr dθ over the unit disc (inner variable r first)
print(area, np.pi)                                            # 3.14159… twice: the disc's area

📝 **Note.** **1-D, 2-D and 3-D flows** `N03` — A flow is 3-D when it depends on all three coordinates, plane (2-D) when one coordinate can be dropped, and — as an
engineering approximation — 1-D when we keep only the average over each cross-section of a pipe or channel (an
axisymmetric pipe flow is still 3-D in the book's sense). The 1-D description keeps the average $\bar u$ over the
section area $A$ and throws away the profile. Number: the parabolic (Poiseuille) profile $u=U(1-r^2/R^2)$ averages to
exactly $U/2$.

$$ \bar u(z)=\frac{1}{A}\int_A u\,dA  $$

In [ ]:
U, R = 2.0, 0.05                                     # centre speed 2 m/s, pipe radius 5 cm
u_pois = lambda r, z: U*(1 - (r/R)**2)               # parabolic profile u(r, z) [m/s] (the same at every z here)
ubar = ch03.cross_section_average(u_pois, R)         # (1/πR²) ∫₀ᴿ u 2πr dr by quad: rings of area 2πr dr
print(f"section average = {ubar:.4f} m/s (U/2 = {U/2})")
r = np.linspace(0, R, 2001)                          # 2001 radii from the axis to the wall [m]
mine = np.trapezoid(u_pois(r, 0)*2*np.pi*r, r)/(np.pi*R**2)   # the same average by the trapezoid rule (Ch. 1 P37)
assert np.allclose(mine, ubar, rtol=1e-6)            # same number → the library does exactly this

**What does the code above do?**

1. `u_pois` is the profile as a function of radius r and axial position z.
2. `cross_section_average` integrates $u\cdot2\pi r\,dr$ (a thin ring of radius r has area $2\pi r\,dr$) with `quad` and
   divides by the area $\pi R^2$.
3. The trapezoid version on 2001 rings agrees — the from-scratch check of this section.

Our model of a developing pipe flow (not a solution of the equations of motion — Ch. 8 has the real one):
$u=U_m\big(1-(r/R)^{n}\big)$ with $n(z)=2+8e^{-z/L_e}$ ($L_e=0.5$ m: flat at the inlet, a parabola far downstream) and
$U_m=\bar u\,(n+2)/n$, which keeps the section average $\bar u$ the same at every z.

In [ ]:
R, Ub = 1.0, 1.0                                     # pipe radius 1 m, section-average speed 1 m/s
r = np.linspace(-R, R, 201)                          # across the pipe, wall to wall [m]
zs = [0.0, 0.5, 2.0]                                 # three stations downstream of the inlet [m]
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.8))  # left: profiles (2-D view); right: averages (1-D view)
for z, al in zip(zs, (0.45, 0.7, 1.0)):              # darker teal = further downstream
    a.plot(ch03.pipe_profile(r, z, Ub, R, n0=10.0, L_e=0.5), r, color=COLORS["teal"], alpha=al, lw=2.2,
           label=f"$z$ = {z} m")                     # our developing profile: flat at the inlet → parabola downstream
a.set_xlabel("$u$ [m/s]"); a.set_ylabel("$r$ [m]"); a.legend(fontsize=8)   # axes with units
a.set_title("2-D view: the profile changes along the pipe")
z = np.linspace(0, 3, 61)                            # stations along the pipe [m]
ubar = [ch03.cross_section_average(lambda rr, zz: ch03.pipe_profile(rr, zz, Ub, R, 10.0, 0.5), R, zz) for zz in z]   # ū(z) by quad
n = 2 + (10 - 2)*np.exp(-z/0.5)                      # the profile's exponent n(z) = 2 + 8 e^{−z/L_e}
b.plot(z, ubar, color=COLORS["accent"], lw=2.5, label=r"$\bar u(z)$ — the whole 1-D description")   # constant: 1 m/s
b.plot(z, Ub*(n + 2)/n, "--", color=COLORS["muted"], label="centre speed $U_m(z)$ (lost in 1-D)")  # what 1-D forgets
b.set_ylim(0, 2.2); b.set_xlabel("$z$ [m]"); b.set_ylabel("speed [m/s]"); b.legend(fontsize=8)     # fixed axes
b.set_title("1-D view: one number per section")
savefig(fig, "ch03", "pipe_profiles"); plt.show()    # save to outputs/ch03 and draw

**What you see.** Three teal profiles that sharpen downstream (left) and one flat purple line with a dashed curve above it (right).

**How to read it.** The 1-D model sees only the right panel: the same average at every z while the shape changes completely.
Mass is conserved (the average does not change); the detail — where the fluid is fast — is thrown away. The dashed
centre speed rises from ≈ 1.2 to 2 m/s, a change the 1-D description cannot see.

**What would change if…** …the pipe narrowed to half the radius: ū would jump ×4 (the same volume per second through a quarter of the
area); what the profile looks like would still be an open question for the 1-D model.

Fig. 3.2's two views of one cylinder — held fixed in a stream, or towed through still water — are the same flow seen by
two observers; we take them up with the Galilean transformation in C05 (§3.3), where note `N04` treats them.

> 📎 **Primer — cylindrical and spherical unit vectors.** `P88` · At every point P the curvilinear coordinates carry their own right-handed unit vectors, and they turn as P moves.
Cylindrical $(R,\varphi,z)$: $\mathbf e_R=(\cos\varphi,\sin\varphi,0)$, $\mathbf e_\varphi=(-\sin\varphi,\cos\varphi,0)$,
$\mathbf e_z=(0,0,1)$. Spherical $(r,\theta,\varphi)$ with θ measured from $+z$:
$\mathbf e_r=(\sin\theta\cos\varphi,\sin\theta\sin\varphi,\cos\theta)$,
$\mathbf e_\theta=(\cos\theta\cos\varphi,\cos\theta\sin\varphi,-\sin\theta)$, $\mathbf e_\varphi=(-\sin\varphi,\cos\varphi,0)$.
A velocity component is the dot product of $\mathbf u$ with one of them — a projection on an orthonormal basis (Ch. 2
primer P65).

In [ ]:
phi = np.pi/4                                                    # azimuth φ = 45°
e_R = np.array([np.cos(phi), np.sin(phi), 0.0])                  # points away from the z-axis
e_phi = np.array([-np.sin(phi), np.cos(phi), 0.0])               # points round the z-axis
print(e_R @ e_phi, np.cross(e_R, e_phi))                         # 0.0 and (0, 0, 1): perpendicular, e_R × e_φ = e_z

📝 **Note.** **Coordinate systems (Fig. 3.3)** `N05` — Plane: $(x,y)=(x_1,x_2)$ or polar $(r,\theta)$ (Ch. 2 Ex. 2.1). Cylindrical $(R,\varphi,z)$ with velocity components
$(u_R,u_\varphi,u_z)$; spherical $(r,\theta,\varphi)$, θ the angle from $+z$, φ the azimuth, components
$(u_r,u_\theta,u_\varphi)$. ⚠️ The same letter θ means the plane polar angle in §3.5 but the angle from the $z$-axis in
spherical coordinates. The conversions (the book states them in its exercises):

$$ R=\sqrt{x^2+y^2},\quad \varphi=\tan^{-1}(y/x);\qquad r=\sqrt{x^2+y^2+z^2},\quad \theta=\tan^{-1}\!\big(\sqrt{x^2+y^2}/z\big)  $$

In [ ]:
P = (1.0, 1.0, 1.0)                                              # a point P [m]
print(np.round(ch03.cylindrical_from_cartesian(*P), 4))          # (R, φ, z) = (1.4142, 0.7854, 1.0)
print(np.round(ch03.spherical_from_cartesian(*P), 4))            # (r, θ, φ) = (1.7321, 0.9553, 0.7854); θ = 54.74°
u = np.array([1.0, 0.0, 0.0])                                    # a 1 m/s wind along x at P
uc = ch03.velocity_components(u, np.array(P), "cylindrical")     # (u_R, u_φ, u_z): projections on e_R, e_φ, e_z
us = ch03.velocity_components(u, np.array(P), "spherical")       # (u_r, u_θ, u_φ): projections on e_r, e_θ, e_φ
print(np.round(uc, 4), np.round(us, 4))                          # (0.7071, −0.7071, 0) and (0.5774, 0.4082, −0.7071)
print(np.sum(uc**2), np.sum(us**2))                              # 1.0 twice: a projection on an orthonormal basis keeps |u|²
print(np.round(ch03.unit_vectors_spherical(*ch03.spherical_from_cartesian(*P)[1:]), 4))   # rows e_r, e_θ, e_φ at P

**What does the code above do?**

1. Position P in the two curvilinear systems (angles in radians).
2. The same wind vector projected on the local unit vectors of each system.
3. The squares add to $|\mathbf u|^2=1$ in every system — the three unit vectors are perpendicular and of length one.

In [ ]:
P = np.array([1.0, 1.0, 1.0])                                    # the point P [m]
_, th, ph = ch03.spherical_from_cartesian(*P)                    # its spherical angles
triads = {"Cartesian": (np.eye(3), COLORS["muted"]),             # rows e_x, e_y, e_z — the same everywhere
          "cylindrical": (ch03.unit_vectors_cylindrical(ph), COLORS["teal"]),   # rows e_R, e_φ, e_z at P
          "spherical": (ch03.unit_vectors_spherical(th, ph), COLORS["orange"])} # rows e_r, e_θ, e_φ at P
L = 0.6                                                          # drawn arrow length [m]
fig = go.Figure()                                                # an empty 3-D plotly figure
groups = []                                                      # which traces belong to which triad (for the menu)
for name, (E3, col) in triads.items():                           # one triad at a time
    xs, ys, zs = [], [], []                                      # shaft coordinates, NaN-separated
    for e in E3:                                                 # one shaft per unit vector, all in one trace
        xs += [P[0], P[0] + L*e[0], None]; ys += [P[1], P[1] + L*e[1], None]; zs += [P[2], P[2] + L*e[2], None]
    fig.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode="lines", line=dict(color=col, width=6), name=name))   # shafts
    tips = P + L*E3                                              # arrow heads at the tips (plotly cones: Ch. 2 P64)
    fig.add_trace(go.Cone(x=tips[:, 0], y=tips[:, 1], z=tips[:, 2], u=E3[:, 0], v=E3[:, 1], w=E3[:, 2],
                          anchor="tip", sizemode="absolute", sizeref=0.15, showscale=False,
                          colorscale=[[0, col], [1, col]], name=name))   # one-colour cones
    groups += [name, name]                                       # shafts and heads belong to this triad
fig.add_trace(go.Scatter3d(x=[0, P[0], P[0], P[0]], y=[0, 0, P[1], P[1]], z=[0, 0, 0, P[2]], mode="lines",
                           line=dict(color=COLORS["grid"], dash="dash"), name="guide to the axes"))   # origin → P
fig.add_trace(go.Scatter3d(x=[P[0]], y=[P[1]], z=[P[2]], mode="markers", marker=dict(size=5, color=COLORS["ink"]), name="P"))
groups += ["always", "always"]                                   # guide and P are always shown
buttons = [dict(label=lab, method="update",
                args=[{"visible": [g == "always" or lab == "all" or g == lab for g in groups]}])
           for lab in ("all", "Cartesian", "cylindrical", "spherical")]   # a dropdown to show one triad at a time
fig.update_layout(height=460, title="Unit vectors at P = (1, 1, 1) m: Cartesian (grey), cylindrical (teal), spherical (orange)",
                  updatemenus=[dict(buttons=buttons, x=0, y=1.08, xanchor="left")],
                  scene=dict(aspectmode="cube", xaxis_title="x [m]", yaxis_title="y [m]", zaxis_title="z [m]"))   # equal axes
fig.show()                                                       # draw it (rotatable)

**What does the code above do?**

1. `unit_vectors_cylindrical` and `unit_vectors_spherical` return the local triads at P as rows.
2. Each triad is drawn as three shafts (one trace, NaN-separated) plus three cone heads; the dashed grey line walks from
   the origin to P along the axes.
3. The dropdown switches the triads on and off.

**What you see.** Three triads of arrows at one point, one grey, one teal, one orange.

**How to read it.** The Cartesian triad is the same everywhere; the teal and orange triads are tied to P: $\mathbf e_R$ points away
from the $z$-axis, $\mathbf e_r$ away from the origin, and $\mathbf e_\varphi$ is shared by both.

**What would change if…** …P moved to (−1, 1, 1) (edit `P` in the cell): $\mathbf e_R$ and $\mathbf e_\varphi$ swing by 90° while the grey
arrows stay put — this is why derivatives of curvilinear components need extra terms (note `N06`).

📝 **Note.** **Curvilinear operators** `N06` — Appendix B of the book lists ∇, ∇², ∇·u and (u·∇)u in cylindrical and spherical coordinates; we meet the first of
them in C13 (the vorticity in polar coordinates,
$\omega_z=\frac1r\frac{\partial}{\partial r}(ru_\theta)-\frac1r\frac{\partial u_r}{\partial\theta}$ *(Eq. 3.23)*, derived
there by hand) and the rest with Ch. 4's Navier–Stokes equations.

---

## 3.2 Particle and Field Descriptions of Fluid Motion

**What is this section about?** Two ways to describe one flow — ride along with every particle, or stand still and
watch every point — and the single formula, the material derivative, that turns one into the other.

### 🧩 Following a particle or watching a point — and the bridge between them (3.2) `C01`

*The question:* A float drifts down a river past a thermometer bolted to a bridge pier. Both report the temperature of the same water.
How do the two sets of numbers fit together?

*In one line:* $F[\mathbf r(t;\mathbf r_o,t_o),t]=F(\mathbf x,t)\ \text{at}\ \mathbf x=\mathbf r(t;\mathbf r_o,t_o)$ *(3.2)*

#### The problem in plain words

Oceanographers throw Argo floats into the sea and read where each one goes (it *is* a water parcel); weather services
keep thermometers at fixed stations; numerical models store fields on a fixed grid. We need to translate freely between
"what happens to this parcel" and "what happens at this place" — fluid dynamics is written in the second language, but
Newton's law is about the first.

#### The idea

```
Lagrangian (follow the particle)            Eulerian (watch the point)
label r_o at time t_o  →  r(t; r_o, t_o)    field F(x, t): four independent variables x, y, z, t
"what does THIS parcel do?"                 "what happens HERE?"
         the bridge:  F[r(t; r_o, t_o), t] = F(x, t)   when   x = r(t; r_o, t_o)      (3.2)
```

In symbols: $F[\mathbf r(t;\mathbf r_o,t_o),t]=F(\mathbf x,t)$ when $\mathbf x=\mathbf r(t;\mathbf r_o,t_o)$ *(3.2)*.

**A label is not a variable**: $\mathbf r_o$ only names the particle (where it was at $t_o$), just as a float's serial
number does. The bridge says: at time t, the field at $\mathbf x$ *is* whatever the particle that happens to be at
$\mathbf x$ carries.

> 📎 **Primer — functions of time with parameters.** `P89` · A trajectory $x=Xe^{\alpha t}$ is a function of time t alone once the label X is chosen; X is a *parameter* that picks
one curve out of a family. Differentiating "with the label held fixed" means following one member of the family. Here
X is where the particle was at t = 0.

In [ ]:
alpha = 0.5                                          # stretching rate α [1/s]
x_of = lambda t, X: X*np.exp(alpha*t)                # one curve per label X [m]
print([round(float(x_of(1.0, X)), 3) for X in (1.0, 2.0)])   # [1.649, 3.297]: two particles at t = 1 s

📝 **Note.** **Lagrangian description** `N07` — Each particle is labelled by its position $\mathbf r_o$ at a reference time $t_o$; its later position is
$\mathbf r(t;\mathbf r_o,t_o)$ (Fig. 3.4). Any property it carries is written $F[\mathbf r(t;\mathbf r_o,t_o),t]$. The
labels are fixed numbers for a given particle, not coordinates.

$$ \mathbf r=\mathbf r(t;\mathbf r_o,t_o)  $$

📝 **Note.** **Velocity and acceleration of a particle** `N08` — are ordinary time derivatives along its own path — exactly single-particle mechanics. Number (the running example):
$x=Xe^{\alpha t}$ with X = 2 m, α = 0.5 s⁻¹, t = 1 s gives $u=\alpha Xe^{\alpha t}=1.649$ m/s and
$a=\alpha^2Xe^{\alpha t}=0.824$ m/s².

$$ \mathbf u=d\mathbf r(t;\mathbf r_o,t_o)/dt\quad\text{and}\quad\mathbf a=d^2\mathbf r(t;\mathbf r_o,t_o)/dt^2 \qquad \text{(3.1)} $$

#### The maths — the Eulerian description and the bridge

Watching fixed points, a property is a field $F(\mathbf x,t)$ of four independent variables. The two descriptions must
agree when the particle position and the field point coincide, in the same coordinates and on a common clock:

$$F[\mathbf r(t;\mathbf r_o,t_o),t]=F(\mathbf x,t)\quad\text{when}\quad\mathbf x=\mathbf r(t;\mathbf r_o,t_o).\qquad(3.2)$$

Read it as a recipe: to get the Eulerian field at $(\mathbf x,t)$, find the label of the particle that sits at
$\mathbf x$ at time t, then read that particle's value. The derivation below does this for a concrete flow.

> 📎 **Primer — inverse functions and sympy solve.** `P90` · To use $F[\mathbf r(t;\mathbf r_o,t_o),t]=F(\mathbf x,t)$ at $\mathbf x=\mathbf r$ *(Eq. 3.2)* we must answer "which
particle is at x at time t?" — that is, solve $x=r(t;X)$ for the label X. `sympy.solve(equation, unknown)` does it
symbolically; it returns a list of solutions.

In [ ]:
X, x, t, alpha = sp.symbols('X x t alpha', positive=True)   # label, position, time, rate (all > 0)
sp.solve(sp.Eq(x, X*sp.exp(alpha*t)), X)                    # [x*exp(-alpha*t)]: the label of the particle now at x

#### 🧮 Derivation — The Eulerian velocity of a Lagrangian map: x = X e^{αt} ⇒ u = αx `D01`

**What we want to show.** We know where every particle is at every time (a Lagrangian map). Find the velocity *field* a fixed probe would record, and the field of accelerations — the concrete version of the bridge (3.2), $F[\mathbf r(t;\mathbf r_o,t_o),t]=F(\mathbf x,t)\ \text{at}\ \mathbf x=\mathbf r(t;\mathbf r_o,t_o)$ that the book never works out.

**Assumptions.** The map is invertible — two particles never occupy the same point (step 3); r is twice differentiable in t (steps 1–2).

**The plan.**

1. Differentiate the path with the label held fixed, (3.1), $\mathbf u=d\mathbf r/dt,\ \mathbf a=d^2\mathbf r/dt^2$.
2. Ask which particle is at x at time t (invert the map).
3. Substitute that label: the bridge (3.2), $F[\mathbf r(t;\mathbf r_o,t_o),t]=F(\mathbf x,t)\ \text{at}\ \mathbf x=\mathbf r(t;\mathbf r_o,t_o)$.
4. Do the same for the acceleration.

**Tools we use** (each explained before this point): (3.1), $u=dr/dt$, $a=d^2r/dt^2$ (N08) · (3.2), $F[\mathbf r(t;\mathbf r_o,t_o),t]=F(\mathbf x,t)$ at $\mathbf x=\mathbf r$ (C01) · functions of time with parameters (primer in C01) · inverse functions and sympy solve (primer in C01) · exponent rules (ch01 P43).

**We start from**

$$ x=r(t;X)=X\,e^{\alpha t} $$

*In words:* the particle labelled X (its position at t = 0) moves away from the origin at a rate proportional to its distance; α [1/s] is a constant stretching rate.

---

**Step 1 of 6 — Differentiate the path, label fixed.**

$$ u=\left.\dfrac{dx}{dt}\right|_X=\alpha X e^{\alpha t} $$

- *Why we can do this:* (3.1), $\mathbf u=d\mathbf r/dt$: a particle's velocity is the time derivative of its own path; X names the particle, so it is held constant.
- *In words:* The particle labelled X speeds up exponentially.

---

**Step 2 of 6 — Differentiate once more.**

$$ a=\left.\dfrac{d^2x}{dt^2}\right|_X=\alpha^2Xe^{\alpha t} $$

- *Why we can do this:* (3.1), $\mathbf a=d^2\mathbf r/dt^2$; differentiating $e^{\alpha t}$ brings down another factor α (chain rule for the exponential).
- *In words:* Its acceleration also grows exponentially.

---

**Step 3 of 6 — Invert the map for the label.**

$$ X=x\,e^{-\alpha t} $$

- *Why we can do this:* To use (3.2), $F[\mathbf r(t;\mathbf r_o,t_o),t]=F(\mathbf x,t)\ \text{at}\ \mathbf x=\mathbf r(t;\mathbf r_o,t_o)$ we need the label of the particle that sits at x at time t; divide $x=Xe^{\alpha t}$ by $e^{\alpha t}\neq0$ — one answer, so the map is invertible.
- *In words:* The particle now at x started at $xe^{-\alpha t}$.

---

**Step 4 of 6 — Substitute the label into u.**

$$ u(x,t)=\alpha\,(xe^{-\alpha t})\,e^{\alpha t} $$

- *Why we can do this:* The bridge (3.2), $F[\mathbf r(t;\mathbf r_o,t_o),t]=F(\mathbf x,t)$ when $\mathbf x=\mathbf r$: the field at x is the velocity carried by the particle that is there now.
- *In words:* The Eulerian velocity written with the particle's label replaced by its position.

---

**Step 5 of 6 — Simplify the exponentials.**

$$ u(x,t)=\alpha x $$

- *Why we can do this:* $e^{-\alpha t}e^{\alpha t}=e^0=1$ (exponent rules, P43); t has disappeared — the field does not depend on time.
- *In words:* A steady field: every point always has velocity α times its distance.

---

**Step 6 of 6 — Same substitution for a.**

$$ a(x,t)=\alpha^2x $$

- *Why we can do this:* Apply (3.2), $F[\mathbf r(t;\mathbf r_o,t_o),t]=F(\mathbf x,t)\ \text{at}\ \mathbf x=\mathbf r(t;\mathbf r_o,t_o)$ to the acceleration of step 2 with the label of step 3; the exponentials cancel as in step 5.
- *In words:* The acceleration field is also steady and never zero away from the origin.

---

**Result**

$$ u(x,t)=\alpha x\ \text{,}\ a(x,t)=\alpha^2x $$

*In words:* the Eulerian velocity field is steady even though every particle accelerates.

**What it means.** "Steady" is about points, "accelerating" about particles: a steady field can accelerate every particle. The missing link — how to get a from the Eulerian u without knowing the paths — is the material derivative (D02). The recipe fails if the map is not invertible (particles colliding, a shock).

**Check it.** Units: α [1/s] × x [m] = m/s ✓; α² x = m/s² ✓. Limit α = 0: nobody moves, u = a = 0 ✓. Numbers (X = 2 m, α = 0.5 s⁻¹, t = 1 s): x = 3.297 m, u = 1.649 m/s = αx ✓, a = 0.824 m/s² ✓. Forward check (C02): the material derivative $\frac{Du}{Dt}=\frac{\partial u}{\partial t}+u\frac{\partial u}{\partial x}=0+\alpha x\cdot\alpha=\alpha^2x$ reproduces step 6 from the field alone (the notebook's sympy line).

> ⚠️ **Common confusion (traps in `D01`):** Differentiating $u=\alpha x$ with respect to t at fixed x gives 0 — that is ∂u/∂t, not the acceleration. Forgetting that X is constant along the path. Writing $u=\alpha Xe^{\alpha t}$ as "the field" — it is a function of the label, not of position.

#### ✏️ Tiny example: the stretching flow x = X e^{αt} with X = 2 m, α = 0.5 s⁻¹, t = 1 s

1. Position: $x=2e^{0.5}=2\times1.6487=3.297$ m.
2. Lagrangian velocity from (3.1), $u=dr/dt$ with the label fixed: $u=\alpha Xe^{\alpha t}=0.5\times3.297=1.649$ m/s.
3. Eulerian velocity field from D01: $u(x,t)=\alpha x=0.5\times3.297=1.649$ m/s — the same number, as
   $F[\mathbf r(t;\mathbf r_o,t_o),t]=F(\mathbf x,t)$ at $\mathbf x=\mathbf r$ *(Eq. 3.2)* demands.
4. Acceleration: $a=\alpha^2x=0.25\times3.297=0.824$ m/s².
5. A fixed probe at x = 3.297 m reads 1.649 m/s for ever (the field is steady) while every particle passing it speeds up.

In [ ]:
X, alpha, t = 2.0, 0.5, 1.0                                        # label [m], stretching rate [1/s], time [s]
x = ch03.lagrangian_map_example(X, t, alpha)                       # position x = X e^{αt} [m]
u, a = ch03.lagrangian_velocity_acceleration(lambda s: ch03.lagrangian_map_example(X, s, alpha), t)   # (3.1) by central differences
print(f"x = {x:.4f} m, u = {u:.4f} m/s, alpha*x = {alpha*x:.4f}, a = {a:.4f} m/s^2")
Xs, xs, ts, al = sp.symbols('X x t alpha', positive=True)          # the same map symbolically
res = ch03.lagrangian_to_eulerian([Xs*sp.exp(al*ts)], [Xs], [xs], ts)   # solve for the label, substitute (3.2)
print(res["label_of_x"], res["u"], res["a_lagrangian"], sp.simplify(res["Du_Dt"][0] - res["a_lagrangian"][0]))

**What does the code above do?**

1. The map and its derivatives at fixed label, $\mathbf u=d\mathbf r/dt$ and $\mathbf a=d^2\mathbf r/dt^2$ *(Eq. 3.1)*,
   by central differences (Ch. 1 P21).
2. The symbolic route solves for the label, substitutes it into $dr/dt$ and $d^2r/dt^2$ (D01), and
3. checks that the material derivative of the Eulerian $u$ — the formula C02 derives — gives the same acceleration (the
   last printed 0).

In [ ]:
h = 1e-4                                                           # time step for the differences [s]
x_ = lambda s: ch03.lagrangian_map_example(X, s, alpha)            # the path of particle X
u_fd = (x_(t + h) - x_(t - h))/(2*h)                               # central difference: slope of the particle's path
assert np.allclose(u_fd, alpha*x, rtol=1e-7)                       # Lagrangian slope = Eulerian field αx at its position
a_fd = (x_(t + h) - 2*x_(t) + x_(t - h))/h**2                      # second difference: the path's curvature in time
assert np.allclose(a_fd, alpha**2*x, rtol=1e-5)                    # = α²x, the acceleration field of D01
print("from scratch:", round(u_fd, 6), round(a_fd, 5))

In [ ]:
alpha = 0.5                                                        # stretching rate [1/s]
t = np.linspace(0, 3, 200)                                         # time [s]
fig, (a, b) = plt.subplots(1, 2, figsize=(10.5, 4))              # (a) Lagrangian view, (b) Eulerian view
for X in np.linspace(0.25, 1.5, 6):                                # six labelled particles (their x at t = 0) [m]
    a.plot(t, X*np.exp(alpha*t), color=COLORS["orange"], lw=1.8)   # path x(t) = X e^{αt}
    a.annotate(f"X = {X:.2f}", (0, X), xytext=(3, 2), textcoords="offset points", fontsize=7)   # its label
a.axhline(2.0, color=COLORS["muted"], lw=1.2)                      # a fixed probe at x = 2 m
X1 = 1.25; tc = np.log(2.0/X1)/alpha                               # when particle X = 1.25 m reaches the probe [s]
a.plot([tc - 0.4, tc + 0.4], [2 - 0.4, 2 + 0.4], color=COLORS["accent"], lw=2.5)   # tangent of slope u = αx = 1 m/s
a.annotate("slope u = αx = 1 m/s at the probe", (tc, 2.0), xytext=(tc - 1.6, 2.9), fontsize=8, color=COLORS["accent"],
           arrowprops=dict(arrowstyle="->", color=COLORS["accent"]))   # point at the tangent
a.set_xlabel("$t$ [s]"); a.set_ylabel("$x$ [m]"); a.set_ylim(0, 4)  # axes with units
a.set_title("(a) Lagrangian: one path per particle")
xg = np.linspace(0, 4, 50)                                         # positions along the line [m]
b.plot(xg, alpha*xg, color=COLORS["teal"], lw=2.5, label="$u(x)=\\alpha x$ at $t$ = 0 s")          # the field now
b.plot(xg, alpha*xg, "--", color=COLORS["ink"], lw=1.2, label="$u(x)$ at $t$ = 2 s (the same line)")   # … and later
X0 = np.linspace(0.25, 1.5, 6)                                     # the same six particles
b.plot(X0, alpha*X0, "o", color=COLORS["orange"], label="particles at $t$ = 0")                    # where they are at 0 s
b.plot(X0*np.e, alpha*X0*np.e, "s", color=COLORS["orange"], mfc="none", label="particles at $t$ = 2 s")   # x = X e^{1}
b.set_xlabel("$x$ [m]"); b.set_ylabel("$u$ [m/s]"); b.legend(fontsize=8)   # axes with units
b.set_title("(b) Eulerian: one field, the same at every time")
savefig(fig, "ch03", "lagrangian_eulerian"); plt.show()            # save to outputs/ch03 and draw

**What you see.** Orange curves fanning out in (a); one teal straight line in (b) that does not move, with the particles sliding along it.

**How to read it.** Every particle crossing the grey probe line crosses it with the same slope 1 m/s (the field is steady), but each
curve bends upward (each particle accelerates). In (b) the particles move up the line — from the circles to the
squares — so each one's velocity grows although the line itself never changes. The two panels are the two descriptions
of one motion.

**What would change if…** …α doubled: the fan in (a) opens twice as fast and the line in (b) doubles its slope — and still does not move in time.

> ⚠️ **Common confusion:** "a steady velocity field means the fluid does not accelerate." Here $u=\alpha x$ does not
> depend on t at any fixed point, yet every particle speeds up at $a=\alpha^2x$. Steadiness is a statement about
> points; acceleration is about particles. C02 shows the missing piece is the advective term.

**What would change if…** …the map were $x=X+Ut$ (uniform drift)? Then $u=U$ everywhere, a = 0, and both descriptions are trivially the same.
The interesting case is when the particle moves *through* a field that varies in space — then a thermometer on the
float and one on the pier disagree. How fast does the float's reading change? That is C02.

### 🧩 The material derivative: the rate a moving parcel feels (3.4)–(3.5) `C02`

*The question:* A weather station reports warming of 0.36 K per hour, yet the air blowing past it has kept exactly its own
temperature. How can both be true?

*In one line:* $\frac{d}{dt}F[\mathbf r,t]=(\nabla F)\cdot\mathbf u+\frac{\partial F}{\partial t}\equiv\frac{DF}{Dt}$ *(3.4)* · $\frac{DF}{Dt}=\frac{\partial F}{\partial t}+\mathbf u\cdot\nabla F$ *(3.5)*

#### The problem in plain words

A southerly wind carries warm air north over a station. The station's thermometer rises; a thermometer tied to a
balloon drifting with the same air does not change at all. Forecasters call the station's warming *warm advection*.
We want the formula that separates "the field changes here" from "the parcel moves to where the field is different" —
the material derivative.

#### The idea

```
thermometer on the pier      reads  ∂T/∂t          (x fixed)
thermometer on the balloon   reads  DT/Dt          (label fixed)
the difference                      u·∇T           (the balloon moves through a gradient)
           DT/Dt  =  ∂T/∂t  +  u·∇T         → blue + amber = purple in every picture below
```

> 📎 **Primer — multivariable chain rule along a path.** `P91` · If $f(t)=F(x(t),y(t),z(t),t)$, then
$\frac{df}{dt}=\frac{\partial F}{\partial x}\frac{dx}{dt}+\frac{\partial F}{\partial y}\frac{dy}{dt}+\frac{\partial F}{\partial z}\frac{dz}{dt}+\frac{\partial F}{\partial t}$:
each argument that changes contributes (sensitivity to it) × (its rate). Time appears twice — through the moving
position and through the explicit clock. This extends the two-variable chain rule of Ch. 1 (P49).

In [ ]:
t = sp.symbols('t')                                  # time
x = sp.cos(t); F = lambda x, t: x**2*t               # a path x(t) and a field F(x, t)
direct = sp.diff(F(x, t), t)                         # differentiate after substituting the path
X, T = sp.symbols('X T')                             # F's own arguments
chain = (sp.diff(F(X, T), X)*sp.diff(x, t) + sp.diff(F(X, T), T)).subs({X: x, T: t})   # (∂F/∂x)(dx/dt) + ∂F/∂t
print(sp.simplify(direct - chain))                   # 0: the chain rule holds

#### 🧮 Derivation — The material derivative: (3.3) → (3.4) → (3.5) `D02` (Eq. 3.5)

**What we want to show.** Find the rate at which a property F changes for an observer riding with a fluid particle, written only in terms of the Eulerian field F(x, t) and the velocity field u(x, t).

**Assumptions.** Continuum: at every instant every point is occupied by exactly one particle (step 6); F differentiable in x and t (step 3).

**The plan.**

1. Fix one particle, so F along its path is a function of t alone.
2. Differentiate with the chain rule — time enters through position and explicitly.
3. Recognise the particle's velocity and the field's slopes.
4. Argue the result holds at every point and name it.

**Tools we use** (each explained before this point): (3.2), $F[\mathbf r(t;\mathbf r_o,t_o),t]=F(\mathbf x,t)\ \text{at}\ \mathbf x=\mathbf r(t;\mathbf r_o,t_o)$ (C01) · multivariable chain rule along a path (primer in C02) · (3.1), $\mathbf u=d\mathbf r/dt$ (N08) · partial derivative (ch01 P25) · summation convention and the gradient (Ch. 2 §2.1, §2.9).

**We start from**

$$ \text{(3.2):}\ F[\mathbf r(t;\mathbf r_o,t_o),t]=F(\mathbf x,t)\ \text{when}\ \mathbf x=\mathbf r(t;\mathbf r_o,t_o) $$

*In words:* the value a particle carries equals the field's value at the point it occupies.

---

**Step 1 of 9 — Follow one particle.**

$$ f(t)\equiv F[\mathbf r(t;\mathbf r_o,t_o),t] $$

- *Why we can do this:* The labels $\mathbf r_o$, $t_o$ are fixed for a chosen particle, so what its thermometer reads is an ordinary function of one variable, t. We want df/dt.
- *In words:* The reading of a thermometer riding with the particle.

---

**Step 2 of 9 — List what depends on t.**

$$ f(t)=F\big(r_1(t),r_2(t),r_3(t),t\big) $$

- *Why we can do this:* F has four arguments — three coordinates and time — and along the path all four change with t: the position because the particle moves, the last because the clock runs.
- *In words:* Time enters twice: through where the particle is, and directly.

---

**Step 3 of 9 — Apply the chain rule.**

$$ \dfrac{df}{dt}=\dfrac{\partial F}{\partial r_i}\dfrac{dr_i}{dt}+\dfrac{\partial F}{\partial t} $$

- *Why we can do this:* Multivariable chain rule (primer): each changing argument contributes its sensitivity times its rate; the repeated i sums the three coordinates. This is (3.3), $\frac{d}{dt}F[\mathbf r,t]=\frac{\partial F}{\partial r_i}\frac{dr_i}{dt}+\frac{\partial F}{\partial t}$ written out.
- *In words:* The reading changes because the particle moves through the field and because the field itself changes.

---

**Step 4 of 9 — Recognise the particle's velocity.**

$$ \dfrac{df}{dt}=\dfrac{\partial F}{\partial r_i}\,u_i+\dfrac{\partial F}{\partial t} $$

- *Why we can do this:* (3.1), $\mathbf u=d\mathbf r/dt$: the rate of change of the particle's position components is its velocity components.
- *In words:* The first term is 'how fast we move through the field'.

---

**Step 5 of 9 — Evaluate the slopes at x = r.**

$$ \dfrac{df}{dt}=u_i\dfrac{\partial F}{\partial x_i}+\dfrac{\partial F}{\partial t} $$

- *Why we can do this:* At time t the particle sits at $\mathbf x=\mathbf r$, so $\partial F/\partial r_i$ is the field's slope in direction i there; renaming the argument changes nothing. The book states this.
- *In words:* Everything on the right is now an Eulerian quantity at (x, t).

---

**Step 6 of 9 — Extend to every point.**

$$ \dfrac{df}{dt}\Big|_{\mathbf r=\mathbf x}=u_i\dfrac{\partial F}{\partial x_i}+\dfrac{\partial F}{\partial t}\quad\text{for all }\mathbf x,t $$

- *Why we can do this:* The book skips this: at any instant every point is occupied by some particle (continuum, Ch. 1), so choosing that particle's labels makes the line hold at any x.
- *In words:* The rule is a statement about the whole field, not one path.

---

**Step 7 of 9 — Give it a name.**

$$ \dfrac{DF}{Dt}\equiv\dfrac{\partial F}{\partial t}+u_i\dfrac{\partial F}{\partial x_i} $$

- *Why we can do this:* A symbol for 'd/dt following the particle, in Eulerian variables' — the book's (3.4), $\frac{d}{dt}F[\mathbf r,t]=(\nabla F)\cdot\mathbf u+\frac{\partial F}{\partial t}\equiv\frac{DF}{Dt}$ defines D/Dt exactly this way (material, substantial or particle derivative).
- *In words:* D/Dt means: the rate seen by a moving parcel.

---

**Step 8 of 9 — Write the sum as a dot product.**

$$ \dfrac{DF}{Dt}\equiv\dfrac{\partial F}{\partial t}+\mathbf u\cdot\nabla F $$

- *Why we can do this:* Summation convention (Ch. 2 §2.1): $u_i\,\partial F/\partial x_i$ is the dot product of u with the gradient $\nabla F$ (Ch. 2 §2.9). This is (3.5), $\frac{DF}{Dt}=\frac{\partial F}{\partial t}+\mathbf u\cdot\nabla F$.
- *In words:* Local rate plus the advective rate u·∇F.

---

**Step 9 of 9 — Apply it to each velocity component.**

$$ \dfrac{D\mathbf u}{Dt}=\dfrac{\partial\mathbf u}{\partial t}+(\mathbf u\cdot\nabla)\mathbf u $$

- *Why we can do this:* Steps 1–8 hold for any scalar field, so apply them with $F=u_j$ for j = 1, 2, 3 separately; the same operator acts on each component. We need this form in C05.
- *In words:* A particle's acceleration = local + advective acceleration.

---

**Result**

$$ \begin{array}{l}\dfrac{DF}{Dt}\equiv\dfrac{\partial F}{\partial t}+\mathbf u\cdot\nabla F=\dfrac{\partial F}{\partial t}+u_i\dfrac{\partial F}{\partial x_i}\ \text{(3.5), and} \\ \frac{D\mathbf u}{Dt}=\frac{\partial\mathbf u}{\partial t}+(\mathbf u\cdot\nabla)\mathbf u\end{array} $$

*In words:* what a moving parcel feels = what a fixed probe sees + what it meets by moving through the gradient.

**What it means.** Every conservation law of Ch. 4 is written with D/Dt because the laws are about particles while the equations are solved on fixed grids. The advective term is where fluid mechanics becomes nonlinear (C05, N22). The formula needs a continuum; at a point where particles do not exist (a free surface's edge, a shock) the derivative is taken one-sidedly.

**Check it.** Units: both terms [F]/s ✓. Limits: F steady ⇒ DF/Dt = u·∇F (all advective); u = 0 ⇒ DF/Dt = ∂F/∂t; u ⟂ ∇F ⇒ no advective part ✓. Numbers (C02 worked example): ∂T/∂t = +1.0 × 10⁻⁴, u·∇T = −1.0 × 10⁻⁴, DT/Dt = 0 K/s ✓. D01's field: F = u = αx gives Du/Dt = 0 + αx·α = α²x — the acceleration found there ✓. The sympy cell below applies the chain rule along the path $y=vt$ to the front $T=T_0-G(y-ct)$.

In [ ]:
t, T0, G, c, v = sp.symbols('t T_0 G c v')           # time, reference temperature, gradient, front speed, wind speed
x, y = sp.symbols('x y')                             # the field's own coordinates
T = T0 - G*(y - c*t)                                 # a linear front moving north at c: T(x, y, t)
y_path = v*t                                         # the balloon's path: carried north by the wind from y = 0
direct = sp.diff(T.subs(y, y_path), t)               # step 1: the balloon's reading f(t), differentiated directly
DTDt = (sp.diff(T, t) + 0*sp.diff(T, x) + v*sp.diff(T, y)).subs(y, y_path)   # step 8: ∂T/∂t + u·∇T with u = (0, v)
print(sp.simplify(direct - DTDt))                    # 0: (3.5) gives the balloon's own rate
print(sp.simplify(direct))                           # G(c − v): zero when the pattern moves with the wind

> ⚠️ **Common confusion (traps in `D02`):** Reading ∂T/∂t as 'the air warms' (it is the fixed station). Holding the label fixed vs holding x fixed: df/dt in step 1 is at fixed label, ∂F/∂t at fixed x. Treating D/Dt of a vector as something new — it acts component by component in Cartesian coordinates (curvilinear components need Appendix B's extra terms).

📝 **Note.** **The chain rule written out, (3.3)** `N09` — Step 3 of the derivation is the book's (3.3) written out — the chain rule along a particle's path, with the four
arguments of F each contributing:

$$ \frac{d}{dt}F[\mathbf r(t;\mathbf r_o,t_o),t]=\frac{\partial F}{\partial r_1}\frac{dr_1}{dt}+\frac{\partial F}{\partial r_2}\frac{dr_2}{dt}+\frac{\partial F}{\partial r_3}\frac{dr_3}{dt}+\frac{\partial F}{\partial t}=\frac{d}{dt}F(\mathbf x,t)\quad\text{when}\quad\mathbf x=\mathbf r(t;\mathbf r_o,t_o) \qquad \text{(3.3)} $$

📝 **Note.** **…and (3.4)** `N09` — Steps 4–7 turn it into the book's (3.4) — velocity components in place of $dr_i/dt$, the field's slopes at
$\mathbf x=\mathbf r$, and a name for the result: the **material**, **substantial** or **particle** derivative.

$$ \frac{d}{dt}F[\mathbf r(t;\mathbf r_o,t_o),t]=\frac{\partial F}{\partial x_1}u_1+\frac{\partial F}{\partial x_2}u_2+\frac{\partial F}{\partial x_3}u_3+\frac{\partial F}{\partial t}=(\nabla F)\cdot\mathbf u+\frac{\partial F}{\partial t}\equiv\frac{D}{Dt}F(\mathbf x,t) \qquad \text{(3.4)} $$

📝 **Note.** **Vector and index forms** `N11` — the forms used for the rest of the book. The repeated index i is the dot product $\mathbf u\cdot\nabla F$ (summation
convention, Ch. 2 §2.1); the code below expands it with the Ch. 2 index parser (comma notation: `F,i` means
$\partial F/\partial x_i$).

$$ \frac{DF}{Dt}\equiv\frac{\partial F}{\partial t}+\mathbf u\cdot\nabla F,\quad\text{or}\quad\frac{DF}{Dt}\equiv\frac{\partial F}{\partial t}+u_i\frac{\partial F}{\partial x_i} \qquad \text{(3.5)} $$

In [ ]:
print(ch03.expand_indices_str("u_i F,i"))            # u_i ∂F/∂x_i with the repeated i summed over 1, 2, 3

**What does the code above do?**

The repeated index i becomes the three-term sum $u_1\,\partial F/\partial x_1+u_2\,\partial F/\partial x_2+u_3\,\partial F/\partial x_3=\mathbf u\cdot\nabla F$ (Ch. 2 §2.1).

📝 **Note.** **Local and advective parts** `N10` — $\partial F/\partial t$ (**blue**) is the *local* or unsteady rate — what a fixed probe sees; it vanishes when F does
not depend on time. $\mathbf u\cdot\nabla F$ (**amber**) is the *advective* rate — the change a particle meets because
it moves to where F is different; it vanishes when F is uniform, when u = 0, or when $\mathbf u\perp\nabla F$ (wind
blowing along the isotherms). The book says *advection* for transport by the flow and keeps *convection* for heat
carried by fluid motion.

📝 **Note.** **The streamwise form** `N12` — Because $\mathbf u\cdot\nabla F=|\mathbf u|\,(\mathbf e_u\cdot\nabla F)$ with the unit vector $\mathbf e_u=\mathbf u/|\mathbf u|$,
and $\mathbf e_u\cdot\nabla F=\partial F/\partial s$ is the directional derivative along the path (Ch. 2 primer P75),
the material derivative can be written with the arc length s along the particle's path. ⚠️ **Book typo:** the printed
(3.6) ends with $|\mathbf u|\,\partial/\partial s$ — the F is missing (both terms must be rates of change of F). The
form is undefined at a stagnation point (u = 0, no direction).

$$ \frac{DF}{Dt}=\frac{\partial F}{\partial t}+|\mathbf u|\frac{\partial F}{\partial s} \qquad \text{(3.6)} $$

> ⚠️ **Common confusion:** "$\partial T/\partial t$ is how fast the air warms." It is how fast a *fixed thermometer*
> warms. The air's own rate is $DT/Dt$. They differ by the advective term $\mathbf u\cdot\nabla T$ — often the largest
> term on a weather map.

#### ✏️ Tiny example: warm advection over a station

Temperature falls northward by 1 K per 100 km: $\partial T/\partial y=-1/10^5=-1\times10^{-5}$ K/m. A southerly wind
v = 10 m/s blows north.

1. Advective term: $\mathbf u\cdot\nabla T=v\,\partial T/\partial y=10\times(-10^{-5})=-1\times10^{-4}$ K/s.
2. The air keeps its temperature (no heating): $DT/Dt=0$.
3. From $\frac{DT}{Dt}=\frac{\partial T}{\partial t}+\mathbf u\cdot\nabla T$ *(Eq. 3.5)*:
   $\partial T/\partial t=0-(-10^{-4})=+1\times10^{-4}$ K/s.
4. Per hour: $10^{-4}\times3600=0.36$ K/h of warming at the station, although no parcel warmed at all.

In [ ]:
G, v = 1e-5, 10.0                                          # gradient: 1 K per 100 km [K/m]; southerly wind [m/s]
terms = ch03.thermal_front_terms(0.0, 0.0, 0.0, 0.0, v, G, heating_K_per_s=0.0, front_speed=v)   # exact terms of (3.5)
print({k: f"{float(terms[k]):+.2e}" for k in ("local", "advective", "total")})   # K/s: blue, amber, purple
print(f"station warming: {float(terms['local'])*3600:.2f} K/h ({terms['regime']})")
F = lambda x, t: ch03.thermal_front(x[0], x[1], t, G, 0.0, v)   # the same field T(x, t) as a callable [K]
u = lambda x, t: np.array([0.0, v]) + 0*x                     # the uniform southerly wind u(x, t) [m/s]
p = np.array([0.0, 0.0])                                      # the station's position [m]
print(ch03.material_derivative_terms(F, u, p, 0.0, h=10.0))   # general stencils (h = 10 m on a 100 km scale)
print(ch03.streamwise_derivative(F, u, p, 0.0, h=10.0))       # (3.6): |u| ∂T/∂s = the advective term
x_, y_, t_ = sp.symbols('x y t')                              # symbolic twin
print(sp.simplify(ch03.material_derivative_sym(288 - sp.Rational(1, 100000)*(y_ - 10*t_), [0, 10], [x_, y_], t_)))

**What does the code above do?**

1. `thermal_front_terms` gives the exact terms for our moving front (the pattern is carried by the wind, so the parcel's
   rate is zero); the dictionary (Ch. 1 P23) holds the local, advective and total rates.
2. `material_derivative_terms` gets the same numbers from general second-order stencils, which work for any F and u.
3. The streamwise form $\frac{DF}{Dt}=\frac{\partial F}{\partial t}+|\mathbf u|\frac{\partial F}{\partial s}$ *(3.6)*
   reproduces the advective term.
4. sympy writes $\frac{DF}{Dt}=\frac{\partial F}{\partial t}+\mathbf u\cdot\nabla F$ *(3.5)* symbolically and gets 0.

In [ ]:
ht, hx = 1.0, 10.0                                            # time step [s] and space step [m] for the differences
local = (F(p, ht) - F(p, -ht))/(2*ht)                         # ∂T/∂t at the fixed station (central difference)
dFdy = (F(p + np.array([0, hx]), 0) - F(p - np.array([0, hx]), 0))/(2*hx)   # ∂T/∂y at the station
adv = v*dFdy                                                  # u·∇T = v ∂T/∂y
assert np.allclose([local, adv], [terms["local"], terms["advective"]], rtol=1e-8)   # stencils = exact terms
ts = np.array([-60.0, 0.0, 60.0])                             # a minute before and after [s]
path = ch03.pathline(u, [0.0, 0.0], 0.0, ts)                  # the balloon's path line (3.8) through the station
dF_along = (F(path[:, 2], ts[2]) - F(path[:, 0], ts[0]))/(ts[2] - ts[0])   # the balloon's own thermometer rate
assert abs(dF_along) < 1e-9                                   # = DT/Dt = 0: the air keeps its temperature
print("fixed-point stencils and the riding thermometer agree:", local, adv, dF_along)

Two routes, one number: stencils at a fixed point, and a thermometer riding the path line.

**A front of finite width.** $T(y,t)=T_0-G\,w\tanh\big((y-ct)/w\big)$: warm to the south, cold to the north, width
w = 100 km, largest gradient $G=10^{-5}$ K/m at its centre, moving north at c = 5 m/s. Its slope is
$\partial T/\partial y=-G\,\mathrm{sech}^2\big((y-ct)/w\big)$ with $\mathrm{sech}^2=1-\tanh^2$. A float starts 200 km south
and drifts north with the wind, v = 10 m/s; a station sits at y = 100 km.

In [ ]:
G, w, c, v = 1e-5, 1e5, 5.0, 10.0                           # largest gradient [K/m], front width [m], front speed, wind [m/s]
ys, y0 = 1e5, -2e5                                          # station at y = 100 km; float released at y = −200 km [m]
hours = np.linspace(0, 24, 241); t = hours*3600             # a day [h] and [s]
T = lambda y, t: ch03.thermal_front(0.0, y, t, G, 0.0, c, width_m=w)   # the tanh front moving north at c [K]
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4))          # (a) snapshots in space, (b) two thermometers in time
yy = np.linspace(-4e5, 5e5, 400)                            # north–south line [m]
for h, al in zip((0, 6, 12), (0.4, 0.7, 1.0)):              # three snapshots, darker = later
    a.plot(yy/1e3, T(yy, h*3600), color=COLORS["blue"], alpha=al, label=f"$T(y)$ at {h} h")    # the front at h hours
    a.plot((y0 + v*h*3600)/1e3, T(y0 + v*h*3600, h*3600), "o", color=COLORS["accent"], ms=8, alpha=al)   # the float then
a.plot(ys/1e3, T(ys, 0), "s", color=COLORS["blue"], ms=9, label="station ($y$ = 100 km)")   # the fixed station
a.plot([], [], "o", color=COLORS["accent"], label="float at those times")                   # legend entry only
a.set_xlabel("$y$ (north) [km]"); a.set_ylabel("$T$ [K]"); a.legend(fontsize=8)             # axes with units
a.set_title("A front slides north; the float overtakes it")
b.plot(hours, T(ys, t), color=COLORS["blue"], lw=2.2, label="station reads $T(y_s, t)$ → slope ∂T/∂t")        # x fixed
b.plot(hours, T(y0 + v*t, t), color=COLORS["accent"], lw=2.2, label="float reads $T(y_0+vt, t)$ → slope DT/Dt")   # label fixed
b.set_xlabel("$t$ [h]"); b.set_ylabel("$T$ [K]"); b.legend(fontsize=8)                      # axes with units
b.set_title("Two thermometers, one moving temperature pattern")
savefig(fig, "ch03", "front_station_float"); plt.show()     # save to outputs/ch03 and draw

**What you see.** Left: a smooth step in temperature (warm south, cold north) sliding north, purple dots for the float catching
up with it. Right: the station's curve rises as the front passes (≈ 5.6 h), the float's curve falls as it crosses into
colder air (≈ 11 h).

**How to read it.** The station sees $\partial T/\partial t=+Gc\,\mathrm{sech}^2(\cdot)$ (the pattern moving past;
$\mathrm{sech}^2=1-\tanh^2$ is the slope of the tanh profile). The float sees $DT/Dt=G(c-v)\,\mathrm{sech}^2(\cdot)<0$
because it outruns the pattern. Where the two curves have the same slope the advective term is zero (far from the
front, where ∇T ≈ 0).

**What would change if…** …the front moved exactly with the wind (c = v): the float's curve would be flat ($DT/Dt=0$) and the station's
warming would be pure advection — the worked example above.

In [ ]:
G, w, c, v, y0 = 1e-5, 1e5, 5.0, 10.0, -2e5                 # the same front and float as the figure above
hrs = np.linspace(0, 24, 97)                                # time axis [h]
yf = y0 + v*hrs*3600                                        # the float's position along its path line [m]
tt = ch03.thermal_front_terms(0.0, yf, hrs*3600, 0.0, v, G, 0.0, c, width_m=w)   # exact terms along the path
loc, adv, tot = (np.asarray(tt[k])*3600 for k in ("local", "advective", "total"))  # K/s → K/h

def curves(T):                                              # the three terms up to the slider time T [h]
    k = hrs <= T + 1e-9                                     # the part of the day already travelled
    return {"∂T/∂t (local, at the float's point)": (hrs[k], loc[k]), "u·∇T (advective)": (hrs[k], adv[k]),
            "DT/Dt (the float's rate)": (hrs[k], tot[k]), "DT/Dt, whole day (ghost)": (hrs, tot)}   # name → (x, y)

steps = np.linspace(0, 24, 25 if not FAST else 13)          # slider positions [h]
fig = slider_figure(curves, "t", steps, unit="h", xlabel="time [h]", ylabel="rate [K/h]", xrange=[0, 24],
                    title="The three terms along the float: the balance shifts as it crosses the front")   # precomputed
recolor(fig, {"∂T/∂t (local, at the float's point)": COLORS["blue"], "u·∇T (advective)": COLORS["amber"],   # blue, amber
              "DT/Dt (the float's rate)": COLORS["accent"], "DT/Dt, whole day (ghost)": COLORS["grid"]},     # purple, grey
        dashes={"DT/Dt, whole day (ghost)": "dot"})         # the ghost dotted
fig.show()                                                  # draw it

**What does the code above do?**

1. The float's path line is $y=y_0+vt$; `thermal_front_terms` evaluates the exact local, advective and total rates at
   each of its positions (arrays in, arrays out).
2. `slider_figure` precomputes the curves for every slider time, so the figure works on the web page without Python.
3. `recolor` gives the curves the notebook's colours: blue local, amber advective, purple total.

**What you see.** Three curves drawn up to the slider time; all three peak near 11 h, when the float is at the front's centre.

**How to read it.** The purple curve is the sum of the blue and the amber ones at every instant, $\frac{DT}{Dt}=\frac{\partial T}{\partial t}+\mathbf u\cdot\nabla T$
*(Eq. 3.5)*: blue is positive (the pattern warms any fixed point it passes), amber is more negative (the float runs into
colder air twice as fast as the pattern moves), so purple is negative.

**What would change if…** …you stopped the slider at t ≈ 11 h (the crossing): all three curves are at their largest values there. With c = v the purple curve would lie on zero.

#### 🎮 Interactive: Why does the station warm while the air does not?

**Why interactive:** one static front shows one wind direction and one speed; only by turning the wind yourself do you see the amber term vanish along the isotherms and the float's rate change sign as it outruns the pattern.
Two observers read one moving temperature pattern on one clock: a fixed probe ($\partial T/\partial t$) and a float
carried by the wind ($DT/Dt$). You set the wind, the gradient and any heating; the term bars show the local and
advective parts adding to the material derivative $\frac{DT}{Dt}=\frac{\partial T}{\partial t}+\mathbf u\cdot\nabla T$
*(Eq. 3.5)*, and the float's measured rate lands on the purple bar.

**What to try:**
- Pick the preset 'pure advection': the float's reading stays flat while the probe warms — read the numbers in Explain.
- Turn the wind until it blows along the isotherms: the amber bar vanishes (u ⟂ ∇T).
- Switch to the stretching-map mode (x = X e^{αt}): now F is the velocity itself and the float's rate is its acceleration α²x.
- Open the Derivation tab and step through D02 with your wind speed.

In [ ]:
show_viz("ch03", "material_derivative_probe")   # full-width explainer; ⤢ Full screen for more room

**What would change if…** …F were a velocity component instead of a temperature? Then D/Dt of u is the particle's acceleration,
$\frac{D\mathbf u}{Dt}=\frac{\partial\mathbf u}{\partial t}+(\mathbf u\cdot\nabla)\mathbf u$ — the left side of Newton's
law in Ch. 4. Section 3.3 asks what these rates look like as curves in the flow, and whether the split into local and
advective parts depends on who is watching (C05).

---

## 3.3 Flow Lines, Fluid Acceleration, and Galilean Transformation

**What is this section about?** Three curves that picture a flow — the instantaneous direction (streamline), the track
of one particle (path line) and the line of dye from a fixed port (streak line) — and the proof that a particle's
acceleration is the same for every observer moving at constant velocity, even though "steady" is not.

### 🧩 Streamlines: the flow's direction at one frozen instant (3.7) `C03`

*The question:* Freeze the flow for an instant and draw curves that follow the velocity arrows everywhere. What equations do those
curves obey, and why do they change from one instant to the next?

*In one line:* $dx/u=dy/v=dz/w$ *(3.7)*

#### The problem in plain words

A satellite image of cloud streaks, or iron filings along a magnet's field lines, shows *directions at one instant*.
A weather chart's wind streamlines are the same idea. To draw one we need a rule: at every point of the curve, its
tangent points along $\mathbf u$ at that instant.

#### The idea

```
freeze the clock at t   →   field of arrows u(x, t)   →   curves everywhere tangent to the arrows
ds ∥ u   ⇔   dx/u = dy/v = dz/w   ⇔   u × ds = 0            (unsteady flow: a new picture every instant)
```

> 📎 **Primer — parametric curves, tangent vector and arc length.** `P92` · A curve can be written $\mathbf x(s)$ with a parameter s; its tangent is $d\mathbf x/ds$. If s is the **arc length**
(distance measured along the curve), the tangent has length 1. To draw a curve tangent to $\mathbf u$, march along
$d\mathbf x/ds=\mathbf u/|\mathbf u|$.

In [ ]:
s = np.linspace(0, 2*np.pi, 5)                       # arc length along the unit circle [m]
x, y = np.cos(s), np.sin(s)                          # the curve x(s)
tx, ty = -np.sin(s), np.cos(s)                       # its tangent dx/ds
print(np.hypot(tx, ty))                              # [1. 1. 1. 1. 1.]: unit length, because s is arc length

> 📎 **Primer — parallel vectors and the cross-product test.** `P93` · Two vectors are parallel when one is a multiple of the other, $\mathbf a=\lambda\mathbf b$ — equivalently when
$\mathbf a\times\mathbf b=0$ (the cross product, Ch. 2 §2.7, measures the area they span; in components
$(\mathbf a\times\mathbf b)_i=\varepsilon_{ijk}a_jb_k$). The ratio test $a_x/b_x=a_y/b_y=a_z/b_z$ says the same but fails
when a component of $\mathbf b$ is zero; the cross product never divides.

In [ ]:
a, b = np.array([2.0, 4.0, 0.0]), np.array([1.0, 2.0, 0.0])   # a = 2b
print(np.cross(a, b), a[0]/b[0], a[1]/b[1])                   # [0 0 0] 2.0 2.0: parallel (the z-ratio would be 0/0)

#### 🧮 Derivation — The streamline equations dx/u = dy/v = dz/w, Eq. (3.7) `D03` (Eq. 3.7)

**What we want to show.** Turn "a curve that is tangent to the velocity everywhere at one instant" into equations we can integrate — the book leaves this to Exercise 3.3.

**Assumptions.** The clock is frozen at t (every step); $\mathbf u\neq0$ at the points used (step 3 divides by components, step 4 by |u|).

**The plan.**

1. Write parallel as "one is a multiple of the other".
2. Take components.
3. Eliminate the multiple.
4. Fix the multiple by arc length to get an ODE we can integrate.

**Tools we use** (each explained before this point): Parallel vectors and the cross-product test (primer in C03) · parametric curves, tangent vector and arc length (primer in C03).

**We start from**

$$ \begin{array}{l}d\mathbf s\parallel\mathbf u(\mathbf x,t)\ \text{at a fixed t, with}\ d\mathbf s=(dx,dy,dz)\ \text{a small step along the curve and} \\ \mathbf u=(u,v,w)\end{array} $$

*In words:* each little piece of the streamline points along the local velocity.

---

**Step 1 of 5 — Write parallel as a multiple.**

$$ d\mathbf s=\lambda\,\mathbf u $$

- *Why we can do this:* Two vectors are parallel when one is a scalar multiple of the other (primer); λ > 0 so the curve runs downstream, and λ has units of time (m ÷ m/s).
- *In words:* The small step is the velocity times a tiny 'time-like' number.

---

**Step 2 of 5 — Split into components.**

$$ dx=\lambda u,\quad dy=\lambda v,\quad dz=\lambda w $$

- *Why we can do this:* Two vectors are equal when each component is equal.
- *In words:* Each coordinate step is proportional to that velocity component.

---

**Step 3 of 5 — Eliminate λ.**

$$ \dfrac{dx}{u}=\dfrac{dy}{v}=\dfrac{dz}{w}\;(=\lambda) $$

- *Why we can do this:* Solve each line for λ by dividing by u, v, w (allowed only where they are nonzero); all three equal the same λ. This is (3.7), $dx/u=dy/v=dz/w$. Equivalently $\mathbf u\times d\mathbf s=0$, which never divides.
- *In words:* The direction numbers of the step match those of the velocity.

---

**Step 4 of 5 — Fix λ by the step's length.**

$$ \lambda=\dfrac{ds}{\lvert\mathbf u\rvert} $$

- *Why we can do this:* Take the length of both sides of step 1: $ds=\lambda\lvert\mathbf u\rvert$ with ds the arc length (primer). Choosing arc length makes the next line well defined everywhere u ≠ 0.
- *In words:* The step's length decides how big λ is.

---

**Step 5 of 5 — Write the arc-length ODE.**

$$ \dfrac{d\mathbf x}{ds}=\dfrac{\mathbf u(\mathbf x,t)}{\lvert\mathbf u(\mathbf x,t)\rvert} $$

- *Why we can do this:* Substitute λ into step 1 and divide by ds. The right side is a known unit vector field at frozen t, so this is an ODE `ch03.streamline` integrates both ways from a seed.
- *In words:* Walk along the unit arrows at one instant.

---

**Result**

$$ dx/u=dy/v=dz/w\ \text{(3.7), equivalently}\ \mathbf u\times d\mathbf s=0\ \text{and}\ d\mathbf x/ds=\mathbf u/\lvert\mathbf u\rvert\ \text{at frozen t} $$

*In words:* a streamline follows the arrows of one instant.

**What it means.** At each instant the flow has a streamline pattern; in unsteady flow it changes, so streamlines are not where particles go (C04). The ODE stops at a stagnation point (u = 0, no direction) — `streamline` ends the curve there.

**Check it.** Units: each ratio in s ✓. Plane flow: $dy/dx=v/u$; with u = (1, 2) the slope is 2 ✓. Solid-body rotation u = −y, v = x: $dy/dx=-x/y$ ⇒ $x^2+y^2=$ const, circles ✓. Ex. 3.1 at ωt′ = 30°: slope tan 30° = 0.577 ✓.

> ⚠️ **Common confusion (traps in `D03`):** Integrating (3.7), $dx/u=dy/v=dz/w$ forward in *time* (that gives a path line, not a streamline — t is frozen). Using the ratio form where a component vanishes (use $\mathbf u\times d\mathbf s=0$ or the arc-length ODE instead).

📝 **Note.** **The streamline equations** `N14` — are the result just derived. In a plane flow the first equality gives the slope $dy/dx=v/u$. Integrating from many
starting points, upstream and downstream, fills the picture. Number: where $\mathbf u=(1,2)$ m/s the streamline climbs
with slope 2.

$$ dx/u=dy/v=dz/w \qquad \text{(3.7)} $$

#### ✏️ Tiny example: slopes and a circle

1. At a point where u = 1 m/s, v = 2 m/s: $dy/dx=v/u=2$ (63.4° to the x-axis).
2. Solid-body rotation u = −y, v = x (s⁻¹ × m): $dy/dx=v/u=-x/y$, so $y\,dy=-x\,dx$ and $x^2+y^2=\text{const}$ — circles.
3. Ex. 3.1 at the instant t′ = π/4 s (ω = 1 s⁻¹): $v/u=\tan(\omega t')=1$ at every point, so every streamline is a
   straight line at 45°.

> 📎 **Primer — solve_ivp options: t_eval, dense_output, events, backward integration.** `P94` · Ch. 1 used `solve_ivp` with its defaults (P31). Here: `t_eval` asks for the solution at chosen times;
`dense_output=True` returns a continuous solution `sol.sol(t)`; an `events` function stops the integration where it
crosses zero (we stop a streamline at a stagnation point, $|\mathbf u|=0$, where the direction is undefined); a time
span that runs backwards (`t_span=(0, -5)`) integrates into the past — how a streamline is drawn upstream and a streak
particle is traced back to the port.

In [ ]:
from scipy.integrate import solve_ivp                           # adaptive ODE solver (Ch. 1 P31)
stop = lambda t, y: y[0] - 0.5; stop.terminal = True           # event: stop when y reaches 0.5
sol = solve_ivp(lambda t, y: -y, (0, 5), [1.0], events=stop, dense_output=True, rtol=1e-8)   # dy/dt = −y, y(0) = 1, tight tolerance
print(sol.t_events[0], sol.sol(0.3))                           # [0.693] (= ln 2) and y(0.3) = e^-0.3 ≈ 0.741

In [ ]:
u31 = ch03.preset_field("ex31", omega=1.0, xi0=1.0)          # Ex. 3.1: u = ωξ_o cos ωt, v = ωξ_o sin ωt as a callable u(x, t)
for tp in (0.0, np.pi/4, np.pi/2):                            # three drawing instants t' [s]
    sl = ch03.streamline(u31, [0.0, 0.0], tp, s_max=2.0)      # dx/ds = u/|u| at frozen t', 2 m each way from the origin
    print(f"t' = {tp:.3f} s: streamline ends at {np.round(sl[:, -1], 3)}")
print(ch03.streamline_slope(u31, np.array([0.3, -0.7]), np.pi/4))   # (3.7) in a plane: dy/dx = v/u at any point
usb = lambda x, t: np.array([-x[1], x[0]])                    # solid-body rotation u = (−y, x) [m/s]
c = ch03.streamline(usb, [1.0, 0.0], 0.0, s_max=2*np.pi)      # the streamline through (1, 0)
print(np.ptp(np.hypot(c[0], c[1])))                           # spread of its radius: ≈ 0 → a circle

**What does the code above do?**

1. `preset_field("ex31")` is the Ex. 3.1 field as a callable $\mathbf u(\mathbf x,t)$ (the same at every point).
2. `streamline` integrates $d\mathbf x/ds=\mathbf u/|\mathbf u|$ with the clock frozen at t′, both directions from the
   seed (D03 step 5); the end points (2, 0), (1.414, 1.414), (0, 2) are straight lines at 0°, 45°, 90°.
3. `streamline_slope` is the first equality of $dx/u=dy/v=dz/w$ *(3.7)*: the slope v/u = 1 at t′ = π/4.
4. For solid-body rotation the computed streamline keeps its radius (spread ≈ 10⁻⁹ m): a circle.

In [ ]:
xg = np.linspace(-2, 2, 21); X, Y = np.meshgrid(xg, xg)      # a 21 × 21 grid [m] (Ch. 2 P76)
fig, axs = plt.subplots(1, 4, figsize=(13, 3.5))
for ax, tp in zip(axs[:3], (0.0, np.pi/4, np.pi/2)):          # three frozen instants t' [s]
    U, V = ch03.unsteady_flow_preset("ex31", X, Y, tp, xi0=1.0, omega=1.0)   # the Ex. 3.1 field at t'
    ax.streamplot(X, Y, U + 0*X, V + 0*Y, color=COLORS["teal"], density=0.7)   # streamlines (Ch. 2 P78)
    ax.plot(0, 0, "s", color=COLORS["ink"])                   # the port at the origin
    ax.quiver(0, 0, float(np.mean(U)), float(np.mean(V)), color=COLORS["ink"], scale=3)   # velocity at the port
    ax.set_title(f"Ex. 3.1, $t'$ = {tp:.2f} s"); ax.set_aspect("equal")
Us, Vs = -Y, X                                                # solid-body rotation for comparison
axs[3].streamplot(X, Y, Us, Vs, color=COLORS["teal"], density=0.8)
axs[3].set_title("solid-body rotation"); axs[3].set_aspect("equal")
for ax in axs: ax.set_xlabel("$x$ [m]")
axs[0].set_ylabel("$y$ [m]")
savefig(fig, "ch03", "streamlines_ex31"); plt.show()

**What you see.** Parallel teal lines that turn by 45° from panel to panel; circles in the last panel.

**How to read it.** Ex. 3.1's velocity is the same at every point at a given instant, so its streamlines are parallel straight lines;
only their direction changes with time — the pattern turns once per period $2\pi/\omega$. Solid-body rotation is steady:
its circles never change.

**What would change if…** …the flow were steady (freeze ωt′): all three panels would be identical — and so, C04 shows, would path and streak lines.

In [ ]:
def lines(tp):                                               # the three Ex. 3.1 curves through the origin at t' [s]
    d = ch03.example_3_1(tp, xi0=1.0, omega=1.0, n=120)      # closed forms (D04, D05)
    return {"streamline": tuple(d["streamline"]), "path line": tuple(d["pathline"]), "streak line": tuple(d["streakline"])}   # (x, y) each

fig = slider_figure(lines, "t'", np.linspace(0, 2*np.pi, 24 if not FAST else 12), unit="s", xlabel="x [m]",
                    ylabel="y [m]", xrange=[-2.5, 2.5], yrange=[-2.5, 2.5], height=520,          # fixed axes [m]
                    title="Streamlines turn with t'; path and streak lines are circles that roll round the port")
recolor(fig, {"streamline": COLORS["teal"], "path line": COLORS["orange"], "streak line": COLORS["rose"]})   # house colours
fig.update_yaxes(scaleanchor="x", scaleratio=1)              # equal scales, so circles look round
fig.show()                                                   # draw it

**What does the code above do?**

1. `example_3_1(t′)` returns the closed-form streamline $y=x\tan\omega t'$ and the two circles of D04 and D05 at the
   drawing instant t′.
2. `slider_figure` precomputes them for 24 instants over one period.

**What you see.** A teal line and two circles, orange and rose, all passing through the origin.

**How to read it.** At every t′ the teal line touches both circles at the origin — the tangency shown in D05. The circles sit on opposite sides of the port.

**What would change if…** …drag t′ by a quarter period, π/2 ≈ 1.57 s: all three turn by 90° together.

📝 **Note.** **Stream tube** `N15` — The streamlines through every point of a closed curve C form a tube (Fig. 3.6). No fluid crosses its wall, because the
wall is everywhere tangent to $\mathbf u$; in a steady flow the same volume per second passes every cross-section.
Number below: the axisymmetric straining flow $\mathbf u=(-x/2,-y/2,z)$ s⁻¹ ($\nabla\cdot\mathbf u=0$) with a tube
seeded on a circle of radius 0.5 m at z = 1 m: the tube narrows as $R(z)=0.5\sqrt{1/z}$ and the flux through every
section is $\pi/4=0.785$ m³/s.

In [ ]:
u3 = lambda x, t: np.array([-x[0]/2, -x[1]/2, x[2]])          # axisymmetric strain [1/s × m]: ∇·u = −½ − ½ + 1 = 0
nl = 12 if not FAST else 8                                    # number of streamlines in the tube
fig = go.Figure()
for k, ph in enumerate(np.linspace(0, 2*np.pi, nl, endpoint=False)):   # seeds on the circle R = 0.5 m at z = 1 m
    c = ch03.streamline(u3, [0.5*np.cos(ph), 0.5*np.sin(ph), 1.0], 0.0, s_max=1.3)   # (3.7), both directions
    fig.add_trace(go.Scatter3d(x=c[0], y=c[1], z=c[2], mode="lines", line=dict(color=COLORS["teal"], width=4),
                               showlegend=(k == 0), name="streamlines of the tube wall"))   # 3-D lines (Ch. 2 P64)
for z0, R0, col in ((1.0, 0.5, COLORS["blue"]), (2.0, 0.5/np.sqrt(2), COLORS["orange"])):   # two cross-sections
    ph = np.linspace(0, 2*np.pi, 41)
    xs = np.r_[0, R0*np.cos(ph)]; ys = np.r_[0, R0*np.sin(ph)]; zs = np.full(42, z0)   # centre + rim points
    fig.add_trace(go.Mesh3d(x=xs, y=ys, z=zs, i=np.zeros(40, int), j=np.arange(1, 41), k=np.arange(2, 42),
                            color=col, opacity=0.45, name=f"section at z = {z0:g} m"))   # a fan of triangles = a disc
q1 = ch03.flux_through_disc(u3, [0, 0, 1], [0, 0, 1], 0.5)             # ∫ u·n dA through the lower section [m³/s]
q2 = ch03.flux_through_disc(u3, [0, 0, 2], [0, 0, 1], 0.5/np.sqrt(2))  # … through the upper, narrower section
print(f"flux through z = 1 m: {q1:.4f} m^3/s, through z = 2 m: {q2:.4f} m^3/s (pi/4 = {np.pi/4:.4f})")
fig.update_layout(height=480, title="A stream tube narrows where the flow speeds up; the flux stays the same",
                  scene=dict(aspectmode="data", xaxis_title="x [m]", yaxis_title="y [m]", zaxis_title="z [m]"))
fig.show()

**What does the code above do?**

1. Twelve streamlines started on a circle form the wall of a tube.
2. Two discs (drawn as fans of triangles) cut the tube at z = 1 m and z = 2 m, where its radius is 0.5 and 0.354 m.
3. `flux_through_disc` integrates $\mathbf u\cdot\mathbf n\,dA$ over each disc (midpoint rule): both 0.7854 m³/s.

**What you see.** Teal lines converging upward into a narrower tube; a blue disc below and a smaller orange disc above.

**How to read it.** The tube narrows where the flow speeds up along z; the two discs carry the same flux because the wall lets nothing through.

**What would change if…** …the seed circle were wider (radius 1 m): a fatter tube carrying four times the flux (∝ πR²), still the same at every section.

**What would change if…** …you followed one particle instead of freezing the clock? In a steady flow it would run along a streamline. In Ex. 3.1
the arrows turn while the particle moves, so its track is not any of the straight teal lines — it is a circle (C04).

### 🧩 Path lines and streak lines: where one particle goes, where the dye sits (3.8) `C04`

*The question:* Dye leaks steadily from a port on the floor of a sloshing tank, and you track one speck of it. Why does the photograph
of the dye not show the path of the speck?

*In one line:* $d\mathbf r/dt=\mathbf u(\mathbf r,t)$ *(3.8)*

#### The problem in plain words

Every flow picture you see in a lab is one of three things: a long exposure of one particle (a **path line**), a
snapshot of all the dye that came out of one port (a **streak line**), or a snapshot of directions (a **streamline**).
Near a beach, the water under long swell sloshes back and forth; a drop of dye released there and a floating leaf trace
quite different curves. We need both as computable objects.

#### The idea

| line | what is fixed | what varies along it | how you see it |
|---|---|---|---|
| streamline | the instant t | position (arc length s) | arrows at one instant |
| path line | the particle (label $\mathbf r_o$, $t_o$) | time t | long exposure of one speck |
| streak line | the port $\mathbf x_o$ and the instant t | release time $t_o$ | photo of continuously injected dye |

**Steady flow: all three are the same curve. Unsteady: three different curves.**

📝 **Note.** **Path line** `N16` — A **path line** is the trajectory of one particle of fixed identity, the Lagrangian $\mathbf x=\mathbf r(t;\mathbf r_o,t_o)$
of C01 drawn in space. Given only the Eulerian velocity field, we get it by solving the ODE below.

#### The maths — the path-line equation

Each particle moves with the velocity of the field at the place it currently occupies:

$$\frac{d\mathbf r}{dt}=[\mathbf u(\mathbf x,t)]_{\mathbf x=\mathbf r}=\mathbf u(\mathbf r,t),\qquad \mathbf r(t_o)=\mathbf r_o\qquad(3.8)$$

— an initial-value problem: three coupled ODEs, one per component. (A discretised version of (3.8) is how **particle
image velocimetry**, PIV, turns pairs of photographs of seeded particles into velocities.)

📝 **Note.** **Streak line** `N17` — A **streak line** through the port $\mathbf x_o$ at time t is the set of particles that passed through $\mathbf x_o$
at earlier times $t_o$: solve $d\mathbf r/dt=\mathbf u(\mathbf r,t)$ *(Eq. 3.8)* once per release time with
$\mathbf r(t_o)=\mathbf x_o$, then plot all positions at the same t. The parameter along it is the release time,
$x_i=r_i(t;\mathbf x_o,t_o)$. Sometimes $t_o$ can be eliminated to give one equation for the curve (D05).

📝 **Note.** **Ex. 3.1** `N18` — (a spatially uniform, time-periodic flow — a caricature of the back-and-forth motion under long surface waves, not a
wave solution; real wave orbits shrink with depth, Ch. 7): $u=\omega\xi_o\cos\omega t$, $v=\omega\xi_o\sin\omega t$.
Find the three lines through the origin at the instant t = t′. ⚠️ Four names that look alike: **t′** the drawing
instant, **t_o** a release time, **c_x, c_y** integration constants (the book's x_o, y_o), **ξ_o** the amplitude [m].
The streamline comes straight from $dx/u=dy/v=dz/w$ *(Eq. 3.7)*, integrated through the origin (the constant is 0):

$$ \frac{dy}{dx}=\frac{v}{u}=\frac{\omega\xi_o\sin(\omega t')}{\omega\xi_o\cos(\omega t')}=\tan(\omega t')\ \Rightarrow\ y=x\tan(\omega t')  $$

#### 🧮 Derivation — Ex. 3.1: the path line is a circle of radius ξ_o `D04`

**What we want to show.** Find the path of the particle that sits at the origin at the drawing instant t′ in the sloshing flow $u=\omega\xi_o\cos\omega t$, $v=\omega\xi_o\sin\omega t$ — and fill in the "little algebra" the book skips.

**Assumptions.** The field is spatially uniform (step 1: the right-hand sides depend on t only); ω ≠ 0.

**The plan.**

1. Write both components and integrate them (the field does not depend on position, so each equation integrates directly).
2. Fix the constants with the condition at t′.
3. Eliminate t with sin² + cos² = 1.

**Tools we use** (each explained before this point): (3.8), $d\mathbf r/dt=\mathbf u(\mathbf r,t)$ (C04) · integrating sin and cos (gloss in step 2) · eliminating a parameter with sin² + cos² = 1 (gloss in steps 6–7) · equation of a circle (gloss in step 7).

**We start from**

$$ \text{(3.8),}\ \dfrac{d\mathbf r}{dt}=\mathbf u(\mathbf r,t)\ \text{,}\ \mathbf r(t')=\mathbf 0 $$

*In words:* the particle moves with the field's velocity at its current position and passes the origin at t′.

---

**Step 1 of 7 — Write the two components.**

$$ \dfrac{dx}{dt}=\omega\xi_o\cos\omega t,\quad\dfrac{dy}{dt}=\omega\xi_o\sin\omega t $$

- *Why we can do this:* (3.8), $d\mathbf r/dt=\mathbf u(\mathbf r,t)$ component by component. The velocity is the same at every point, so the right sides are known functions of t alone — no coupling to x, y.
- *In words:* The particle's velocity is just the field's velocity at that time.

---

**Step 2 of 7 — Integrate each once.**

$$ x=\xi_o\sin\omega t+c_x,\quad y=-\xi_o\cos\omega t+c_y $$

- *Why we can do this:* An antiderivative of $\omega\cos\omega t$ is $\sin\omega t$ and of $\omega\sin\omega t$ is $-\cos\omega t$ (the ω cancels); $c_x,c_y$ are the constants (the book's x_o, y_o).
- *In words:* Every particle traces the same shape; only the constants differ.

---

**Step 3 of 7 — Impose r(t′) = 0.**

$$ c_x=-\xi_o\sin\omega t',\quad c_y=\xi_o\cos\omega t' $$

- *Why we can do this:* The particle we want is at the origin at the drawing instant: set t = t′ and x = y = 0 in step 2 and solve for the constants.
- *In words:* The condition picks one particle.

---

**Step 4 of 7 — Substitute the constants.**

$$ x=\xi_o(\sin\omega t-\sin\omega t'),\quad y=\xi_o(\cos\omega t'-\cos\omega t) $$

- *Why we can do this:* Put step 3 into step 2. These are the path's parametric equations, with t the parameter — the book prints them.
- *In words:* Where that particle is at any time t.

---

**Step 5 of 7 — Isolate the parts that depend on t.**

$$ x+\xi_o\sin\omega t'=\xi_o\sin\omega t,\quad y-\xi_o\cos\omega t'=-\xi_o\cos\omega t $$

- *Why we can do this:* The book skips this: move the constants to the left so each right side holds only t — the form in which squaring and adding removes t.
- *In words:* Offsets from a fixed point equal a sine and a cosine of the same angle.

---

**Step 6 of 7 — Square both and add.**

$$ (x+\xi_o\sin\omega t')^2+(y-\xi_o\cos\omega t')^2=\xi_o^2(\sin^2\omega t+\cos^2\omega t) $$

- *Why we can do this:* Squaring removes the minus sign; adding pairs sin² with cos² of the *same* angle ωt, ready for the identity.
- *In words:* The squared distance from the fixed point.

---

**Step 7 of 7 — Use sin² + cos² = 1.**

$$ (x+\xi_o\sin\omega t')^2+(y-\xi_o\cos\omega t')^2=\xi_o^2 $$

- *Why we can do this:* Pythagorean identity; t has gone, so this is the curve itself. $(x-a)^2+(y-b)^2=r^2$ is a circle of centre (a, b) and radius r.
- *In words:* A circle of radius ξ_o centred at $(-\xi_o\sin\omega t',\ \xi_o\cos\omega t')$.

---

**Result**

$$ (x+\xi_o\sin\omega t')^2+(y-\xi_o\cos\omega t')^2=\xi_o^2 $$

*In words:* the particle runs round a circle of radius ξ_o that touches the origin, once per period 2π/ω.

**What it means.** In a uniform but rotating velocity field every particle circles: a caricature of the orbital motion under long surface waves (Ch. 7 has the real ones, which shrink with depth). The circle does not coincide with the straight streamline — the flow is unsteady.

**Check it.** Units: every term m² ✓. The origin lies on it: $\xi_o^2\sin^2+\xi_o^2\cos^2=\xi_o^2$ ✓. Numbers (ξ_o = 1 m, ω = 1 s⁻¹, t′ = 0): centre (0, 1) m ✓. Direction: at t = 0 the particle is at the origin (the bottom of that circle) moving with u = (ωξ_o, 0) — to the right — so it runs counterclockwise ✓.

> ⚠️ **Common confusion (traps in `D04`):** Confusing t′ (drawing instant), t_o (release time, D05), c_x, c_y (constants) and ξ_o (amplitude). Squaring before isolating the t-terms (the constants mix in). Forgetting the direction of travel.

#### 🧮 Derivation — Ex. 3.1: the streak line is the mirror circle, and all three lines touch at the origin `D05`

**What we want to show.** Find where the dye that has been leaking from the origin sits at the instant t′, and prove the book's claim that the streamline, path line and streak line are tangent at the origin — both skipped in the book.

**Assumptions.** Dye has been released continuously for at least one period before t′ (step 6).

**The plan.**

1. Freeze the clock at t′ and let the release time vary.
2. Eliminate t_o as in D04.
3. Compare the three slopes at the origin.

**Tools we use** (each explained before this point): D04 · eliminating a parameter with sin² + cos² = 1 (gloss) · implicit differentiation (gloss in step 7) · the Ex. 3.1 streamline $y=x\tan(\omega t')$ (N18).

**We start from**

$$ \begin{array}{l}\text{D04 steps 2–3 with the release time}\ t_o\ \text{in place of t′:}\ x=\xi_o(\sin\omega t-\sin\omega t_o)\ \text{,} \\ y=\xi_o(\cos\omega t_o-\cos\omega t)\end{array} $$

*In words:* the path of the particle that was at the origin at time t_o.

---

**Step 1 of 8 — Label particles by their release time.**

$$ x=\xi_o(\sin\omega t-\sin\omega t_o),\ \ y=\xi_o(\cos\omega t_o-\cos\omega t) $$

- *Why we can do this:* Same integration as D04 steps 1–2, but the constants are fixed by $\mathbf r(t_o)=\mathbf 0$: every release time labels a different dye particle.
- *In words:* Where the dye released at t_o is at any later time t.

---

**Step 2 of 8 — Freeze the clock at t′.**

$$ x=\xi_o(\sin\omega t'-\sin\omega t_o),\ \ y=\xi_o(\cos\omega t_o-\cos\omega t') $$

- *Why we can do this:* A streak line is a snapshot at t = t′ of all dye released at earlier times $t_o\le t'$ (N17): now t_o is the curve's parameter.
- *In words:* The photograph of the dye at t′.

---

**Step 3 of 8 — Isolate the t_o-terms.**

$$ x-\xi_o\sin\omega t'=-\xi_o\sin\omega t_o,\quad y+\xi_o\cos\omega t'=\xi_o\cos\omega t_o $$

- *Why we can do this:* Move the known terms to the left; the parameter to be eliminated — t_o, not t — stays on the right.
- *In words:* Offsets from a fixed point, now in terms of the release angle.

---

**Step 4 of 8 — Square both and add.**

$$ (x-\xi_o\sin\omega t')^2+(y+\xi_o\cos\omega t')^2=\xi_o^2(\sin^2\omega t_o+\cos^2\omega t_o) $$

- *Why we can do this:* As in D04 step 6, squaring removes signs and pairs sin² with cos² of the same angle $\omega t_o$.
- *In words:* The squared distance of each dye particle from one fixed point.

---

**Step 5 of 8 — Use sin² + cos² = 1.**

$$ (x-\xi_o\sin\omega t')^2+(y+\xi_o\cos\omega t')^2=\xi_o^2 $$

- *Why we can do this:* Pythagorean identity, as in D04 step 7; the release time t_o has disappeared, so this is the curve itself.
- *In words:* A circle of radius ξ_o centred at $(\xi_o\sin\omega t',-\xi_o\cos\omega t')$ — the path circle reflected through the origin.

---

**Step 6 of 8 — Check which part is filled.**

$$ t'-2\pi/\omega\le t_o\le t'\ \Rightarrow\ \text{whole circle} $$

- *Why we can do this:* The book is silent here: as t_o runs over one period, $(\sin\omega t_o,\cos\omega t_o)$ goes once round; older dye repeats the same points because the flow is periodic.
- *In words:* After one period of leaking, the dye covers the full circle.

---

**Step 7 of 8 — Slope of the path circle at 0.**

$$ \left.\dfrac{dy}{dx}\right|_{\rm path}=-\dfrac{x+\xi_o\sin\omega t'}{y-\xi_o\cos\omega t'}\Big|_{\mathbf 0}=\tan\omega t' $$

- *Why we can do this:* Implicit differentiation of D04's circle: $2(x+\dots)dx+2(y-\dots)dy=0$; at the origin the ratio is $-\xi_o\sin\omega t'/(-\xi_o\cos\omega t')$.
- *In words:* The path leaves the origin at angle ωt′.

---

**Step 8 of 8 — Slope of the streak circle at 0.**

$$ \left.\dfrac{dy}{dx}\right|_{\rm streak}=-\dfrac{x-\xi_o\sin\omega t'}{y+\xi_o\cos\omega t'}\Big|_{\mathbf 0}=\tan\omega t' $$

- *Why we can do this:* Implicit differentiation of step 5 at the origin gives $\xi_o\sin\omega t'/(\xi_o\cos\omega t')$; the streamline $y=x\tan(\omega t')$ has the same slope.
- *In words:* All three curves touch at the origin, along the velocity there.

---

**Result**

$$ \begin{array}{l}(x-\xi_o\sin\omega t')^2+(y+\xi_o\cos\omega t')^2=\xi_o^2\ \text{, and the three lines share the slope tan ωt′ at the} \\ \text{origin}\end{array} $$

*In words:* the dye lies on the mirror image of the particle's circle; streamline, path line and streak line only touch.

**What it means.** A photograph of dye (streak line) and a long exposure of one speck (path line) are different curves in an unsteady flow, and neither is the instantaneous direction (streamline). They share only their tangent at the port — the velocity there at t′.

**Check it.** Units m² ✓. Numbers (ξ_o = 1, ωt′ = 0): centre (0, −1) — opposite the path centre (0, 1) ✓; the streak circle passes the origin ✓. `ch03.streakline` of 181 releases lies within 1e-7 of this circle ✓.

> ⚠️ **Common confusion (traps in `D05`):** Eliminating t instead of t_o. Thinking the streak line is only a partial arc (a full period of release fills it). Mixing up which circle is which: path centre (−ξ_o sin ωt′, ξ_o cos ωt′), streak centre (ξ_o sin ωt′, −ξ_o cos ωt′).

#### ✏️ Tiny example: Ex. 3.1 with ω = 1 s⁻¹, ξ_o = 1 m, t′ = 0

1. Streamline: $y=x\tan0=0$ — the x-axis.
2. Path line (D04): $(x+\sin0)^2+(y-\cos0)^2=1$, i.e. $x^2+(y-1)^2=1$: a unit circle centred at (0, 1).
3. Streak line (D05): $(x-\sin0)^2+(y+\cos0)^2=1$, i.e. $x^2+(y+1)^2=1$: a unit circle centred at (0, −1).
4. All three touch the origin with slope $\tan0=0$ (horizontal): the circles lie on opposite sides of the x-axis, as in
   the book's Fig. 3.7.
5. A quarter period later (t′ = π/2 s) everything has turned by 90°: the streamline is the y-axis, the centres are
   (−1, 0) and (1, 0).

> 📎 **Primer — RK4 by hand.** `P95` · The classical fourth-order Runge–Kutta step for $d\mathbf r/dt=\mathbf f(\mathbf r,t)$ samples the slope four times per
step: $k_1=f(r,t)$, $k_2=f(r+\tfrac{h}{2}k_1,t+\tfrac h2)$, $k_3=f(r+\tfrac h2k_2,t+\tfrac h2)$, $k_4=f(r+hk_3,t+h)$, then
$r\leftarrow r+\tfrac h6(k_1+2k_2+2k_3+k_4)$. Its error per unit time falls like $h^4$ (Euler's explicit step, Ch. 1 P30,
like h). The explainers use exactly this step in JavaScript.

In [ ]:
f = lambda y, t: -y                                  # dy/dt = −y, exact y = e^−t
y, h = 1.0, 0.1                                      # start value and step [s]
for n in range(10):                                  # ten steps to t = 1 s
    k1 = f(y, n*h); k2 = f(y + h/2*k1, n*h + h/2); k3 = f(y + h/2*k2, n*h + h/2); k4 = f(y + h*k3, n*h + h)
    y += h/6*(k1 + 2*k2 + 2*k3 + k4)                 # the weighted average of the four slopes
print(y, np.exp(-1.0))                               # 0.3678798 vs 0.3678794: 5e-7 after ten steps

In [ ]:
d = ch03.example_3_1(0.0, xi0=1.0, omega=1.0)                # closed forms at t' = 0 (D04, D05)
print(d["path_center"], d["streak_center"], d["slope"])      # centres (0, 1) and (0, −1) m, slope tan 0 = 0
T = 2*np.pi                                                  # the period 2π/ω [s]
ts = np.linspace(-T/2, T/2, 201)                             # half a period back and forward [s]
p = ch03.pathline(u31, [0.0, 0.0], 0.0, ts)                  # (3.8): dr/dt = u(r, t), r(0) = 0
print("path residual:", np.max(np.abs(np.hypot(p[0] - d["path_center"][0], p[1] - d["path_center"][1]) - 1.0)))
tr = np.linspace(-T, 0.0, 181 if not FAST else 91)           # dye released over the last period [s]
s = ch03.streakline(u31, [0.0, 0.0], 0.0, tr)                # positions at t' = 0 of every release
print("streak residual:", np.max(np.abs(np.hypot(s[0] - d["streak_center"][0], s[1] - d["streak_center"][1]) - 1.0)))

**What does the code above do?**

1. The closed forms of D04 and D05.
2. `pathline` solves $d\mathbf r/dt=\mathbf u(\mathbf r,t)$ *(3.8)* for one particle (t_eval may lie before $t_0$);
   every point is 1 m from the centre (0, 1).
3. `streakline` solves (3.8) once per release time (one vector ODE for the whole ensemble) and collects the positions at
   t′ = 0: a circle about (0, −1).

In [ ]:
r = np.array([0.0, 0.0]); h = T/400                          # start at the port; 400 steps per period
f = lambda r, t: u31(r, t)                                   # the right side of (3.8)
for n in range(200):                                         # 200 RK4 steps = half a period (primer P95)
    t_ = n*h
    k1 = f(r, t_); k2 = f(r + h/2*k1, t_ + h/2); k3 = f(r + h/2*k2, t_ + h/2); k4 = f(r + h*k3, t_ + h)
    r = r + h/6*(k1 + 2*k2 + 2*k3 + k4)
lib_end = ch03.pathline(u31, [0.0, 0.0], 0.0, [0.0, T/2])[:, -1]   # the library's answer at t = T/2
assert np.allclose(r, lib_end, atol=1e-8)                    # hand-written RK4 = library
assert np.allclose(r, [np.sin(T/2), 1 - np.cos(T/2)], atol=1e-8)   # = D04's circle (x = sin t, y = 1 − cos t)
print("RK4 by hand:", r, " library:", lib_end)

Twenty lines of arithmetic reproduce the library to 10⁻⁸ — no magic inside `pathline`.

📝 **Note.** **Fig. 3.7, our drawing** `N19` — the three lines of Ex. 3.1 at ωt′ = 30°, computed from the closed forms (curves) and by the ODE solvers (dots).

In [ ]:
from scripts.ch03_drawings import flow_lines_figure          # drawing helper (no physics inside)
fig, ax, info = flow_lines_figure(t_prime=np.pi/6, xi0=1.0, omega=1.0)   # ωt' = 30°
ax.plot([-2, 2], [-2*np.tan(np.pi/6), 2*np.tan(np.pi/6)], ":", color=COLORS["muted"], lw=1)   # the common tangent
print({k: f"{v:.1e} m" for k, v in info.items()})            # numerical vs closed form: max distance
savefig(fig, "ch03", "fig3_7_flow_lines"); plt.show()

**What you see.** One straight teal line and two circles of equal radius on opposite sides, all touching at the origin (dots: the numerical curves).

**How to read it.** Same point, same instant, three different curves — because the flow is unsteady. The common tangent at the
origin (grey dotted) is the velocity there at t′ (D05's last steps). The path circle is run counterclockwise.

**What would change if…** …t′ larger by π/(2ω): the whole picture turns by 90° counterclockwise; the circles keep their radius ξ_o.

In [ ]:
from scripts.ch03_drawings import flow_lines_animation       # dye + one particle + turning streamline over a period
anim = flow_lines_animation(frames=60 if not FAST else 30, xi0=1.0, omega=1.0, n_dye=60)   # one period of Ex. 3.1
show_animation(anim, player="video")                         # smooth MP4 (P16)

**What does the code above do?**

1. For each frame the streamline through the port is redrawn at the current instant (teal), the tagged particle
   released at t = 0 advances along its path line (orange), and all dye released so far is placed with `streakline` (rose).

**What you see.** A teal line turning steadily, an orange particle drawing a circle directly above the port, and rose dye on a circle that swings round the port.

**How to read it.** The tagged particle (released at t = 0) keeps to its own circle, centred at (0, 1) m directly above the port. The
dye lies on the mirror image of the path circle of *the particle at the port now* (D05): its centre
$(\xi_o\sin\omega t,\,-\xi_o\cos\omega t)$ goes once round the port per period, so at t = T/2 the dye circle coincides
with the tagged particle's circle. After one period the particle is back at the port and the pattern repeats.

**What would change if…** …there were a mean current (the explainer's '+ current' mode): the particle would not come back; its path becomes a
looping curve drifting downstream and the dye forms a wave.

📝 **Note.** **Steady flow ⇒ the three lines coincide** `N13` — If $\mathbf u$ does not depend on t, a particle at a point moves along the streamline through it and every later
particle from the same port follows the same track. Check on the steady hyperbolic flow $\mathbf u=(x,-y)$ s⁻¹ from the
port (1, 1) m, whose streamlines are the hyperbolas $xy=$ const:

In [ ]:
us = ch03.preset_field("rotating_strain", s=1.0, Omega=0.0)        # Ω = 0: the steady strain u = (x, −y) [1/s × m]
x0 = [1.0, 1.0]                                                    # the port [m]
c_stream = ch03.streamline(us, x0, 0.0, s_max=1.0)                 # (3.7) at t = 0
c_path = ch03.pathline(us, x0, 0.0, np.linspace(0, 1, 50))         # (3.8) for the particle leaving at t = 0
c_streak = ch03.streakline(us, x0, 1.0, np.linspace(0, 1, 50))     # dye released 0…1 s, seen at t = 1 s
for name, c in (("streamline", c_stream), ("path line", c_path), ("streak line", c_streak)):
    print(f"{name:12s} max |xy − 1| = {np.max(np.abs(c[0]*c[1] - 1)):.1e}")   # distance from the hyperbola xy = 1

**What does the code above do?**

In a steady flow all three curves lie on the same hyperbola $xy=1$ (residuals ≈ 10⁻⁹). Flow-visualisation atlases (Van
Dyke's *Album of Fluid Motion*) are full of such pictures; in unsteady flow you must know which of the three a
photograph shows.

#### 🎮 Interactive: Three lines through one point — why do they disagree?

**Why interactive:** a still picture shows three curves at one instant; the running clock shows *how* they are made — the streamline redrawn, the particle's trail growing, the dye laid down — and lets you switch unsteadiness off.
A clock runs: the streamline pattern is redrawn every instant, one particle leaves its trail and a port leaks dye. You
change the period, the amplitude and a mean current, or switch to a steady flow — and watch the three curves separate
or collapse onto one.

**What to try:**
- Press ▶ with the Ex. 3.1 preset and stop at ωt = 30°: compare with our Fig. 3.7 above.
- Choose 'steady': the three curves become one line — the N13 statement as an experiment.
- Add a mean current U₀ = 0.5 m/s: the particle no longer returns to the port; the dye forms a wave.
- Click a dye particle (inspector) to see when it left the port and the path it took.

In [ ]:
show_viz("ch03", "flow_lines_unsteady")   # full-width explainer; ⤢ Full screen for more room

**What would change if…** …you watched Ex. 3.1 from a boat drifting with a mean current? The streamlines would look different again — whether a
flow is steady depends on the observer. The next block asks whether anything important depends on the observer: the
particle's acceleration does not.

### 🧩 Galilean invariance: the acceleration does not care who is watching (3.9) `C05`

*The question:* A cylinder is towed through a still lake; a pier stands in a steady river. One flow is unsteady, the other steady, yet
they are the same flow seen from two boats. Does a water particle's acceleration depend on the boat you watch it from?

*In one line:* $\frac{\partial\mathbf u}{\partial t}+(\mathbf u\cdot\nabla)\mathbf u=\frac{\partial\mathbf u'}{\partial t'}+(\mathbf u'\cdot\nabla')\mathbf u'$ *(3.9)*

#### The problem in plain words

Wind-tunnel engineers hold a model still and blow air past it; the real car moves through still air. Oceanographers
describe a wave in a frame moving with the crest, where the flow is steady. All of this is legitimate only if Newton's
law — force equals mass times *acceleration* — reads the same in both frames. We check that the acceleration, written as
local + advective, is frame-independent for observers moving at constant velocity.

#### The idea

```
body frame (pier fixed):   ∂u/∂t = 0 (steady)          (u·∇)u ≠ 0     ─┐
fluid frame (lake fixed):  ∂u'/∂t' ≠ 0 (unsteady)      (u'·∇')u'       ├─ same particle, same total Du/Dt
only U (constant) separates them: the split moves, the sum does not     ─┘
```

> 📎 **Primer — frames of reference and relative velocity.** `P96` · A frame of reference is a set of axes plus a clock. Two frames whose axes stay parallel and whose origins separate at a
constant velocity $\mathbf U$ are related by a *Galilean transformation*: positions differ by $\mathbf Ut$ (+ a fixed
offset), velocities by $\mathbf U$, and both use the same clock (no relativity here). A frame that rotates is different:
its points move with $\boldsymbol\Omega\times\mathbf x$, which is not uniform.

In [ ]:
U = np.array([2.0, 0.0])                             # frame O' moves at 2 m/s along x
u_lab = np.array([2.5, 0.3])                         # a particle's velocity seen from O [m/s]
print(u_lab - U)                                     # [0.5 0.3]: the same particle seen from O'

📝 **Note.** **Fig. 3.2, one flow in two frames** `N04` — Ideal flow past a circular cylinder of radius a in a stream U (a uniform stream plus a *doublet*; derived in Ch. 6
§6.3): $u=U[1-a^2(x^2-y^2)/r^4]$, $v=-2Ua^2xy/r^4$. Seen from the cylinder the streamlines are fixed (steady). Seen from
the still fluid, $\mathbf u'=\mathbf u-\mathbf U$: streamlines start and end on the moving body and change as it passes
(unsteady). The two velocity fields differ by the constant vector $\mathbf U$:

$$ \mathbf u=\mathbf U+\mathbf u'  $$

In [ ]:
x = np.linspace(-3, 3, 121); y = np.linspace(-2, 2, 81); X, Y = np.meshgrid(x, y)   # 121 × 81 grid [m]
th = np.linspace(0, 2*np.pi, 100)                            # to draw the cylinder's outline
P_ = (0.0, 1.5)                                              # the tagged point P [m]
fig, axs = plt.subplots(1, 2, figsize=(11, 3.9))             # (a) body frame, (b) fluid frame
for ax, frame, Uf, ttl in zip(axs, ("body", "fluid"), (1.0, 0.0), ("(a) body frame: steady", "(b) fluid frame at t = 0: unsteady")):
    Uu, Vv = ch03.cylinder_flow(X, Y, 1.0, 1.0, frame=frame)   # U = 1 m/s, a = 1 m; NaN inside the body
    ax.streamplot(X, Y, np.ma.masked_invalid(Uu), np.ma.masked_invalid(Vv), color=COLORS["teal"], density=1.1, linewidth=0.9)   # NaN masked
    ax.fill(np.cos(th), np.sin(th), color=COLORS["grid"], zorder=3)   # the cylinder
    r = ch03.frame_acceleration_terms(*P_, 1.0, 1.0, Uf)      # velocity and acceleration at P in this frame
    ax.quiver(*P_, *r["u"], color=COLORS["ink"], scale=4, width=0.006, zorder=4, label="velocity at P")   # black arrow
    ax.quiver(*P_, *r["total"], color=COLORS["accent"], scale=4, width=0.008, zorder=4,
              label=f"acceleration ({r['total'][0]:.3f}, {r['total'][1]:.3f}) m/s²")   # purple arrow, same scale
    ax.set_aspect("equal"); ax.set_title(ttl); ax.set_xlabel("$x$ [m]"); ax.legend(fontsize=7, loc="lower left")   # equal axes
axs[0].set_ylabel("$y$ [m]")                                  # shared y label
savefig(fig, "ch03", "cylinder_two_frames"); plt.show()       # save to outputs/ch03 and draw

**What you see.** Two completely different streamline pictures and one identical purple arrow at P = (0, 1.5) m.

**How to read it.** (a) is steady — the arrows never change at a fixed point; (b) is unsteady — the body moves through the water
(towards −x) and the loops leave its front and re-enter its back. The black velocity arrows differ by exactly U; the
purple acceleration arrow, (0, −0.856) m/s², is the same in both.

**What would change if…** …the observer moved at half the towing speed: a third streamline picture, the same purple arrow (the observer slider below and the Galilean-frames explainer).

📝 **Note.** **The Galilean transformation (Fig. 3.8)** `N21` — Frame O′x′y′z′ moves at constant velocity $\mathbf U$ relative to Oxyz with parallel axes; $\mathbf x'_o$ is the vector
from O to O′ at t = 0. `ch03.galilean_transform(u, U)` returns the primed field
$\mathbf u'(\mathbf x',t')=\mathbf u(\mathbf x'+\mathbf Ut'+\mathbf x'_o,t')-\mathbf U$.

$$ \mathbf u(\mathbf x,t)=\mathbf U+\mathbf u'(\mathbf x',t'),\qquad t=t',\qquad \mathbf x=\mathbf x'+\mathbf Ut+\mathbf x'_o  $$

> 📎 **Primer — chain rule with a moving frame.** `P97` · If $g(x,t)=f(x-Ut,t)$, then at fixed x,
$\frac{\partial g}{\partial t}=\frac{\partial f}{\partial t'}-U\frac{\partial f}{\partial x'}$: holding x fixed while t
advances means the moving coordinate $x'=x-Ut$ *decreases*. A time derivative "at fixed position" depends on whose
position is held fixed — the classic trap of this section.

In [ ]:
x, t, U = sp.symbols('x t U')                        # lab position, time, frame speed
xp, tp = sp.symbols('xp tp')                         # moving-frame coordinates x', t'
f = sp.sin(xp)*sp.exp(-tp)                           # any f(x', t')
g = f.subs({xp: x - U*t, tp: t})                     # the same field seen in the unprimed frame
rhs = (sp.diff(f, tp) - U*sp.diff(f, xp)).subs({xp: x - U*t, tp: t})   # ∂f/∂t' − U ∂f/∂x'
print(sp.simplify(sp.diff(g, t) - rhs))              # 0

#### 🧮 Derivation — Galilean invariance of the acceleration, Eq. (3.9) `D06` (Eq. 3.9)

**What we want to show.** Show that local + advective acceleration is the same in two frames moving at constant relative velocity, although each term separately is not — the book leaves it to Exercise 3.12 ("it can be shown").

**Assumptions.** U constant in time (step 3: $\partial\mathbf U/\partial t=0$) and in space (step 5); axes parallel and not rotating (step 2: $\partial x'_j/\partial x_k=\delta_{jk}$); one common clock.

**The plan.**

1. Express the primed coordinates through the unprimed ones and differentiate them.
2. Chain rule for the local term (the trap).
3. Chain rule for the space derivatives (they do not change).
4. Add: the two U-terms cancel.

**Tools we use** (each explained before this point): Galilean transformation (N21) · chain rule with a moving frame (primer in C05) · $\partial x_j/\partial x_k=\delta_{jk}$ (Ch. 2 §2.7) · the acceleration $\frac{D\mathbf u}{Dt}=\frac{\partial\mathbf u}{\partial t}+(\mathbf u\cdot\nabla)\mathbf u$ (D02 step 9) · summation convention.

**We start from**

$$ \mathbf u(\mathbf x,t)=\mathbf U+\mathbf u'(\mathbf x',t')\ \text{,}\ t=t'\ \text{,}\ \mathbf x=\mathbf x'+\mathbf Ut+\mathbf x'_o\ \text{, with U and}\ \mathbf x'_o\ \text{constant} $$

*In words:* the primed frame moves at constant velocity U; velocities differ by U, positions by Ut plus a fixed offset, the clocks agree.

---

**Step 1 of 9 — Invert the transformation.**

$$ x'_j=x_j-U_jt-x'_{o,j},\qquad t'=t $$

- *Why we can do this:* The field u′ is known in primed coordinates; to differentiate u with respect to x and t we need how x′, t′ depend on them — solve the start line for x′.
- *In words:* A primed coordinate is the unprimed one minus how far O′ has moved.

---

**Step 2 of 9 — Differentiate the primed coordinates.**

$$ \dfrac{\partial x'_j}{\partial t}=-U_j,\quad\dfrac{\partial x'_j}{\partial x_k}=\delta_{jk},\quad\dfrac{\partial t'}{\partial t}=1 $$

- *Why we can do this:* Differentiate step 1 term by term: U and $\mathbf x'_o$ are constants, $\partial x_j/\partial x_k=\delta_{jk}$ (Ch. 2 §2.7), and t′ does not depend on x. These are the chain-rule factors.
- *In words:* At a fixed lab point, the primed coordinate slides backwards at U.

---

**Step 3 of 9 — Chain rule for the local term.**

$$ \dfrac{\partial u_i}{\partial t}=\dfrac{\partial u'_i}{\partial t'}-U_j\dfrac{\partial u'_i}{\partial x'_j} $$

- *Why we can do this:* $u_i=U_i+u'_i(x',t')$ with U constant; t enters u′ through t′ (factor 1) and through each $x'_j$ (factor $-U_j$) — the moving-frame chain rule (primer). The book skips this.
- *In words:* A fixed lab probe sees the primed pattern slide past.

---

**Step 4 of 9 — Chain rule for space derivatives.**

$$ \dfrac{\partial u_i}{\partial x_k}=\dfrac{\partial u'_i}{\partial x'_j}\delta_{jk}=\dfrac{\partial u'_i}{\partial x'_k} $$

- *Why we can do this:* x enters only through x′ with factor $\delta_{jk}$ (step 2); the Kronecker delta substitutes its index. So $\nabla=\nabla'$ for a pure translation.
- *In words:* Slopes in space are the same in both frames.

---

**Step 5 of 9 — Write the advective term.**

$$ u_k\dfrac{\partial u_i}{\partial x_k}=(U_k+u'_k)\dfrac{\partial u'_i}{\partial x'_k} $$

- *Why we can do this:* Substitute $u_k=U_k+u'_k$ (the transformation) and step 4 into $(\mathbf u\cdot\nabla)u_i$.
- *In words:* The lab advects with the full velocity, U included.

---

**Step 6 of 9 — Expand the product.**

$$ u_k\dfrac{\partial u_i}{\partial x_k}=U_k\dfrac{\partial u'_i}{\partial x'_k}+u'_k\dfrac{\partial u'_i}{\partial x'_k} $$

- *Why we can do this:* Distribute the sum over k. The first piece is the extra advection by the frame speed; we expect it to meet step 3's extra term.
- *In words:* Advective = advection by U + advection by u′.

---

**Step 7 of 9 — Add local and advective terms.**

$$ \dfrac{\partial u_i}{\partial t}+u_k\dfrac{\partial u_i}{\partial x_k}=\dfrac{\partial u'_i}{\partial t'}-U_j\dfrac{\partial u'_i}{\partial x'_j}+U_k\dfrac{\partial u'_i}{\partial x'_k}+u'_k\dfrac{\partial u'_i}{\partial x'_k} $$

- *Why we can do this:* The left side is $Du_i/Dt$ in Oxyz (D02 step 9); add step 3 and step 6.
- *In words:* Four terms: two of them contain U.

---

**Step 8 of 9 — Cancel the two U-terms.**

$$ \dfrac{\partial u_i}{\partial t}+u_k\dfrac{\partial u_i}{\partial x_k}=\dfrac{\partial u'_i}{\partial t'}+u'_k\dfrac{\partial u'_i}{\partial x'_k} $$

- *Why we can do this:* j and k are dummy indices, so $U_j\,\partial u'_i/\partial x'_j$ and $U_k\,\partial u'_i/\partial x'_k$ are the same sum with opposite signs.
- *In words:* What the lab calls 'local' and the frame's advection trade places and cancel.

---

**Step 9 of 9 — Return to vector notation.**

$$ \dfrac{\partial\mathbf u}{\partial t}+(\mathbf u\cdot\nabla)\mathbf u=\dfrac{\partial\mathbf u'}{\partial t'}+(\mathbf u'\cdot\nabla')\mathbf u' $$

- *Why we can do this:* Summation convention read backwards: the index line for each i is the vector equation (3.9), $\frac{\partial\mathbf u}{\partial t}+(\mathbf u\cdot\nabla)\mathbf u=\frac{\partial\mathbf u'}{\partial t'}+(\mathbf u'\cdot\nabla')\mathbf u'$.
- *In words:* The particle's acceleration is the same for both observers.

---

**Result**

$$ \begin{array}{l}\frac{\partial\mathbf u}{\partial t}+(\mathbf u\cdot\nabla)\mathbf u=\left(\frac{D\mathbf u}{Dt}\right)_{Oxyz}=\left(\frac{D\mathbf u'}{Dt'}\right)_{O'x'y'z'}=\frac{\partial\mathbf u'}{\partial t'}+(\mathbf u'\cdot\nabla')\mathbf u' \\ \text{(3.9)}\end{array} $$

*In words:* observers in uniform relative motion agree on every particle's acceleration.

**What it means.** Newton's law can be written in any frame moving steadily — wind tunnels, wave frames and moving coordinate systems are legitimate. The split into 'unsteady' and 'advective' is a choice of observer. It fails for accelerating or rotating frames: then step 3 gains −dU/dt, or U = Ω × x varies in space and step 4 changes — Coriolis and centrifugal terms (Ch. 4 §4.7).

**Check it.** Units m/s² ✓. U = 0: identity ✓. Numbers: lab sine wave local −0.707 + advective 0.832 = 0.125 m/s², wave frame 0 + 0.125 ✓; cylinder at P = (0, 1.5) m: fluid frame (0, −0.593) + (0, −0.263) = body frame (0, −0.856) ✓.

In [ ]:
import sympy as sp                                        # symbolic algebra
x, y, t, U1, U2 = sp.symbols('x y t U_1 U_2')             # lab coordinates, time, frame velocity
xp, yp, tp = sp.symbols('xp yp tp')                       # primed coordinates
f, g = sp.Function('f'), sp.Function('g')                 # u' = (f, g): any smooth primed field
up = [f(xp, yp, tp), g(xp, yp, tp)]                       # u'(x', t')
sub = {xp: x - U1*t, yp: y - U2*t, tp: t}                 # x' = x − Ut (offset 0), t' = t
u = [U1 + up[0].subs(sub), U2 + up[1].subs(sub)]          # u = U + u'  (step 0: the transformation)
a_lab = [sp.diff(q, t) + u[0]*sp.diff(q, x) + u[1]*sp.diff(q, y) for q in u]   # ∂u/∂t + (u·∇)u
a_pr = [(sp.diff(q, tp) + up[0]*sp.diff(q, xp) + up[1]*sp.diff(q, yp)).subs(sub) for q in up]  # primed, same point
# f, g are unknown functions and sympy cannot always simplify their derivatives, so test on
# generic cubic polynomials (any smooth field is locally one) — the difference then expands to exactly 0
cs = sp.symbols('c0:20'); X, Y, T = sp.symbols('X Y T')   # 20 coefficients and three dummy arguments
mons = [X**i * Y**j * T**k for i in range(4) for j in range(4) for k in range(4) if i + j + k <= 3]   # the 20 monomials of degree ≤ 3
repl = {f: sp.Lambda((X, Y, T), sum(c * m for c, m in zip(cs, mons))),   # f → a generic cubic
        g: sp.Lambda((X, Y, T), sum(c * m for c, m in zip(cs[::-1], mons)))}   # g → another one
print([sp.expand((a - b).subs(repl).doit()) for a, b in zip(a_lab, a_pr)])   # [0, 0]: (3.9)

> ⚠️ **Common confusion (traps in `D06`):** Writing $\partial\mathbf u/\partial t=\partial\mathbf u'/\partial t'$ (forgets the $-\mathbf U\cdot\nabla'$ term — the classic slip). Forgetting that U also advects in step 5. Applying the result to a rotating frame.

📝 **Note.** **Back to Fig. 3.2** `N20` — In the body frame the unsteady term is zero but the streamlines bend, so particles still accelerate — all of it
advective. In the fluid frame both terms are nonzero. Because the frames differ by a constant $\mathbf U$,
$\frac{\partial\mathbf u}{\partial t}+(\mathbf u\cdot\nabla)\mathbf u=\frac{\partial\mathbf u'}{\partial t'}+(\mathbf u'\cdot\nabla')\mathbf u'$
*(Eq. 3.9)* guarantees the same acceleration at the same place relative to the cylinder.

📝 **Note.** **The split is observer-dependent, the sum is not** `N23` — the term bars below trade blue for amber as the observer's speed changes, while the purple total stays put.

📝 **Note.** **Two everyday pictures** `N24` — a traffic light that switches from east–west to north–south traffic is an *unsteady* (local) change at a fixed
crossing; a roller coaster on a fixed track gives its riders *advective* acceleration although the track never changes.

📝 **Note.** **Linear and quadratic** `N22` — The local term $\partial\mathbf u/\partial t$ is linear in $\mathbf u$; the advective term $(\mathbf u\cdot\nabla)\mathbf u$
is quadratic — doubling every velocity doubles the first and quadruples the second. This nonlinearity is the heart of
fluid mechanics (turbulence, waves steepening). When $\mathbf u$ is tiny it can be dropped (acoustics, Ch. 15); when
$\mathbf u=0$ only statics is left (Ch. 1 §1.7).

In [ ]:
f1 = ch03.cylinder_velocity_field(1.0, 1.0, U_frame=0.0)       # the fluid-frame cylinder flow u(x, t) [m/s]
f2 = lambda x, t: 2*f1(x, t)                                   # every velocity doubled
Pp = np.array([0.0, 1.5])                                      # the point P [m]
a1, a2 = ch03.acceleration(f1, Pp, 0.0), ch03.acceleration(f2, Pp, 0.0)   # (a, local, advective) by stencils
print(round(a2.local[1]/a1.local[1], 6), round(a2.advective[1]/a1.advective[1], 6))   # 2.0 and 4.0

**What does the code above do?**

The scaling test in two numbers: the local term doubles, the advective term quadruples.

#### ✏️ Tiny example: a travelling sine wave, two frames

In a frame moving with the wave the flow is steady: $u'=0.5\sin x'$ m/s (k = 1 m⁻¹). The lab sees the same pattern
moving at U = 2 m/s: $u=2+0.5\sin(x-2t)$. At the lab point $x-2t=\pi/4$:

1. Wave frame: local 0; advective $u'\,du'/dx'=0.5\sin\frac\pi4\times0.5\cos\frac\pi4=0.3536\times0.3536=0.125$ m/s².
2. Lab frame: local $\partial u/\partial t=-2\times0.5\cos\frac\pi4=-0.707$ m/s²; advective
   $u\,\partial u/\partial x=(2+0.354)\times0.354=0.832$ m/s².
3. Sum $-0.707+0.832=0.125$ m/s² — the same, as
   $\frac{\partial\mathbf u}{\partial t}+(\mathbf u\cdot\nabla)\mathbf u=\frac{\partial\mathbf u'}{\partial t'}+(\mathbf u'\cdot\nabla')\mathbf u'$
   *(Eq. 3.9)* says.
4. The cylinder at P = (0, 1.5) m (U = a = 1): body frame local 0, advective −0.856; fluid frame local −0.593, advective
   −0.263 (y-components, m/s²) — again −0.856 in total.

In [ ]:
U = np.array([2.0])                                            # the wave moves at 2 m/s
up = lambda xp, tp: 0.5*np.sin(xp)                             # steady in the wave frame: u' = 0.5 sin x' [m/s]
u = lambda x, t: U + up(x - U*t, t)                            # the lab field u = U + u'(x − Ut)
print("lab:       ", ch03.acceleration(u, np.array([np.pi/4]), 0.0))               # (a, local, advective)
print("wave frame:", ch03.acceleration(ch03.galilean_transform(u, U), np.array([np.pi/4]), 0.0))
for Uf in (1.0, 0.5, 0.0):                                     # three observers: body, half-way, fluid [m/s]
    r = ch03.frame_acceleration_terms(0.0, 1.5, 1.0, 1.0, Uf)  # at P = (0, 1.5) m, U = a = 1
    print(f"U_frame = {Uf}: local {np.round(r['local'], 3)}, advective {np.round(r['advective'], 3)}, total {np.round(r['total'], 3)}")

**What does the code above do?**

1. A field that is steady in the moving frame.
2. `acceleration` returns (a, local, advective) by second-order stencils in the lab: $(0.125,\,-0.707,\,0.832)$ m/s².
3. `galilean_transform` builds the primed field; the same call gives the wave-frame split $(0.125,\,0,\,0.125)$.
4. `frame_acceleration_terms` does it for the cylinder for three observers — the totals agree to 10⁻⁸.

In [ ]:
Ufs = np.array([0.0, 0.25, 0.5, 0.75, 1.0])                    # observer speeds: fluid frame → body frame [m/s]
R_ = [ch03.frame_acceleration_terms(0.0, 1.5, 1.0, 1.0, Uf) for Uf in Ufs]   # the (3.9) terms at P for each
loc = np.array([r["local"][1] for r in R_]); adv = np.array([r["advective"][1] for r in R_])   # y-components [m/s²]
fig, ax = plt.subplots(figsize=(7, 3.8))                       # one panel of stacked bars
ax.bar(Ufs, loc, width=0.18, color=COLORS["blue"], label=r"local $\partial v/\partial t$")               # blue: local
ax.bar(Ufs, adv, width=0.18, bottom=loc, color=COLORS["amber"], label=r"advective $(\mathbf{u}\cdot\nabla)v$")   # amber on top
ax.plot(Ufs, loc + adv, "D", color=COLORS["accent"], ms=9, label="total $Dv/Dt$")                          # purple: the sum
ax.axhline(-0.856, ls="--", color=COLORS["accent"], lw=1)      # the common total −0.856 m/s²
ax.set_xlabel("observer speed $U_{frame}$ [m/s]  (0 = still lake, 1 = cylinder)"); ax.set_ylabel("y-acceleration at P [m/s²]")
ax.set_title("The split moves with the observer; the total does not"); ax.legend(fontsize=8, loc="lower left")   # message title
savefig(fig, "ch03", "galilean_term_bars"); plt.show()         # save to outputs/ch03 and draw

**What you see.** Blue bars shrinking to zero as the observer approaches the body frame, amber bars growing, purple diamonds on one horizontal line.

**How to read it.** Reading left to right is moving the observer from the lake to the cylinder; the acceleration of the particle at P never changes — only how it is booked.

**What would change if…** …P were far from the body (P = (0, 5) m): all bars would shrink toward zero (the flow there is almost uniform).

📎 *Gloss — a stream function as a drawing tool.* For a plane flow with $\nabla\cdot\mathbf u=0$ there is a function
$\psi(x,y)$ (Ch. 6) with $u=\partial\psi/\partial y$, $v=-\partial\psi/\partial x$. Then
$\mathbf u\cdot\nabla\psi=\frac{\partial\psi}{\partial y}\frac{\partial\psi}{\partial x}-\frac{\partial\psi}{\partial x}\frac{\partial\psi}{\partial y}=0$:
ψ does not change along a streamline, so its contour lines *are* the streamlines. Adding $c\,y$ to ψ adds $u=c$, $v=0$
everywhere — a uniform flow along x. For the cylinder in the body frame $\psi=Uy(1-a^2/r^2)$
(`ch03.cylinder_streamfunction`); the observer moving at $U_{frame}$ sees the far fluid at $U_{frame}$ instead of $U$,
i.e. $\psi+(U_{frame}-U)\,y$. We use it below only to draw many streamlines cheaply; the cell checks both claims.

In [ ]:
xs_, ys_, U_, a_, c_ = sp.symbols('x y U a c', positive=True)       # position, stream speed, radius, a constant
psi = U_*ys_*(1 - a_**2/(xs_**2 + ys_**2))                           # body-frame ψ of the cylinder
print(sp.limit(sp.diff(psi, ys_), xs_, sp.oo), sp.simplify(sp.diff(psi + c_*ys_, ys_) - sp.diff(psi, ys_)))   # U far away; adding c·y adds u = c

In [ ]:
x = np.linspace(-3, 3, 151); y = np.linspace(-2.5, 2.5, 126); X, Y = np.meshgrid(x, y)   # drawing grid [m]
psi_body = ch03.cylinder_streamfunction(X, Y, 1.0, 1.0)        # body-frame ψ [m²/s] (NaN inside the cylinder)
levels = np.linspace(-1.5, 1.5, 13)                            # which streamlines to draw
th = np.linspace(0, 2*np.pi, 80)

def frame(Uf):                                                 # streamlines + body + P for observer speed Uf [m/s]
    psi = psi_body + (Uf - 1.0)*Y                              # ψ seen by this observer
    cs = plt.contour(X, Y, psi, levels=levels); segs = cs.allsegs; plt.close()   # contour lines as point lists
    xs, ys = [], []                                            # all contour pieces in one NaN-separated line
    for level in segs:                                         # one list of pieces per level
        for sgm in level:                                      # each piece is an (n, 2) array of points
            xs += list(sgm[:, 0]) + [np.nan]; ys += list(sgm[:, 1]) + [np.nan]
    return {"streamlines": (xs, ys), "cylinder": (np.cos(th), np.sin(th)), "point P": ([0.0], [1.5])}   # name → (x, y)

vals = np.linspace(0, 1, 21 if not FAST else 11)               # observer speeds [m/s]
fig = slider_figure(frame, "U_frame", vals, unit="m/s", xlabel="x [m]", ylabel="y [m]", xrange=[-3, 3],
                    yrange=[-2.5, 2.5], modes={"point P": "markers"}, height=480,
                    title="The observer's speed reshapes the streamlines (P's terms are in its legend entry)")
Ps = [tr for tr in fig.data if tr.name == "point P"]           # one P trace per slider step, in slider order
for tr, Uf in zip(Ps, vals):                                   # write that observer's (3.9) terms into P's name
    r = ch03.frame_acceleration_terms(0.0, 1.5, 1.0, 1.0, Uf)  # local, advective, total at P [m/s²]
    tr.name = f"P: local {r['local'][1]:.3f} + advective {r['advective'][1]:.3f} = {r['total'][1]:.3f} m/s²"
    tr.marker.size = 11; tr.marker.color = COLORS["accent"]    # a big purple dot
recolor(fig, {"streamlines": COLORS["teal"], "cylinder": COLORS["ink"]})   # house colours
fig.update_yaxes(scaleanchor="x", scaleratio=1)                # equal scales, so the cylinder is round
fig.show()                                                     # draw it

**What does the code above do?**

1. For each observer speed the stream function is shifted by $(U_{frame}-U)\,y$ and its contour lines (matplotlib's
   `contour`, Ch. 2 P78, used here only to get the curves) are passed to plotly.
2. The point P carries that observer's local, advective and total accelerations in its legend entry.

**What you see.** Loops leaving the front of the body and re-entering its back at U_frame = 0; streamlines flowing round the body at U_frame = 1 m/s.

**How to read it.** The loops of the fluid frame open into the flow-past-a-body picture as $U_{frame}\to U$; the numbers in P's label trade places while their sum stays −0.856 m/s².

**What would change if…** …U_frame went beyond U: the far fluid would stream past faster than U (still towards +x) and the cylinder itself would drift downstream (+x); the total at P would still be −0.856 m/s².

#### 🎮 Interactive: Steady or not — does the acceleration care?

**Why interactive:** the point is that one number stays fixed while everything else changes; you only believe it when you drag the observer yourself and watch the bars trade places around a purple total that never moves.
One slider moves the observer from the still fluid to the cylinder. The streamline picture morphs from "unsteady, loops
on the body" to "steady, around the body", while the acceleration arrow of a tagged particle stays exactly the same and
its two bars trade places —
$\frac{\partial\mathbf u}{\partial t}+(\mathbf u\cdot\nabla)\mathbf u=\frac{\partial\mathbf u'}{\partial t'}+(\mathbf u'\cdot\nabla')\mathbf u'$
*(Eq. 3.9)* as an experiment.

**What to try:**
- Drag the observer speed from 0 to U and watch the blue bar hand its share to the amber bar.
- Click any point: the inspector gives both frames' terms with numbers.
- Set the preset 'body frame': the status says steady — yet the purple arrow is not zero.
- Step through D06 in the Derivation tab; at the cancellation step the two U·∇′ bars light up.

In [ ]:
show_viz("ch03", "galilean_frames_cylinder")   # full-width explainer; ⤢ Full screen for more room

> ⚠️ **Common confusion:** "Galilean invariance means any moving frame is as good as any other." Only frames moving at
> *constant* velocity without rotating. In a rotating frame (the Earth!) the acceleration picks up Coriolis and
> centrifugal terms (Ch. 4 §4.7) and even the vorticity changes by 2Ω (N28 in C10) — the heart of geophysical fluid
> dynamics (Ch. 13).

**What would change if…** …the observer accelerated (U depends on t)? Step 3 of D06 gains $-d\mathbf U/dt$, which does not cancel: an
accelerating observer sees a fictitious force. Next, §3.4 zooms into a small neighbourhood of a point and asks what the
velocity *differences* do to a fluid element.

---

## 3.4 Strain and Rotation Rates

**What is this section about?** Zoom into a tiny blob of fluid. Its neighbours move at slightly different velocities;
those differences stretch it, shear it, swell it and spin it. Each of these motions is one piece of the
velocity-gradient tensor you met in Ch. 2 — here you *measure* them on a moving element.

> 🔁 **Recap — The split of the velocity gradient** (Ch. 2 §2.10). Any velocity gradient splits uniquely into a symmetric and an antisymmetric part:
$\frac{\partial u_i}{\partial x_j}=S_{ij}+\tfrac12R_{ij}$ *(Eq. 3.11)* (Ch. 2 proved the split is unique).
`ch03.strain_rate_tensor(G) + 0.5*ch03.rotation_tensor(G)` returns G exactly.

> 🔁 **Recap — The strain-rate tensor** (Ch. 2 §2.10). The symmetric part $S_{ij}=\tfrac12\Big(\frac{\partial u_i}{\partial x_j}+\frac{\partial u_j}{\partial x_i}\Big)$
*(Eq. 3.12)* — `ch03.strain_rate_tensor(G)`. §3.4 shows what each of its entries *measures*.

> 🔁 **Recap — The rotation tensor — the book's, without ½** (Ch. 2 §2.10). $R_{ij}=\frac{\partial u_i}{\partial x_j}-\frac{\partial u_j}{\partial x_i}$ *(Eq. 3.13)*, `ch03.rotation_tensor(G)` =
G − Gᵀ. ⚠️ The book's R is **twice** the antisymmetric part A of Ch. 2 (R = 2A); that is why a ½ stands in front of R
in $\frac{\partial u_i}{\partial x_j}=S_{ij}+\tfrac12R_{ij}$ *(Eq. 3.11)*.

### 🧩 Relative velocity of a neighbour: du = G·dx (3.10) `C06`

*The question:* Two corks float 1 cm apart. How fast does one move relative to the other — and what does that single matrix of nine
numbers already tell us about the blob of water between them?

*In one line:* $du_i=(\partial u_i/\partial x_j)\,dx_j$ *(3.10)*

#### The problem in plain words

A drop of ink in a stream is stretched into a thin filament, a cloud is sheared into a streak, air in a rising thermal
swells. All of these are about *differences* of velocity between neighbouring points. Chapter 4's stress law needs
exactly these differences (the rates of deformation), not the velocity itself.

#### The idea

```
O at x, velocity u          P at x + dx, velocity u + du          (Fig. 3.9)
zoom in far enough and every smooth field looks linear:   du = G · dx,   G_ij = ∂u_i/∂x_j   (nine numbers)
G = S + ½R :   S deforms the blob  (stretch, shear, swell)   ½R spins it
```

> 📎 **Primer — multivariable first-order Taylor expansion.** `P98` · Ch. 1's $f(z+dz)\approx f(z)+f'(z)\,dz$ (P26) in several variables:
$u_i(\mathbf x+d\mathbf x)\approx u_i(\mathbf x)+\frac{\partial u_i}{\partial x_j}dx_j$ — one partial derivative per
direction, summed over j. The neglected terms shrink like $|d\mathbf x|^2$, so after dividing by $|d\mathbf x|$ they
vanish as the neighbour comes closer (Ch. 2 P68, orders of smallness).

In [ ]:
f = lambda x, y: np.sin(x)*np.exp(y)                 # a smooth function of two variables
x0, y0, dx, dy = 0.3, 0.1, 0.01, -0.02               # a point and a small step
lin = f(x0, y0) + np.cos(x0)*np.exp(y0)*dx + np.sin(x0)*np.exp(y0)*dy   # value + (∂f/∂x)dx + (∂f/∂y)dy
print(f(x0 + dx, y0 + dy) - lin)                     # ≈ −1e-4: the leftover is second order in the step

#### 🧮 Derivation — Relative velocity near a point, Eq. (3.10) `D07` (Eq. 3.10)

**What we want to show.** Find how fast a neighbouring point moves relative to a given point, to first order in their separation — the relation everything in §3.4 is built on.

**Assumptions.** u is smooth (twice differentiable) near x (step 1); |dx| small compared with the distance over which G changes (step 3).

**The plan.**

1. Taylor-expand the velocity at P about O.
2. Subtract O's velocity.
3. Drop what vanishes faster than the separation itself.
4. Recognise a matrix–vector product.

**Tools we use** (each explained before this point): Multivariable first-order Taylor expansion (primer in C06) · orders of smallness (ch02 P68) · the velocity gradient G (R01) · matrix–vector product (ch02 P63).

**We start from**

$$ u_i(\mathbf x+d\mathbf x,t) $$

*In words:* the velocity at the neighbour P, a small step dx away from O (Fig. 3.9), at the same instant.

---

**Step 1 of 4 — Taylor-expand u about x.**

$$ u_i(\mathbf x+d\mathbf x)=u_i(\mathbf x)+\dfrac{\partial u_i}{\partial x_j}dx_j+O(\lvert d\mathbf x\rvert^2) $$

- *Why we can do this:* Multivariable Taylor expansion (primer): one first derivative per direction, summed over j; the remainder is of second order because u is smooth.
- *In words:* Near O the velocity changes linearly with position.

---

**Step 2 of 4 — Subtract the velocity at O.**

$$ du_i\equiv u_i(\mathbf x+d\mathbf x)-u_i(\mathbf x)=\dfrac{\partial u_i}{\partial x_j}dx_j+O(\lvert d\mathbf x\rvert^2) $$

- *Why we can do this:* The relative velocity of P seen from O is the difference of the two velocities; $u_i(\mathbf x)$ cancels.
- *In words:* How fast the neighbour moves away from, toward or round O.

---

**Step 3 of 4 — Drop the second-order remainder.**

$$ du_i=(\partial u_i/\partial x_j)\,dx_j $$

- *Why we can do this:* The remainder shrinks like $\lvert d\mathbf x\rvert^2$ while the kept term shrinks like $\lvert d\mathbf x\rvert$: their ratio → 0 as P approaches O (orders of smallness). This is (3.10), $du_i=(\partial u_i/\partial x_j)\,dx_j$.
- *In words:* For close neighbours the linear term is all that matters.

---

**Step 4 of 4 — Write it as a matrix product.**

$$ d\mathbf u=\mathbf G\cdot d\mathbf x,\qquad G_{ij}=\partial u_i/\partial x_j $$

- *Why we can do this:* Row i of G times the column dx is the sum over j (Ch. 2 convention: row = velocity component). One matrix describes every neighbour.
- *In words:* Nine numbers tell how the whole neighbourhood moves.

---

**Result**

$$ du_i=(\partial u_i/\partial x_j)\,dx_j\ \text{(3.10),}\ d\mathbf u=\mathbf G\cdot d\mathbf x $$

*In words:* near any point, relative velocity is a linear function of separation.

**What it means.** A small blob of fluid moves as a linear map: all of its stretching, shearing, swelling and spinning is in G. The rest of §3.4 reads G's pieces. It fails for blobs as large as the scale on which G itself varies.

**Check it.** Units: (1/s)(m) = m/s ✓. Linear field u = G·x: exact for any dx (no remainder) ✓. Numbers: G = [[1, 2], [0, −1]], dx = (0.01, 0.02) m → du = (0.05, −0.02) m/s; the nonlinear field of the worked example differs by (1, 2) × 10⁻⁴ m/s and the difference falls with slope 2 ✓.

> ⚠️ **Common confusion (traps in `D07`):** Thinking the remainder is zero (it is only small). Transposing G ($G_{ij}=\partial u_i/\partial x_j$: row = component, column = derivative direction).

📝 **Note.** **Why rates of deformation?** `N25` — A solid resists being *deformed* (strain); a fluid resists being deformed *quickly* (strain rate). Ch. 4 §4.5 relates
the stress to S — the Newtonian law that generalises Ch. 1's $\tau=\mu\,du/dy$.

📝 **Note.** **Names and roles** `N26` — S (deformation) is linked to the stress in a moving fluid; R (rotation) is not — a rigidly spinning fluid feels no
viscous stress. C11 shows why only S can enter.

#### ✏️ Tiny example: G = [[1, 2], [0, −1]] s⁻¹ and a neighbour 2 cm away

1. $d\mathbf x=(0.01,0.02)$ m.
2. $du_1=G_{11}dx_1+G_{12}dx_2=1\times0.01+2\times0.02=0.05$ m/s.
3. $du_2=G_{21}dx_1+G_{22}dx_2=0+(-1)\times0.02=-0.02$ m/s.
4. Split: $\mathbf S=[[1,1],[1,-1]]$, $\tfrac12\mathbf R=[[0,1],[-1,0]]$ s⁻¹, and $\mathbf S+\tfrac12\mathbf R=[[1,2],[0,-1]]=\mathbf G$ ✓
   (R01).
5. The field $\mathbf u=(x_1+2x_2+x_1^2,\ -x_2+x_1x_2)$ has exactly this G at the origin; its true du at that neighbour is
   (0.0501, −0.0198) m/s — the difference (1, 2) × 10⁻⁴ m/s is the second-order remainder.

In [ ]:
ufield = lambda x, t: np.array([x[0] + 2*x[1] + x[0]**2, -x[1] + x[0]*x[1]])   # a nonlinear plane field [m/s]
G = ch03.velocity_gradient_at(ufield, np.zeros(2), 0.0)   # G_ij = ∂u_i/∂x_j at the origin by central stencils [1/s]
dx = np.array([0.01, 0.02])                                 # the neighbour's offset [m]
print(G)
print(ch03.relative_velocity(G, dx), ufield(dx, 0) - ufield(np.zeros(2), 0))   # (3.10) vs the exact difference [m/s]
S, R = ch03.strain_rate_tensor(G), ch03.rotation_tensor(G)  # (3.12) and the book's (3.13)
assert np.allclose(S + 0.5*R, G)                            # (3.11): G = S + ½R
e = dx/np.linalg.norm(dx); hs = 0.02*0.5**np.arange(6)      # one direction, six shrinking distances [m]
err = [np.linalg.norm(ufield(h*e, 0) - ufield(np.zeros(2), 0) - G @ (h*e)) for h in hs]   # |exact du − G·dx|
print(f"observed order of the remainder: {observed_order(hs, err):.2f}")   # 2: the neglected terms are O(|dx|²)

**What does the code above do?**

1. A nonlinear field and its gradient at the origin, $G=[[1,2],[0,-1]]$ s⁻¹, by second-order stencils.
2. $d\mathbf u=\mathbf G\cdot d\mathbf x$, i.e. $du_i=(\partial u_i/\partial x_j)\,dx_j$ *(3.10)*, gives (0.05, −0.02) m/s;
   the exact difference is (0.0501, −0.0198) m/s.
3. The R01 split holds exactly.
4. The remainder falls like $|d\mathbf x|^2$: slope 2 on log–log axes (Ch. 1 P13).

In [ ]:
from scripts.ch03_drawings import ring_arrows                # draws du = G·dx on a ring of neighbours (no physics inside)
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 4.2))          # (a) the ring, (b) the remainder
ring_arrows(a, G, radius=0.1, parts=("total",), n=24, scale=0.25)   # 24 neighbours 10 cm from O; arrows × 0.25
a.plot(0, 0, "o", color=COLORS["ink"]); a.set_aspect("equal"); a.set_xlim(-0.2, 0.2); a.set_ylim(-0.2, 0.2)   # O at the centre
a.set_xlabel("$dx_1$ [m]"); a.set_ylabel("$dx_2$ [m]"); a.set_title("(a) relative velocity du = G·dx of each neighbour")
b.loglog(hs, err, "o-", color=COLORS["accent"], label="|exact du − G·dx|")   # measured remainder
b.loglog(hs, err[0]*(hs/hs[0])**2, "--", color=COLORS["muted"], label="slope 2")   # a guide ∝ |dx|²
b.set_xlabel("distance |dx| [m]"); b.set_ylabel("remainder [m/s]"); b.legend(fontsize=8)   # log–log axes (Ch. 1 P13)
b.set_title("(b) the linear picture becomes exact as the ring shrinks")
savefig(fig, "ch03", "relative_velocity_ring"); plt.show()   # save to outputs/ch03 and draw

**What you see.** (a) Purple arrows on a ring, pointing out along one direction and in along another, plus a swirl; (b) a straight line of slope 2.

**How to read it.** (a) is everything the element does in the next instant: each neighbour's velocity relative to O, zero at O and
growing with distance. (b) says the linear picture is exact in the limit — the neglected terms fall like the square of
the distance.

**What would change if…** …G were antisymmetric (a pure rotation, e.g. [[0, −1], [1, 0]]): every arrow on the ring would be tangential — no stretching at all.

**What would change if…** …you watched only two neighbours on the $x_1$-axis? Their separation changes at $\partial u_1/\partial x_1$ per unit
length — the diagonal of S. That is C07.

### 🧩 Linear strain rate: how fast a material line stretches `C07`

*The question:* A short thread of dye lies along the flow. At what rate, per unit of its length, is it being stretched — and in which
direction would it stretch fastest?

#### The problem in plain words

Stretching is how ocean eddies draw tracers into long filaments and how vortex tubes intensify (Ch. 5). A rubber band
held at both ends and pulled: the stretching rate *per unit length* is what matters, not the length itself.

#### The idea

```
A ●────────● B     ends move at u₁ and u₁ + (∂u₁/∂x₁)δx₁   (Fig. 3.10)
B outruns A by (∂u₁/∂x₁)δx₁ per second   →   stretch per length per time = ∂u₁/∂x₁ = S₁₁
any direction n:   (1/ℓ) Dℓ/Dt = n·S·n
```

> 📎 **Primer — material line element.** `P99` · A *material* line element $\delta\mathbf x$ joins two nearby fluid particles and is carried with them — it stretches
and turns as they move. Its rate of change following the particles is the difference of their velocities:
$\frac{D(\delta\mathbf x)}{Dt}=\delta\mathbf u=\mathbf G\cdot\delta\mathbf x$ (C06). Every rate of §3.4 is measured on
such elements. For a linear flow the element after time t is $e^{\mathbf Gt}\cdot\delta\mathbf x$ (`linear_flow_map`,
the matrix exponential of Ch. 2 P79).

In [ ]:
G2 = np.array([[2.0, 0.0], [0.0, -2.0]])             # u = (2x, −2y) [1/s × m]
dx0 = np.array([0.01, 0.0])                          # a 1 cm element along x [m]
dx1 = ch03.linear_flow_map(G2, 0.01) @ dx0           # carried for 0.01 s: e^{G t}·dx0
print(dx1*100)                                       # in cm: [1.0202 0.]: it grew by 2 %

#### 🧮 Derivation — Linear strain rate: stretching per length = ∂u₁/∂x₁, and n·S·n in any direction `D08`

**What we want to show.** Find how fast a short material thread stretches, per unit of its own length — first along x₁ (the book's Fig. 3.10), then along any direction n (which the book only mentions).

**Assumptions.** dt small (terms of order dt² dropped, step 2); δx₁ small (D07 applies).

**The plan.**

1. Move both ends for a short time dt.
2. Measure the new length.
3. Divide by the old length and dt and take the limit.
4. Generalise to any axis and any direction.

**Tools we use** (each explained before this point): Material line element (primer in C07) · D07, $du_i=(\partial u_i/\partial x_j)dx_j$ · limit of a difference quotient and orders of smallness (ch02 P68) · the strain-rate tensor (3.12), $S_{ij}=\tfrac12\big(\frac{\partial u_i}{\partial x_j}+\frac{\partial u_j}{\partial x_i}\big)$ (R02).

**We start from**

$$ \begin{array}{l}\text{A material segment AB along x₁ of length δx₁; by D07 its ends move at}\ u_1\ \text{(A)} \\ \text{and}\ u_1+(\partial u_1/\partial x_1)\,\delta x_1\ \text{(B)}\end{array} $$

*In words:* the far end moves faster by the slope times the length.

---

**Step 1 of 8 — Move both ends for dt.**

$$ AA'=u_1\,dt,\qquad BB'=\Big(u_1+\dfrac{\partial u_1}{\partial x_1}\delta x_1\Big)dt $$

- *Why we can do this:* Each end moves with its own velocity; over a short dt the displacement is velocity × dt (first order in dt).
- *In words:* B travels a little further than A.

---

**Step 2 of 8 — Book-keep the new length.**

$$ A'B'=AB+BB'-AA'=\delta x_1+\dfrac{\partial u_1}{\partial x_1}\delta x_1\,dt $$

- *Why we can do this:* Along x₁ the new length is the old one plus how much further B went than A (Fig. 3.10). Sideways velocity differences only tilt AB — a dt² change (the book skips this).
- *In words:* The thread grows by the extra distance of its far end.

---

**Step 3 of 8 — Divide by length and time.**

$$ \dfrac{A'B'-AB}{AB\,dt}=\dfrac{\partial u_1}{\partial x_1} $$

- *Why we can do this:* The fractional growth per unit time is what we want (a rate independent of the thread's length); δx₁ and dt cancel exactly.
- *In words:* The thread stretches by ∂u₁/∂x₁ of its length per second.

---

**Step 4 of 8 — Take the limit dt → 0.**

$$ \dfrac{1}{\delta x_1}\dfrac{D}{Dt}(\delta x_1)=\dfrac{\partial u_1}{\partial x_1} $$

- *Why we can do this:* The difference quotient becomes a derivative following the material thread — hence D/Dt; the neglected order-dt² pieces vanish after dividing by dt (P68).
- *In words:* The book's linear strain rate.

---

**Step 5 of 8 — Same construction on each axis.**

$$ \dfrac{1}{\delta x_\eta}\dfrac{D(\delta x_\eta)}{Dt}=\dfrac{\partial u_\eta}{\partial x_\eta}=S_{\eta\eta} $$

- *Why we can do this:* Nothing in steps 1–4 used the choice of axis; a Greek index means no summation. The diagonal of G equals the diagonal of S, since $S_{\eta\eta}=\tfrac12(G_{\eta\eta}+G_{\eta\eta})$ by (3.12), $S_{ij}=\tfrac12\big(\frac{\partial u_i}{\partial x_j}+\frac{\partial u_j}{\partial x_i}\big)$.
- *In words:* The diagonal of S is the stretching rate along each axis.

---

**Step 6 of 8 — Differentiate the squared length.**

$$ \ell\,\dfrac{D\ell}{Dt}=\tfrac12\dfrac{D(\delta\mathbf x\cdot\delta\mathbf x)}{Dt}=\delta\mathbf x\cdot\dfrac{D\delta\mathbf x}{Dt}=\delta\mathbf x\cdot\mathbf G\cdot\delta\mathbf x $$

- *Why we can do this:* For a thread $\delta\mathbf x=\ell\mathbf n$ in any direction, $\ell^2=\delta\mathbf x\cdot\delta\mathbf x$; the product rule gives $2\,\delta\mathbf x\cdot D\delta\mathbf x/Dt$, and a material element changes at $D\delta\mathbf x/Dt=\mathbf G\cdot\delta\mathbf x$ (D07, primer P99). The book skips this.
- *In words:* The length changes by the part of the ends' relative velocity that lies along the thread.

---

**Step 7 of 8 — Divide by ℓ².**

$$ \dfrac{1}{\ell}\dfrac{D\ell}{Dt}=\mathbf n\cdot\mathbf G\cdot\mathbf n $$

- *Why we can do this:* $\delta\mathbf x\cdot\mathbf G\cdot\delta\mathbf x=\ell^2\,\mathbf n\cdot\mathbf G\cdot\mathbf n$ and $\ell\neq0$; dividing gives a rate per unit length that does not depend on the thread's length.
- *In words:* The stretching rate along n.

---

**Step 8 of 8 — Drop the antisymmetric part.**

$$ \dfrac{1}{\ell}\dfrac{D\ell}{Dt}=\mathbf n\cdot\mathbf S\cdot\mathbf n $$

- *Why we can do this:* $\mathbf G=\mathbf S+\tfrac12\mathbf R$ by (3.11), $\frac{\partial u_i}{\partial x_j}=S_{ij}+\tfrac12R_{ij}$, and $\mathbf n\cdot\mathbf R\cdot\mathbf n=0$ for an antisymmetric R ($n_in_jR_{ij}=-n_jn_iR_{ji}$ is its own negative).
- *In words:* Stretching along any direction is a quadratic form of S; rotation stretches nothing.

---

**Result**

$$ \frac{1}{\delta x_1}\frac{D}{Dt}(\delta x_1)=\frac{\partial u_1}{\partial x_1}\ \text{; in general}\ \frac1\ell\frac{D\ell}{Dt}=\mathbf n\cdot\mathbf S\cdot\mathbf n $$

*In words:* the diagonal entries of S (in any axes) are stretching rates per unit length.

**What it means.** Stretching per length is a property of the flow at a point, not of the thread; its extreme values over all directions are the principal strain rates (C12). Vortex stretching (Ch. 5) and tracer filamentation (Ch. 12) are this rate at work.

**Check it.** Units 1/s ✓. Rigid rotation: n·S·n = 0 for every n ✓. Numbers: u = (2x, −2y): S₁₁ = 2 s⁻¹, a 1 cm thread → 1.0202 cm after 0.01 s (e^{0.02}) ✓; at 45°: ½(2 − 2) = 0 ✓.

> ⚠️ **Common confusion (traps in `D08`):** Using AB + BB′ + AA′ (A moves too, so subtract AA′). Believing the sideways shear stretches the thread at first order. Keeping R in n·G·n (it contributes nothing).

#### ✏️ Tiny example: u = (2x, −2y) s⁻¹, a 1 cm thread along x for 0.01 s

1. $S_{11}=\partial u_1/\partial x_1=2$ s⁻¹.
2. First order: new length $\approx1\times(1+2\times0.01)=1.02$ cm.
3. Exact (the ends move as $e^{2t}$): $e^{0.02}=1.0202$ cm.
4. A thread along y: $S_{22}=-2$ s⁻¹ — it shrinks to $e^{-0.02}=0.9802$ cm.
5. A thread at 45°: $\mathbf n\cdot\mathbf S\cdot\mathbf n=\tfrac12(2-2)=0$ — its length does not change at first order.

In [ ]:
G = np.array([[1.0, 2.0], [0.0, -1.0]])              # C06's velocity gradient [1/s]
for deg in (0, 22.5, 45, 90):                        # probe directions [degrees]
    n = [np.cos(np.deg2rad(deg)), np.sin(np.deg2rad(deg))]   # unit vector n (angles in radians inside)
    print(f"{deg:5.1f}°: n·S·n = {ch03.linear_strain_rate(G, n):+.3f} 1/s")   # stretching rate per unit length

**What does the code above do?**

1. The probe direction n (`np.deg2rad` converts degrees to radians, Ch. 2 P66).
2. `linear_strain_rate` returns n·S·n (the antisymmetric part drops out of any n·G·n, D08 step 8):
   $\cos2\theta+\sin2\theta$ for this G.
3. The largest stretching, √2 = 1.414 s⁻¹ at 22.5°, is along a principal axis (C12).

In [ ]:
dt = 1e-6                                            # a very short time [s]
for deg in (0, 22.5, 45, 90):
    n = np.array([np.cos(np.deg2rad(deg)), np.sin(np.deg2rad(deg))])
    dx0 = 0.01*n                                     # a 1 cm thread along n [m]
    M = ch03.linear_flow_map(G, dt)                  # carry it for dt
    mine = (np.linalg.norm(M @ dx0) - np.linalg.norm(dx0))/(np.linalg.norm(dx0)*dt)   # (ΔL/L)/Δt, measured
    assert np.allclose(mine, ch03.linear_strain_rate(G, n), rtol=1e-5, atol=1e-5)   # = n·S·n
    assert np.allclose(ch03.measured_strain_rates(G, n, dt)["stretch"], mine, rtol=1e-5, atol=1e-5)   # library's ruler
print("tracked threads stretch at n·S·n in every direction")

In [ ]:
th = np.linspace(0, np.pi, 72, endpoint=False)        # 72 directions from 0° to 180° [rad]
meas = [ch03.measured_strain_rates(G, [np.cos(a), np.sin(a)], 1e-6)["stretch"] for a in th]   # tracked threads
form = [ch03.linear_strain_rate(G, [np.cos(a), np.sin(a)]) for a in th]                       # n·S·n
fig, ax = plt.subplots(figsize=(7.5, 3.8))                    # one panel
ax.plot(np.degrees(th), form, color=COLORS["teal"], lw=2.5, label="formula n·S·n")            # the tensor formula
ax.plot(np.degrees(th), meas, "o", color=COLORS["accent"], ms=3, label="measured (1/ℓ)Δℓ/Δt")   # the ruler on moving threads
ax.axhline(0, color=COLORS["muted"], lw=0.8)                  # no stretching
ax.axvline(22.5, ls="--", color=COLORS["blue"], label="fastest stretching (22.5°)")          # principal axis (C12)
ax.axvline(112.5, ls="--", color=COLORS["rose"], label="fastest squeezing (112.5°)")         # the other one, 90° away
ax.set_xlabel("direction θ of the thread [°]"); ax.set_ylabel("stretching rate [1/s]"); ax.legend(fontsize=8)
ax.set_title("Each direction has its own stretching rate; the extremes are 90° apart")
savefig(fig, "ch03", "stretching_rose"); plt.show()           # save to outputs/ch03 and draw

**What you see.** Purple dots on a smooth teal wave between −√2 and +√2 s⁻¹, with dashed blue and rose lines at the peak and the trough.

**How to read it.** The measurement on moving threads and the tensor formula are the same curve. Threads near 22.5° stretch fastest,
threads near 112.5° are squeezed fastest; the zero crossings (at 67.5° and 157.5°) are directions whose length does not
change at first order.

**What would change if…** …G were antisymmetric (solid-body rotation): the curve would be flat at zero — nothing stretches.

**What would change if…** …two threads start perpendicular? Besides stretching they turn toward (or away from) each other; the rate at which the
right angle closes is the off-diagonal of S (C08).

> 🔁 **Recap — The rotation tensor is a vector in disguise** (Ch. 2 §2.10). R is antisymmetric, $R_{ij}=-R_{ji}$: zero diagonal and three independent entries, which pair up with the three
components of the vorticity $\boldsymbol\omega=\nabla\times\mathbf u$.

> 🔁 **Recap — R ↔ ω** (Ch. 2 §2.10). $R_{ij}=-\varepsilon_{ijk}(\nabla\times\mathbf u)_k=-\varepsilon_{ijk}\omega_k=\begin{bmatrix}0&-\omega_3&\omega_2\\\omega_3&0&-\omega_1\\-\omega_2&\omega_1&0\end{bmatrix}$
*(Eq. 3.15, the book's (2.26)–(2.27) with R = 2A)* — `ch03.antisymmetric_from_vector(omega)` builds the matrix;
`ch03.vorticity_from_gradient(G)` returns ω = the vector of R.

### 🧩 Shear strain rate: how fast a right angle closes `C08`

*The question:* Two dye threads cross at a right angle. How fast does that angle close, and why does the book call half of that rate
$S_{12}$?

#### The problem in plain words

Push the top of a deck of cards sideways: the corners of each card stop being right angles. Fluids flowing past a wall
are sheared this way all the time (Couette and boundary-layer flows, Ch. 8–9), and the shear *rate* is what the viscous
shear stress depends on.

#### The idea

```
     C ┐ dα                 vertical side tilts clockwise by dα        ∝ ∂u₁/∂x₂
       │ ╲                  horizontal side tilts counterclockwise by dβ ∝ ∂u₂/∂x₁
     B └───── dβ            closing rate α + β  →  S₁₂ = ½ D(α+β)/Dt        (Fig. 3.11)
```

> 📎 **Primer — small-angle approximation.** `P100` · For an angle ε measured in radians, $\tan\varepsilon\approx\varepsilon$ and $\cos\varepsilon\approx1$ with errors of
order ε³ and ε²: a tiny tilt is its own tangent. Every angle below is proportional to dt, so the neglected pieces
vanish after dividing by dt.

In [ ]:
eps = np.array([0.1, 0.01, 0.001])                   # three small angles [rad]
print([f"{v:.2e}" for v in np.tan(eps) - eps])     # ['3.35e-04', '3.33e-07', '3.33e-10']: the error falls like ε³

#### 🧮 Derivation — Shear strain rate: S₁₂ = ½ D(α + β)/Dt `D09`

**What we want to show.** Find how fast two initially perpendicular material threads close their right angle, and show that half of that rate is the off-diagonal entry S₁₂ (Fig. 3.11).

**Assumptions.** dt small (angles ∝ dt, step 3); threads short (D07).

**The plan.**

1. Find each thread's tilt in dt from the relative velocity of its far end.
2. Use the small-angle approximation.
3. Add the tilts, halve, divide by dt.
4. Recognise S₁₂; generalise to any perpendicular pair.

**Tools we use** (each explained before this point): D07 · small-angle approximation (primer in C08) · (3.12), $S_{ij}=\tfrac12(\partial u_i/\partial x_j+\partial u_j/\partial x_i)$ (R02) · tensor components in rotated axes (Ch. 2 §2.4).

**We start from**

$$ \begin{array}{l}\text{Two threads from the corner B: one along x₂ of length δx₂ (up to C), one along} \\ \text{x₁ of length δx₁; dα is the clockwise tilt of the vertical thread, dβ the} \\ \text{counterclockwise tilt of the horizontal one}\end{array} $$

*In words:* both tilts close the right angle.

---

**Step 1 of 8 — Relative velocity of C (top).**

$$ du_1=\dfrac{\partial u_1}{\partial x_2}\,\delta x_2 $$

- *Why we can do this:* D07 with $d\mathbf x=(0,\delta x_2)$: the top of the vertical thread moves sideways (along x₁) faster than the bottom by this much.
- *In words:* The top slides ahead of the bottom.

---

**Step 2 of 8 — Its sideways offset over its height.**

$$ \tan d\alpha=\dfrac{(\partial u_1/\partial x_2)\,\delta x_2\,dt}{\delta x_2} $$

- *Why we can do this:* In dt the top gains the offset du₁·dt; a right triangle with that offset over the height δx₂ gives the tilt angle.
- *In words:* Tilt = sideways shift ÷ height.

---

**Step 3 of 8 — Small-angle approximation.**

$$ d\alpha=\dfrac{\partial u_1}{\partial x_2}\,dt $$

- *Why we can do this:* tan dα ≈ dα with an error of order dt³ (primer); δx₂ cancels. Stretching of the thread changes its height only at order dt² — the book skips this.
- *In words:* The vertical thread tilts clockwise at ∂u₁/∂x₂.

---

**Step 4 of 8 — Same for the horizontal thread.**

$$ d\beta=\dfrac{\partial u_2}{\partial x_1}\,dt $$

- *Why we can do this:* D07 with $d\mathbf x=(\delta x_1,0)$: its right end rises by $(\partial u_2/\partial x_1)\delta x_1 dt$ over the run δx₁; small angle again.
- *In words:* The horizontal thread tilts counterclockwise at ∂u₂/∂x₁.

---

**Step 5 of 8 — Add the two tilts.**

$$ d\alpha+d\beta=\Big(\dfrac{\partial u_1}{\partial x_2}+\dfrac{\partial u_2}{\partial x_1}\Big)dt $$

- *Why we can do this:* α is clockwise from the vertical and β counterclockwise from the horizontal: both move the threads toward each other, so their sum is how much the right angle closes.
- *In words:* The corner shrinks by α + β.

---

**Step 6 of 8 — Halve and divide by dt.**

$$ \tfrac12\dfrac{D(\alpha+\beta)}{Dt}=\tfrac12\Big(\dfrac{\partial u_1}{\partial x_2}+\dfrac{\partial u_2}{\partial x_1}\Big) $$

- *Why we can do this:* The book defines the shear rate as the *average* closing rate of the two threads (hence ½); D/Dt because the threads are material; the limit is exact.
- *In words:* The average rate at which each thread turns toward the other.

---

**Step 7 of 8 — Recognise the strain-rate entry.**

$$ \tfrac12\dfrac{D(\alpha+\beta)}{Dt}=S_{12}=S_{21} $$

- *Why we can do this:* (3.12), $S_{ij}=\tfrac12\big(\frac{\partial u_i}{\partial x_j}+\frac{\partial u_j}{\partial x_i}\big)$ with i = 1, j = 2 is exactly the bracket of step 6; S is symmetric.
- *In words:* S₁₂ is half the closing rate of a right angle.

---

**Step 8 of 8 — Any perpendicular pair.**

$$ \tfrac12\dfrac{D(\alpha+\beta)}{Dt}=\mathbf n_1\cdot\mathbf S\cdot\mathbf n_2 $$

- *Why we can do this:* Repeat steps 1–7 in axes along $\mathbf n_1$, $\mathbf n_2$: the off-diagonal component in rotated axes is $\mathbf n_1\cdot\mathbf S\cdot\mathbf n_2$ (tensor rule, Ch. 2 §2.4). The book skips this.
- *In words:* Every pair of directions has its own closing rate.

---

**Result**

$$ \tfrac12\frac{D(\alpha+\beta)}{Dt}=\tfrac12\Big(\frac{\partial u_1}{\partial x_2}+\frac{\partial u_2}{\partial x_1}\Big)=S_{12}=S_{21} $$

*In words:* the off-diagonal strain rate is half the rate at which a right angle closes.

**What it means.** S₁₂ measures shearing — what a Newtonian fluid resists with its viscosity (Ch. 4: τ₁₂ = 2μS₁₂, which reduces to Ch. 1's τ = μ du/dy for u = (u(y), 0)). Its value depends on the pair of directions; along the principal axes it vanishes (C12).

**Check it.** Units rad/s = 1/s ✓. Solid-body rotation: dα = −dβ (both threads turn the same way) ⇒ S₁₂ = 0 ✓. Numbers: shear u = (γy, 0), γ = 1 s⁻¹, dt = 0.01 s: dα = 0.01 rad, dβ = 0 ⇒ S₁₂ = 0.5 s⁻¹ = γ/2 ✓.

> ⚠️ **Common confusion (traps in `D09`):** Signs: α is clockwise, β counterclockwise — both *close* the angle. Forgetting the ½ (γ = 2S₁₂, not S₁₂; Ch. 2's Ex. 2.4 used Γ = S₁₂). Keeping the stretching of the sides (second order).

> 📎 **Primer — rigid-body velocity Ω × x.** `P101` · A rigid body turning at angular velocity $\boldsymbol\Omega$ about an axis through the origin moves each of its points
with $\mathbf v=\boldsymbol\Omega\times\mathbf x$ — perpendicular to both the axis and the arm, of size Ω times the
distance from the axis. Adding a translation $\mathbf U$ gives every rigid motion: $\mathbf u=\mathbf U+\boldsymbol\Omega\times\mathbf x$.

In [ ]:
Omega = np.array([0.0, 0.0, 1.0])                    # 1 rad/s about z
print(np.cross(Omega, [2.0, 0.0, 0.0]))              # [0. 2. 0.]: 2 m from the axis → 2 m/s, tangential

#### 🧮 Derivation — Rigid motion does not deform: U + Ω × x ⇒ S = 0, ω = 2Ω `D10`

**What we want to show.** Show that a rigid motion — any translation plus any rotation — has zero strain rate, so that S is the same for every translating or rotating observer; the book leaves it to Exercise 3.17.

**Assumptions.** U and Ω do not depend on position (step 2); they may depend on time — only spatial derivatives enter.

**The plan.**

1. Write the velocity in index form.
2. Differentiate to get G.
3. Show its symmetric part vanishes.
4. Read off R and the vorticity.
5. Conclude what a change of frame does to S.

**Tools we use** (each explained before this point): Rigid-body velocity Ω × x (primer in C08) · $(\mathbf a\times\mathbf b)_i=\varepsilon_{ijk}a_jb_k$ and the antisymmetry of ε (Ch. 2 §2.7) · $\partial x_k/\partial x_m=\delta_{km}$ · (3.12), $S_{ij}=\tfrac12\big(\frac{\partial u_i}{\partial x_j}+\frac{\partial u_j}{\partial x_i}\big)$, (3.13), $R_{ij}=\frac{\partial u_i}{\partial x_j}-\frac{\partial u_j}{\partial x_i}$ (R02, R03) · (3.15) $R_{ij}=-\varepsilon_{ijk}\omega_k$ (from Ch. 2; recapped as R05 before C10).

**We start from**

$$ \mathbf u=\mathbf U+\boldsymbol\Omega\times\mathbf x\ \text{, U and Ω uniform in space} $$

*In words:* every point moves like a point of a rigid body.

---

**Step 1 of 5 — Write the velocity in index form.**

$$ u_i=U_i+\varepsilon_{ijk}\Omega_jx_k $$

- *Why we can do this:* The cross product in components (Ch. 2 §2.7): $(\boldsymbol\Omega\times\mathbf x)_i=\varepsilon_{ijk}\Omega_jx_k$. Index form lets us differentiate term by term.
- *In words:* A translation plus a rotation, written with ε.

---

**Step 2 of 5 — Differentiate with respect to x_m.**

$$ \dfrac{\partial u_i}{\partial x_m}=\varepsilon_{ijm}\Omega_j $$

- *Why we can do this:* U and Ω are uniform, so only $x_k$ varies and $\partial x_k/\partial x_m=\delta_{km}$ picks k = m.
- *In words:* The velocity gradient of a rigid motion.

---

**Step 3 of 5 — Symmetrise.**

$$ S_{im}=\tfrac12(\varepsilon_{ijm}+\varepsilon_{mji})\Omega_j=0 $$

- *Why we can do this:* (3.12), $S_{ij}=\tfrac12\big(\frac{\partial u_i}{\partial x_j}+\frac{\partial u_j}{\partial x_i}\big)$ adds G and its transpose; swapping the first and last index of ε flips its sign, $\varepsilon_{mji}=-\varepsilon_{ijm}$, so the bracket vanishes.
- *In words:* A rigid motion stretches and shears nothing.

---

**Step 4 of 5 — Read off R and ω.**

$$ R_{im}=2\varepsilon_{ijm}\Omega_j=-2\varepsilon_{imj}\Omega_j\ \Rightarrow\ \boldsymbol\omega=2\boldsymbol\Omega $$

- *Why we can do this:* (3.13), $R_{ij}=\frac{\partial u_i}{\partial x_j}-\frac{\partial u_j}{\partial x_i}$: R = G − Gᵀ = 2G here; one index swap, compared with (3.15), $R_{ij}=-\varepsilon_{ijk}\omega_k$, gives the vorticity.
- *In words:* The vorticity of a rigid rotation is twice its angular velocity.

---

**Step 5 of 5 — Add a rigid motion to any flow.**

$$ \mathbf G_{\rm new}=\mathbf G+\mathbf G_{\rm rigid}\ \Rightarrow\ \mathbf S_{\rm new}=\mathbf S $$

- *Why we can do this:* Changing to a translating or rotating observer adds (or subtracts) a rigid motion; G is linear in u and the rigid part has S = 0 (step 3), so only R changes. For a rotating observer the comparison is made at the instant the two sets of axes coincide (as in D13); later the components of S turn as a tensor, but S itself is unchanged.
- *In words:* S is observer-independent; ω is not.

---

**Result**

$$ \mathbf u=\mathbf U+\boldsymbol\Omega\times\mathbf x\ \Rightarrow\ S_{ij}=0,\ \boldsymbol\omega=2\boldsymbol\Omega $$

*In words:* rigid motion does not deform, and its vorticity is twice its rotation rate.

**What it means.** A fluid in rigid rotation (a stirred cup after it settles, a spun-up tank) has no internal friction: the viscous stress can depend only on S (Ch. 4 §4.5). And "S is frame-independent" means: every observer who moves rigidly agrees on how fluid elements deform — while they disagree on how they spin (N28).

**Check it.** Units 1/s ✓. Numbers: Ω = (0, 0, 1) s⁻¹: G = [[0, −1], [1, 0]], S = 0, ω₃ = 1 − (−1) = 2 ✓; the random (U, Ω) of the notebook: S = 0 to 1e-12, ω = 2Ω = (1, −2, 4) ✓.

> ⚠️ **Common confusion (traps in `D10`):** Letting Ω depend on position (then it is no longer rigid). Concluding "ω is frame-independent too" — step 5 changes R. Losing the factor 2 (ω = 2Ω, spin = Ω).

📝 **Note.** **Rigid motion does not deform** `N27` — For $\mathbf u=\mathbf U+\boldsymbol\Omega\times\mathbf x$ with $\mathbf U$ and $\boldsymbol\Omega$ uniform, S = 0 and R
carries all of G (ω = 2Ω). Hence adding a rigid motion — watching from a translating or rotating frame — changes G only
by an antisymmetric part: **S is the same for every such observer** (even if U varies in time); ω is not (N28).

#### ✏️ Tiny example: simple shear u = (γy, 0), γ = 1 s⁻¹, for dt = 0.01 s

1. The top of a 1 cm vertical side moves $\gamma\,\delta x_2\,dt=1\times0.01\times0.01=10^{-4}$ m further than its foot:
   $\tan d\alpha=10^{-4}/10^{-2}=0.01$, so $d\alpha\approx0.01$ rad.
2. The horizontal side does not tilt: dβ = 0 ($u_2=0$ everywhere).
3. $S_{12}=\tfrac12(0.01+0)/0.01=0.5$ s⁻¹ $=\gamma/2$.
4. So $\gamma=2S_{12}$. ⚠️ Ch. 2's Ex. 2.4 called $S_{12}$ itself Γ; here γ is twice that.

In [ ]:
G = ch03.velocity_gradient_preset("simple_shear", Gamma=1.0)   # G = [[0, γ], [0, 0]], γ = 1 1/s: Gamma= is the preset's rate
                                                               # (γ here, ω0 for solid body) — NOT the circulation Γ
print(ch03.shear_strain_rate(G, [1, 0], [0, 1]))               # S12 = n1·S·n2 = 0.5 1/s
print(ch03.measured_strain_rates(G, [1, 0], 1e-6)["closing"]/2)   # half the measured closing rate of two threads
U, Om = np.array([0.3, -0.1, 0.2]), np.array([0.5, -1.0, 2.0])   # a translation [m/s] and a rotation [rad/s]
Gr = ch03.velocity_gradient_at(lambda x, t: ch03.rigid_body_velocity(U, Om, x), np.array([0.4, 0.1, -0.7]), 0.0)
print(np.round(ch03.strain_rate_tensor(Gr), 10) + 0.0)          # a 3 × 3 zero matrix: rigid motion does not deform
print(ch03.vorticity_from_gradient(Gr))                        # [ 1. -2.  4.] = 2Ω

**What does the code above do?**

1. $S_{12}$ from the formula $\mathbf n_1\cdot\mathbf S\cdot\mathbf n_2$.
2. Half the closing rate of two tracked perpendicular threads — the same 0.5 s⁻¹ (D09).
3. A rigid motion $\mathbf U+\boldsymbol\Omega\times\mathbf x$ (D10): its gradient has no symmetric part, and its vorticity
   is twice its angular velocity.

In [ ]:
ts = np.linspace(0, 1, 101)                                    # time [s]
flows = {"simple shear γ = 1 s⁻¹": (ch03.velocity_gradient_preset("simple_shear", 1.0), COLORS["accent"]),
         "pure strain along ±45°": (ch03.velocity_gradient_preset("irrotational_strain", 0.5), COLORS["teal"]),
         "solid-body rotation": (ch03.velocity_gradient_preset("solid_body_rotation", 1.0), COLORS["orange"])}
fig, ax = plt.subplots(figsize=(7.5, 3.8))                      # one panel
for name, (Gf, col) in flows.items():                           # three linear flows
    ang = [ch03.material_line_angle(Gf, t, np.pi/2) - ch03.material_line_angle(Gf, t, 0.0) for t in ts]   # angle between the threads
    s12 = ch03.strain_rate_tensor(Gf)[0, 1]                     # S12 of this flow [1/s]
    ax.plot(ts, np.degrees(ang), color=col, lw=2.3, label=f"{name} ($S_{{12}}$ = {s12:.1f} 1/s)")   # measured angle [°]
    ax.plot(ts[:40], 90 - np.degrees(2*s12*ts[:40]), ":", color=col, lw=1.2)   # initial tangent: slope −2S12
ax.set_xlabel("$t$ [s]"); ax.set_ylabel("angle between the threads [°]"); ax.legend(fontsize=8)   # axes with units
ax.set_title("A right angle closes at 2S₁₂ at first")
savefig(fig, "ch03", "closing_angle"); plt.show()               # save to outputs/ch03 and draw

**What you see.** The orange curve stays at 90°; the purple and teal curves fall, each starting along its dotted tangent.

**How to read it.** The initial slope is $-2S_{12}$ in rad/s: shear and pure strain close the angle at the same initial rate (both
have $S_{12}=0.5$ s⁻¹); rotation turns both threads together and closes nothing (S = 0). Later the curves part: the
shear keeps turning its threads, the pure strain pulls both toward its stretching axis.

**What would change if…** …the threads were turned by 45°: in the pure-strain flow their angle would stay 90° (they lie on the principal
axes) — the off-diagonal of S depends on the axes (C12).

**What would change if…** …three edges of a small box stretch at once? Then its volume changes — at the sum of the three linear rates (C09).

### 🧩 Volumetric strain rate: divergence is how fast a blob swells (3.14) `C09`

*The question:* A small parcel of air rises and expands. At what rate does its volume grow per unit volume, and how is that read off
the velocity field?

*In one line:* $\frac{1}{\delta V}\frac{D(\delta V)}{Dt}=\frac{\partial u_i}{\partial x_i}$ *(3.14)*

#### The problem in plain words

Rising air expands, sinking air is compressed; in the ocean water hardly changes volume at all. The fractional growth
rate of a parcel's volume is the kinematic half of mass conservation (Ch. 4: $D\rho/Dt=-\rho\nabla\cdot\mathbf u$). We
find it from the three stretching rates of C07.

#### The idea

```
δV = δx₁ δx₂ δx₃   each edge stretches at S₁₁, S₂₂, S₃₃   →   (1/δV) D(δV)/Dt = S₁₁ + S₂₂ + S₃₃ = ∇·u   (3.14)
shear only tilts the faces (second order);  the trace does not depend on the orientation of the box
```

In symbols: $\frac{1}{\delta V}\frac{D(\delta V)}{Dt}=\frac{\partial u_1}{\partial x_1}+\frac{\partial u_2}{\partial x_2}+\frac{\partial u_3}{\partial x_3}=\frac{\partial u_i}{\partial x_i}$ *(3.14)*.

#### 🧮 Derivation — Volumetric strain rate: (1/δV) D(δV)/Dt = S_ii, Eq. (3.14) `D11` (Eq. 3.14)

**What we want to show.** Show that the fractional rate at which a small volume of fluid grows is the divergence of the velocity — the book states it and leaves the proof to Exercise 3.18.

**Assumptions.** The box is small (D08 applies to each edge); first order in dt (step 5).

**The plan.**

1. Differentiate the product following the fluid.
2. Use the linear strain rate of each edge (D08).
3. Divide by δV.
4. Explain why shear does not count and why the answer does not depend on orientation.
5. The exact finite-time factor.

**Tools we use** (each explained before this point): Product rule for three factors (gloss in step 1; ch01 P38) · D08, $\frac{1}{\delta x_\eta}\frac{D(\delta x_\eta)}{Dt}=\frac{\partial u_\eta}{\partial x_\eta}$ · trace invariance (Ch. 2 §2.5) · Jacobi's formula (gloss in step 7).

**We start from**

$$ \delta V=\delta x_1\,\delta x_2\,\delta x_3 $$

*In words:* a small box with material edges along the axes.

---

**Step 1 of 7 — Product rule for three factors.**

$$ \dfrac{D(\delta V)}{Dt}=\dfrac{D\delta x_1}{Dt}\delta x_2\delta x_3+\delta x_1\dfrac{D\delta x_2}{Dt}\delta x_3+\delta x_1\delta x_2\dfrac{D\delta x_3}{Dt} $$

- *Why we can do this:* Differentiate a product of three factors: one term per factor, the others untouched (the two-factor rule of Ch. 1 applied twice).
- *In words:* The volume grows because each edge grows.

---

**Step 2 of 7 — Insert each edge's stretching rate.**

$$ \dfrac{D\delta x_\eta}{Dt}=\dfrac{\partial u_\eta}{\partial x_\eta}\,\delta x_\eta\quad(\text{no sum}) $$

- *Why we can do this:* D08 step 5: each material edge stretches at its own linear strain rate; η is a Greek index, so no summation.
- *In words:* Edge η grows at ∂u_η/∂x_η of its length.

---

**Step 3 of 7 — Substitute into the product rule.**

$$ \dfrac{D(\delta V)}{Dt}=\Big(\dfrac{\partial u_1}{\partial x_1}+\dfrac{\partial u_2}{\partial x_2}+\dfrac{\partial u_3}{\partial x_3}\Big)\delta x_1\delta x_2\delta x_3 $$

- *Why we can do this:* Each term of step 1 becomes one rate times the same product δx₁δx₂δx₃; factor it out.
- *In words:* Three stretching rates add up.

---

**Step 4 of 7 — Divide by δV.**

$$ \dfrac{1}{\delta V}\dfrac{D(\delta V)}{Dt}=\dfrac{\partial u_i}{\partial x_i}=S_{ii} $$

- *Why we can do this:* δV ≠ 0; the sum is the summation convention's $\partial u_i/\partial x_i=\nabla\cdot\mathbf u$, equal to the trace of S since $S_{ii}=G_{ii}$. This is (3.14), $\frac{1}{\delta V}\frac{D(\delta V)}{Dt}=\frac{\partial u_i}{\partial x_i}$.
- *In words:* The volume grows at ∇·u of itself per second.

---

**Step 5 of 7 — Shear changes volume only at second order.**

$$ \delta V(t+dt)=\delta V\,(1+S_{ii}\,dt)+O(dt^2) $$

- *Why we can do this:* The book skips this: shearing tilts the faces by angles ∝ dt; a parallelogram's area changes as cos of the tilt, 1 − O(dt²). Only edge stretching acts at first order.
- *In words:* Tilting a box does not change its volume to first order.

---

**Step 6 of 7 — Independence of the box's orientation.**

$$ S'_{ii}=S_{ii} $$

- *Why we can do this:* The trace is an invariant under a rotation of axes (Ch. 2 §2.5); a box turned any way gives the same rate.
- *In words:* Any small blob, any orientation, same fractional growth rate.

---

**Step 7 of 7 — The exact factor for a linear flow.**

$$ \dfrac{\delta V(t)}{\delta V(0)}=\det e^{\mathbf Gt}=e^{t\,\mathrm{tr}\,\mathbf G} $$

- *Why we can do this:* A linear flow maps positions by $e^{\mathbf Gt}$ (ch02 P79) and volumes scale by its determinant; Jacobi's formula det e^M = e^{tr M} (gloss, checked numerically). Step 5 is its first-order tangent.
- *In words:* Over finite times the growth compounds exponentially.

---

**Result**

$$ \frac{1}{\delta V}\frac{D}{Dt}(\delta V)=\frac{\partial u_1}{\partial x_1}+\frac{\partial u_2}{\partial x_2}+\frac{\partial u_3}{\partial x_3}=\frac{\partial u_i}{\partial x_i}=S_{ii}\ \text{(3.14)} $$

*In words:* the divergence is the fractional growth rate of a small blob of fluid.

**What it means.** Incompressible flow (∇·u = 0) keeps every blob's volume; compressible flow changes it. With mass conservation this becomes the continuity equation $D\rho/Dt=-\rho\nabla\cdot\mathbf u$ (Ch. 4). D23 derives the same result from the Reynolds transport theorem.

**Check it.** Units 1/s ✓. Simple shear: ∇·u = 0, volume kept ✓. Numbers: u = (x, y, z): 3 s⁻¹; a 1 cm³ box after 0.01 s → 1.03045 cm³ = e^{0.03} (first order 1.03) ✓; the random G of the notebook: det e^{Gt} = e^{t tr G} to 1e-12 ✓.

> ⚠️ **Common confusion (traps in `D11`):** Summing over the Greek index in step 2. Believing shear changes volume at first order. Using 1 + t tr G for finite times (the exact factor is e^{t tr G}).

#### ✏️ Tiny example: u = (x, y, z) s⁻¹ and a 1 cm³ box for 0.01 s

1. $\nabla\cdot\mathbf u=1+1+1=3$ s⁻¹.
2. First order: $\delta V\approx1\times(1+3\times0.01)=1.03$ cm³.
3. Exact for this linear flow (Jacobi's formula, D11 step 7): $e^{0.03}=1.03045$ cm³.
4. Simple shear $\mathbf u=(\gamma y,0)$: $\nabla\cdot\mathbf u=0$ — the sheared box keeps its volume although it changes
   shape.

In [ ]:
G3 = np.eye(3)                                            # u = (x, y, z): uniform expansion [1/s]
print(ch03.volumetric_strain_rate(G3), ch03.material_volume_ratio(G3, 0.01))   # tr G = 3 1/s; det e^{Gt} after 0.01 s
rng = np.random.default_rng(3)                            # reproducible random numbers (Ch. 1 P10)
Grand = rng.normal(size=(3, 3))                           # a random 3-D velocity gradient [1/s]
print(ch03.material_volume_ratio(Grand, 0.7), np.exp(0.7*np.trace(Grand)))   # Jacobi: det e^{Gt} = e^{t tr G}
C = ch03.rotation_matrix_2d(0.4)                          # axes turned by 0.4 rad
G2 = np.array([[1.0, 2.0], [0.0, -1.0]])                  # C06's G
print(np.trace(ch03.transform_tensor(G2, C)), np.trace(G2))   # the trace does not care about the axes

**What does the code above do?**

1. The trace of G is the volumetric rate $\frac{1}{\delta V}\frac{D(\delta V)}{Dt}=\frac{\partial u_i}{\partial x_i}$ *(3.14)*;
   after 0.01 s the exact volume factor is 1.030455.
2. The exact finite-time factor $\det e^{\mathbf Gt}$ equals $e^{t\,\mathrm{tr}\,\mathbf G}$ (Jacobi) for any G.
3. The trace does not change when the axes turn (Ch. 2 §2.5).

In [ ]:
for t in (0.01, 0.5):                                     # two times [s]
    M = ch03.linear_flow_map(G3, t)                       # carries every tracer: x(t) = e^{Gt} x(0)
    edges = M @ (0.01*np.eye(3))                          # the three 1 cm edge vectors of the cube after time t [m]
    vol = abs(np.linalg.det(edges))                       # volume of the carried cube [m³] (Ch. 1 P56)
    assert np.allclose(vol/1e-6, ch03.material_volume_ratio(G3, t))   # δV(t)/δV(0) = det e^{Gt}
    print(f"t = {t}: tracked cube {vol/1e-6:.6f} × its initial volume")

In [ ]:
ts = np.linspace(0, 1, 11)                                # time [s]
fig, ax = plt.subplots(figsize=(7, 3.8))                  # one panel
tracked = [abs(np.linalg.det(ch03.linear_flow_map(G3, t))) for t in ts]   # tracked cube, expansion G = I
ax.plot(ts, tracked, "o", color=COLORS["blue"], label="tracked cube, $\\mathbf{G}=\\mathbf{I}$")   # dots
tt = np.linspace(0, 1, 101)                               # a fine time axis [s]
ax.plot(tt, np.exp(3*tt), color=COLORS["blue"], lw=2, label="$e^{3t}$ (exact)")                  # Jacobi: e^{t tr G}
ax.plot(tt, 1 + 3*tt, "--", color=COLORS["muted"], label="$1+3t$ (first order, (3.14))")         # the tangent
Gsh = ch03.velocity_gradient_preset("simple_shear", 1.0)  # a shear: tr G = 0
ax.plot(tt, [ch03.material_volume_ratio(Gsh, t) for t in tt], color=COLORS["teal"], lw=2, label="simple shear: 1")
ax.set_xlabel("$t$ [s]"); ax.set_ylabel(r"$\delta V(t)/\delta V(0)$ [–]"); ax.legend(fontsize=8)   # dimensionless ratio
ax.set_title("Divergence sets the initial swelling rate; shear changes shape, not volume")
savefig(fig, "ch03", "volume_ratio"); plt.show()          # save to outputs/ch03 and draw

**What you see.** An exponential (blue) that leaves its dashed tangent line; a flat teal line for shear.

**How to read it.** $\frac{1}{\delta V}\frac{D(\delta V)}{Dt}=\frac{\partial u_i}{\partial x_i}$ *(3.14)* gives the initial slope,
3 s⁻¹ per unit volume; over finite time the growth compounds ($e^{3t}$). Shear alone changes shape, never volume.

**What would change if…** …G = −I (compression): the volume would decay as $e^{-3t}$ — a sinking, compressed parcel.

#### 🎮 Interactive: What does each number in S measure?

**Why interactive:** S has several entries and each measures a different motion; changing G one entry at a time and seeing which measured rate responds is the fastest way to learn what each number means.
Drag the four entries of G and watch a square element and a ring of tracers deform. The measured rates — stretching
per length along a probe direction, the closing rate of a right angle, the area growth — land on the formula rates
n·S·n, $2S_{12}$ and tr G, and the rose curve shows the stretching rate in every direction at once.

**What to try:**
- Preset 'solid-body rotation': every measured rate is zero — S = 0 (N27).
- Preset 'simple shear': rotate the probe to 45° and find the fastest stretching.
- Preset 'expansion': the area ratio follows the ghost e^{t tr G}.
- Click a tracer: the inspector gives its du = G·dx arithmetic (D07).

In [ ]:
show_viz("ch03", "fluid_element_deformation")   # full-width explainer; ⤢ Full screen for more room

**What would change if…** …you looked not at how the element deforms but at how it turns? The two perpendicular threads of C08 turn by −α and
+β; their *average* turning is the spin, and it is half the vorticity (C10).

> 🔁 **Recap — The components of the vorticity** (Ch. 2 §2.9). $\omega_1=\frac{\partial u_3}{\partial x_2}-\frac{\partial u_2}{\partial x_3},\ \omega_2=\frac{\partial u_1}{\partial x_3}-\frac{\partial u_3}{\partial x_1},\ \omega_3=\frac{\partial u_2}{\partial x_1}-\frac{\partial u_1}{\partial x_2}$
*(Eq. 3.16, the curl of Ch. 2's (2.25))* — `ch03.curl` on a grid, `ch03.vorticity(u, x, t)` at a point.

> 🔁 **Recap — Circulation** (Ch. 2 §2.13). $\Gamma\equiv\oint_C\mathbf u\cdot d\mathbf s=\int_A\boldsymbol\omega\cdot\mathbf n\,dA$ *(Eq. 3.18)* — Stokes' theorem,
$\oint_C\mathbf u\cdot\mathbf t\,ds=\int_A(\nabla\times\mathbf u)\cdot\mathbf n\,dA$ *(2.34)*: the circulation round a loop
equals the flux of vorticity through any surface it bounds, so **vorticity is circulation per unit area**,
$\mathbf n\cdot(\nabla\times\mathbf u)=\lim_{A\to0}\frac1A\oint_C\mathbf u\cdot\mathbf t\,ds$ *(2.35)*. ⚠️ Γ here is a
circulation [m²/s], not Ch. 2's shear rate. `ch03.circulation(u, ch03.planar_loop(...))` computes it. Used again in C13
and C14; Kelvin's theorem (Ch. 5) and lift (Ch. 6, 14) are built on it.

### 🧩 Vorticity is twice the element's spin — for any pair of lines, in any frame but its own `C10`

*The question:* Drop a tiny paddle wheel into a river where the water near the bank is slower. Does it turn, how fast, and does the
answer depend on which two sticks of the wheel you watch — or on whether you watch from the spinning Earth?

#### The problem in plain words

The vorticity $\boldsymbol\omega=\nabla\times\mathbf u$ is the most important derived field in atmosphere and ocean
dynamics (cyclones, eddies, potential vorticity). Its meaning is physical: a small paddle wheel carried by the flow
turns at half of it. We prove that from the two threads of C08 — and find that a flow in perfectly straight lines can
spin every element, while (C13) a flow in circles need not.

#### The idea

```
horizontal thread turns at +dβ/dt, vertical thread at −dα/dt (counterclockwise +)
spin of the element ≡ their average  = ½(−α̇ + β̇) = ½ω₃ = R₂₁/2                (Fig. 3.11)
single threads may turn at different rates (shear!) — the average of ANY perpendicular pair is the same
```

> 📎 **Primer — angular velocity of a line.** `P102` · A line segment at angle θ (counterclockwise from +x) turns at $\dot\theta=d\theta/dt$; counterclockwise is positive. If
its tip moves relative to its tail with velocity $\delta\mathbf u$, only the part of $\delta\mathbf u$ perpendicular to
the segment turns it: $\dot\theta=(\mathbf e_\theta\cdot\delta\mathbf u)/\ell$ with $\mathbf e_\theta=(-\sin\theta,\cos\theta)$.

In [ ]:
theta, ell = np.pi/2, 0.01                           # a vertical 1 cm thread
du = np.array([0.01, 0.0])                           # its tip moves 1 cm/s to the right relative to its tail
print(np.array([-np.sin(theta), np.cos(theta)]) @ du/ell)   # -1.0 rad/s: clockwise

#### 🧮 Derivation — The spin of a fluid element is half the vorticity: ½ D(−α + β)/Dt = ½ω₃ `D12`

**What we want to show.** Define how fast a small fluid element rotates and show that the rate is half the vorticity component ω₃ = R₂₁ (Fig. 3.11).

**Assumptions.** dt small; threads short (as in D09).

**The plan.**

1. Measure both turns counterclockwise-positive.
2. Define the element's rotation as their average (and say why).
3. Substitute the tilts and take the limit.
4. Recognise R₂₁ and ω₃.

**Tools we use** (each explained before this point): D09 steps 3–4 · angular velocity of a line (primer in C10) · (3.13) $R_{ij}=\partial u_i/\partial x_j-\partial u_j/\partial x_i$ (R03) · (3.15) $R_{ij}=-\varepsilon_{ijk}\omega_k$ (R05) · (3.16) $\omega_3=\partial u_2/\partial x_1-\partial u_1/\partial x_2$ (R06).

**We start from**

$$ \begin{array}{l}\text{The two perpendicular threads of D09: in dt the vertical thread tilts} \\ \text{clockwise by}\ d\alpha=(\partial u_1/\partial x_2)dt\ \text{and the horizontal thread counterclockwise by} \\ d\beta=(\partial u_2/\partial x_1)dt\end{array} $$

*In words:* in general the two threads turn by different amounts.

---

**Step 1 of 8 — Signed turns, counterclockwise positive.**

$$ d\theta_{\rm horiz}=+d\beta,\qquad d\theta_{\rm vert}=-d\alpha $$

- *Why we can do this:* Angles of lines are measured counterclockwise (primer); α was defined clockwise, so the vertical thread's turn enters with a minus sign.
- *In words:* One thread turns one way, the other the other way (in shear).

---

**Step 2 of 8 — Define the spin as their average.**

$$ \Omega_{\rm el}\equiv\tfrac12\dfrac{D(-\alpha+\beta)}{Dt} $$

- *Why we can do this:* The book skips the reason: averaging a perpendicular pair cancels the closing (shear) part, which turns the two threads oppositely, and keeps what turns the element as a whole.
- *In words:* The element's rotation = the average turning of two perpendicular threads.

---

**Step 3 of 8 — Substitute the two tilts.**

$$ \Omega_{\rm el}=\lim_{dt\to0}\dfrac{1}{2dt}\Big(-\dfrac{\partial u_1}{\partial x_2}dt+\dfrac{\partial u_2}{\partial x_1}dt\Big) $$

- *Why we can do this:* D09 steps 3–4 give dα and dβ to first order in dt.
- *In words:* Turn of each thread in dt, averaged.

---

**Step 4 of 8 — Cancel dt.**

$$ \Omega_{\rm el}=\tfrac12\Big(-\dfrac{\partial u_1}{\partial x_2}+\dfrac{\partial u_2}{\partial x_1}\Big) $$

- *Why we can do this:* dt cancels exactly, so the limit is immediate; the neglected pieces are order dt².
- *In words:* The spin rate in terms of two velocity slopes.

---

**Step 5 of 8 — Recognise the rotation tensor.**

$$ \Omega_{\rm el}=-\dfrac{R_{12}}{2}=\dfrac{R_{21}}{2} $$

- *Why we can do this:* (3.13): $R_{12}=\partial u_1/\partial x_2-\partial u_2/\partial x_1$, which is minus the bracket; R is antisymmetric, $R_{21}=-R_{12}$.
- *In words:* The spin is half an entry of the book's R.

---

**Step 6 of 8 — Recognise the vorticity.**

$$ \Omega_{\rm el}=\tfrac12\omega_3 $$

- *Why we can do this:* (3.15), $R_{ij}=-\varepsilon_{ijk}\omega_k$ with i = 2, j = 1: $R_{21}=-\varepsilon_{213}\omega_3=\omega_3$ (ε₂₁₃ = −1).
- *In words:* The element spins at half the vorticity.

---

**Step 7 of 8 — Cross-check with the curl.**

$$ \omega_3=\dfrac{\partial u_2}{\partial x_1}-\dfrac{\partial u_1}{\partial x_2} $$

- *Why we can do this:* (3.16), $\omega_3=\frac{\partial u_2}{\partial x_1}-\frac{\partial u_1}{\partial x_2}$ gives the same combination as step 4 without the ½ — the two routes agree.
- *In words:* Consistent with ω = ∇ × u.

---

**Step 8 of 8 — Repeat in the other two planes.**

$$ \boldsymbol\Omega_{\rm el}=\tfrac12\boldsymbol\omega $$

- *Why we can do this:* The same construction with threads in the (x₂, x₃) and (x₃, x₁) planes gives ½ω₁ and ½ω₂; the three rates form a vector.
- *In words:* A small element spins with angular velocity ω/2.

---

**Result**

$$ \tfrac12\frac{D(-\alpha+\beta)}{Dt}=\tfrac12\Big(-\frac{\partial u_1}{\partial x_2}+\frac{\partial u_2}{\partial x_1}\Big)=-\frac{R_{12}}2=\frac{R_{21}}2=\tfrac12\omega_3 $$

*In words:* vorticity is twice the spin of a small fluid element.

**What it means.** A paddle wheel carried by the flow turns at ½ω; the vorticity field tells where fluid spins. Because single threads may turn at different rates, "the" rotation needs this average — D14 shows it is the same for *every* perpendicular pair. The spin depends on the observer's own rotation (D13).

**Check it.** Units rad/s ✓. Solid-body rotation at ω₀: G = [[0, −ω₀], [ω₀, 0]]: ½(ω₀ + ω₀) = ω₀ — the element spins as fast as it revolves ✓ (Fig. 3.15). Shear γ: ½(−γ + 0) = −γ/2 ✓. Pure strain: 0 ✓.

> ⚠️ **Common confusion (traps in `D12`):** Averaging α + β (that is the strain rate, D09) instead of −α + β. Forgetting the ½ (ω is twice the spin). Sign: R₁₂ vs R₂₁.

**The running example: a parallel shear flow.** Near a river bank (or in the wind near the ground) the velocity is
$\mathbf u=(u_1(x_2),0)$ and locally $u_1\approx\gamma x_2$ with the shear rate $\gamma\equiv du_1/dx_2$. By
$\omega_3=\frac{\partial u_2}{\partial x_1}-\frac{\partial u_1}{\partial x_2}$ *(Eq. 3.16)*, $\omega_3=-\gamma$: straight
streamlines, yet nonzero vorticity — clockwise for γ > 0. How do its threads turn? (§3.5 re-states this flow as note `N34`.)

#### 🧮 Derivation — In a parallel shear flow every perpendicular pair of threads spins at −γ/2 `D14`

**What we want to show.** Show the book's claim (§3.5, stated without proof) that in the shear flow u = (γx₂, 0) the average turning rate of two perpendicular threads is −γ/2 = ω₃/2 whatever pair you pick — while single threads turn at different rates.

**Assumptions.** Threads short enough for the linear field (D07).

**The plan.**

1. Turning rate of one thread from the relative velocity of its tip.
2. Evaluate for this G.
3. Do the same for its perpendicular partner.
4. Average.

**Tools we use** (each explained before this point): Angular velocity of a line (primer in C10) · D07, $d\mathbf u=\mathbf G\cdot d\mathbf x$ · sin² + cos² = 1 · (3.16), $\omega_3=\frac{\partial u_2}{\partial x_1}-\frac{\partial u_1}{\partial x_2}$ (R06).

**We start from**

$$ \begin{array}{l}\text{Locally}\ \mathbf u=(\gamma x_2,0)\ \text{, so}\ \mathbf G=\begin{bmatrix}0&\gamma\\0&0\end{bmatrix}\ \text{; a material thread at angle θ has} \\ \text{unit direction}\ \mathbf e(\theta)=(\cos\theta,\sin\theta)\end{array} $$

*In words:* layers slide over each other at shear rate γ.

---

**Step 1 of 6 — Turning rate from the tip's velocity.**

$$ \dot\theta=\mathbf e_\theta\cdot\mathbf G\cdot\mathbf e(\theta),\qquad\mathbf e_\theta=(-\sin\theta,\cos\theta) $$

- *Why we can do this:* The tip of a unit thread moves relative to its tail by G·e (D07); only the part perpendicular to the thread turns it (primer: angular velocity of a line).
- *In words:* How fast a thread at angle θ turns.

---

**Step 2 of 6 — Multiply G by e(θ).**

$$ \mathbf G\cdot\mathbf e(\theta)=(\gamma\sin\theta,\ 0) $$

- *Why we can do this:* Matrix–vector product: the only nonzero entry, $G_{12}=\gamma$, multiplies the second component sin θ.
- *In words:* The tip is pushed along the flow in proportion to its height.

---

**Step 3 of 6 — Take the perpendicular part.**

$$ \dot\theta=-\gamma\sin^2\theta $$

- *Why we can do this:* Take the dot product of step 2 with $\mathbf e_\theta=(-\sin\theta,\cos\theta)$: $-\sin\theta\cdot\gamma\sin\theta+\cos\theta\cdot0$; only the perpendicular part turns the thread.
- *In words:* Every thread turns clockwise, fastest when vertical.

---

**Step 4 of 6 — Check the book's two threads.**

$$ \dot\theta(\pi/2)=-\gamma,\qquad\dot\theta(0)=0 $$

- *Why we can do this:* Substitute θ = 90° (the vertical thread AB) and θ = 0 (the horizontal thread BC) — the values the book reads off Fig. 3.14.
- *In words:* AB turns at −γ, BC not at all.

---

**Step 5 of 6 — The perpendicular partner.**

$$ \dot\theta(\theta+\pi/2)=-\gamma\cos^2\theta $$

- *Why we can do this:* Step 3 at θ + π/2, with sin(θ + π/2) = cos θ.
- *In words:* The partner turns at a different rate.

---

**Step 6 of 6 — Average the pair.**

$$ \tfrac12\big(-\gamma\sin^2\theta-\gamma\cos^2\theta\big)=-\tfrac{\gamma}{2}=\tfrac12\omega_3 $$

- *Why we can do this:* sin²θ + cos²θ = 1 removes θ; (3.16), $\omega_3=\frac{\partial u_2}{\partial x_1}-\frac{\partial u_1}{\partial x_2}$ gives $\omega_3=\partial u_2/\partial x_1-\partial u_1/\partial x_2=-\gamma$.
- *In words:* Every perpendicular pair spins at −γ/2, half the vorticity.

---

**Result**

$$ \dot\theta=-\gamma\sin^2\theta\ \text{for one thread;}\ \tfrac12[\dot\theta(\theta)+\dot\theta(\theta+\tfrac\pi2)]=-\gamma/2=\omega_3/2\ \text{for every θ} $$

*In words:* single threads disagree, every perpendicular pair agrees.

**What it means.** "Spin" is well defined even when threads turn at different rates, and it equals ½ω. A flow in straight lines (a river near its bank, wind near the ground) spins every element clockwise at γ/2 — the half of the shear that is rotation (C11). This is why D12's average of two particular threads is the right definition.

**Check it.** Units rad/s ✓. γ = 1 s⁻¹, θ = 30°: −0.25 and −0.75, average −0.5 ✓. Solid-body G = [[0, −ω₀], [ω₀, 0]] in the same formula gives θ̇ = ω₀ for every θ ✓.

> ⚠️ **Common confusion (traps in `D14`):** Sign: ω₃ = −γ is clockwise for γ > 0. Averaging threads that are not perpendicular. Confusing γ with ch02's Γ = S₁₂ (γ = 2S₁₂).

> 📎 **Primer — rotating frame of reference.** `P103` · An observer on a turntable (or on the Earth) turning at Ω about z sees every point that is fixed on the turntable at
rest, although it moves with $\boldsymbol\Omega\times\mathbf x$ in the lab. At the instant the two sets of axes
coincide, velocities are related by $\mathbf u_{\rm lab}=\boldsymbol\Omega\times\mathbf x+\mathbf u_{\rm rot}$. Unlike a
Galilean frame, this correction varies from point to point — so derivatives (and the vorticity) change. Ch. 4 §4.7
develops the full rotating-frame equations.

In [ ]:
Omega = np.array([0, 0, 0.5]); x = np.array([2.0, 0.0, 0.0])   # turntable at 0.5 rad/s; a point 2 m from the axis
u_lab = np.array([0.0, 1.0, 0.0])                    # a point on the disc, seen from the lab [m/s]
print(u_lab - np.cross(Omega, x))                    # [0. 0. 0.]: at rest for the rotating observer

#### 🧮 Derivation — Vorticity depends on the observer's rotation: ω′_z = ω_z − 2Ω `D13`

**What we want to show.** Show how the vorticity changes for an observer who rotates — the book says only that it does (Exercise 3.19); the answer is the relation between relative and absolute vorticity that geophysical fluid dynamics runs on.

**Assumptions.** Ω constant (step 3); the comparison is made at the instant the axes coincide, so components agree (step 2).

**The plan.**

1. Solve for the observer's velocity.
2. Take the curl of both sides.
3. Use the curl of a rigid rotation.
4. Read off the z-component and the special frame where the element looks still.

**Tools we use** (each explained before this point): Rotating frame of reference (primer in C10) · the curl is linear and a vector (Ch. 2 §2.9) · D10, ∇ × (Ω × x) = 2Ω (ω of a rigid rotation) · (3.16), $\omega_3=\frac{\partial u_2}{\partial x_1}-\frac{\partial u_1}{\partial x_2}$ (R06).

**We start from**

$$ \begin{array}{l}\mathbf u=\boldsymbol\Omega\times\mathbf x+\mathbf u'\ \text{with}\ \boldsymbol\Omega=\Omega\,\mathbf e_z\ \text{constant, at the instant the two sets of} \\ \text{axes coincide}\end{array} $$

*In words:* the lab velocity is the velocity the rotating observer measures plus the velocity of the observer's own frame at that point.

---

**Step 1 of 6 — Solve for the rotating observer's velocity.**

$$ \mathbf u'=\mathbf u-\boldsymbol\Omega\times\mathbf x $$

- *Why we can do this:* A point fixed on the rotating frame moves with Ω × x in the lab (rigid-body velocity, primer); the observer measures motion relative to that.
- *In words:* Subtract the frame's own motion.

---

**Step 2 of 6 — Take the curl of both sides.**

$$ \nabla\times\mathbf u'=\nabla\times\mathbf u-\nabla\times(\boldsymbol\Omega\times\mathbf x) $$

- *Why we can do this:* The curl is linear. At the instant the axes coincide both observers use the same coordinates, and the curl is a vector (same components), so ∇′× = ∇× here.
- *In words:* The observer's vorticity is the lab vorticity minus that of the frame's rotation.

---

**Step 3 of 6 — Curl of a rigid rotation.**

$$ \nabla\times(\boldsymbol\Omega\times\mathbf x)=2\boldsymbol\Omega $$

- *Why we can do this:* D10 step 4: a rigid rotation at Ω has vorticity 2Ω (also Ch. 2's Ex. 2.3, ∇ × (b × x) = 2b).
- *In words:* The frame itself carries vorticity 2Ω.

---

**Step 4 of 6 — Combine.**

$$ \boldsymbol\omega'=\boldsymbol\omega-2\boldsymbol\Omega $$

- *Why we can do this:* Substitute step 3 into step 2 with ω = ∇ × u and ω′ = ∇ × u′.
- *In words:* A rotating observer sees the vorticity reduced by twice its rotation rate.

---

**Step 5 of 6 — The z-component.**

$$ \omega'_z=\omega_z-2\Omega $$

- *Why we can do this:* Ω points along z, so only the z-component changes.
- *In words:* The book's statement, with the factor 2.

---

**Step 6 of 6 — The frame where the element is still.**

$$ \omega'_z=0\iff\Omega=\tfrac12\omega_z $$

- *Why we can do this:* Set step 5 to zero. Rotating with the element's own spin (½ω_z, D12) makes it look non-rotating — the book's 'co-rotating frame'.
- *In words:* Spin with the paddle wheel and it stops.

---

**Result**

$$ \omega'_z=\omega_z-2\Omega\ \text{, i.e.}\ \boldsymbol\omega=\boldsymbol\omega'+2\boldsymbol\Omega $$

*In words:* vorticity measured by a rotating observer is the absolute vorticity minus twice the frame's rotation rate.

**What it means.** On the Earth (Ω = 7.29 × 10⁻⁵ s⁻¹) the vorticity measured relative to the ground, ζ, and the absolute vorticity differ by the planetary vorticity (its local vertical part is f = 2Ω sin φ ≈ 10⁻⁴ s⁻¹) — often larger than ζ itself. Conservation laws for vorticity (Ch. 5, potential vorticity in Ch. 13) hold for the absolute one. Unlike S (D10), ω is not observer-independent.

**Check it.** Units 1/s ✓. Ω = 0: no change ✓. Solid body ω₀ = 1 s⁻¹ (ω_z = 2) seen from a turntable at Ω = 1 rad/s: ω′ = 0 — the tank looks at rest ✓. The notebook's numeric curl of u − Ω × x equals ω − 2Ω to 1e-8 ✓.

> ⚠️ **Common confusion (traps in `D13`):** Forgetting the factor 2 (the frame's *vorticity* is 2Ω). Confusing this with Galilean invariance (C05): a rotating frame is not Galilean. Applying ω′ = ω − 2Ω to components not along Ω.

In [ ]:
u_lab = lambda x, t: np.array([-0.8*x[1] + 0.3*x[0]*x[1], 1.2*x[0] + 0.1*x[1]**2])   # a smooth lab field [m/s]
Om = 0.7                                                         # the observer's rotation rate Ω about z [rad/s]
u_rot = lambda x, t: u_lab(x, t) - Om*np.array([-x[1], x[0]])     # step 1: u' = u − Ω × x (in the plane)
p = np.array([0.4, -0.3])                                        # a point [m]
w_lab, w_rot = ch03.vorticity(u_lab, p, 0.0)[2], ch03.vorticity(u_rot, p, 0.0)[2]   # numerical curls ω_z, ω'_z
print(f"omega = {w_lab:.6f}, omega' = {w_rot:.6f}, omega - 2*Omega = {w_lab - 2*Om:.6f}")
assert abs(w_rot - (w_lab - 2*Om)) < 1e-8                        # ω'_z = ω_z − 2Ω (D13 step 5)

**What does the code above do?**

The numerical curl of $\mathbf u-\boldsymbol\Omega\times\mathbf x$ equals the lab vorticity minus $2\Omega$ — the check D13 cites.

📝 **Note.** **Vorticity depends on the frame** `N28` — (unlike S, N27): an observer rotating with the element sees ω′ = 0; one rotating at $\Omega\,\mathbf e_z$ sees
$\omega'_z=\omega_z-2\Omega$. **Climate hook:** on the Earth the vorticity measured relative to the ground (the
*relative* vorticity ζ) and the one seen from space (the *absolute* vorticity) differ by the local normal component of
the planetary vorticity, $2\Omega\sin\varphi=f$ ($1.03\times10^{-4}$ s⁻¹ at 45°N) — larger than ζ of a typical
mid-latitude cyclone (≈ 10⁻⁵–10⁻⁴ s⁻¹). Ch. 4 §4.7 and Ch. 13 build on this.

$$ \omega'_z=\omega_z-2\Omega  $$

📝 **Note.** **Irrotational flow** `N29` — A flow with no vorticity anywhere. Then $\mathbf u$ can be written as the gradient of a *velocity potential*,
$u_i=\partial\phi/\partial x_i$, because $\nabla\times\nabla\phi=0$ (Ch. 2 §2.13). ⚠️ The converse needs a region
without holes (*simply connected*: every loop can be shrunk to a point without leaving the region): the line vortex of
C13 ($u_\theta=B/r$, B in m²/s) has ω = 0 everywhere except its axis, yet the circulation round the axis is 2πB and the "potential" φ = Bθ jumps by
2πB after one turn. Potential flow is Ch. 6.

$$ \boldsymbol\omega=0,\quad\text{or equivalently}\quad R_{ij}=\partial u_i/\partial x_j-\partial u_j/\partial x_i=0 \qquad \text{(3.17)} $$

In [ ]:
xs, ys = sp.symbols('x y')                                   # plane coordinates
print(ch03.potential_velocity(xs**2 - ys**2, [xs, ys]))      # u = ∇φ = (2x, −2y) and its curl: 0
ng = 60 if not FAST else 40                                  # grid nodes per side (even: no node on the axis)
xg = np.linspace(-3, 3, ng)                                  # grid around a line vortex [m]
phi, jump = ch03.velocity_potential_2d(ch03.vortex_velocity_field("line", Gamma=2*np.pi, sigma=1.0), (xg, xg),
                                       x_ref=(-3.0, -3.0))   # φ by line integrals along two routes from a corner
print(f"route dependence of phi: {jump:.3f} m^2/s   vs   2*pi*B = {2*np.pi*1.0:.3f} m^2/s")

**What does the code above do?**

1. A potential always gives an irrotational field: ∇φ of $x^2-y^2$ and its curl, 0 (sympy).
2. Around the line vortex (B = Γ/2π = 1 m²/s, irrotational except on its axis) the line integral of u·ds from a corner
   depends on the route: the two routes differ by the circulation 2πB — no single-valued potential exists in a region
   with a hole.

#### ✏️ Tiny example: solid-body rotation and shear

1. Solid body, ω₀ = 1 rad/s: $\mathbf G=[[0,-1],[1,0]]$ s⁻¹; $\omega_3=1-(-1)=2$ s⁻¹; every thread turns at
   $\dot\theta=1$ rad/s; spin = ½ω₃ = 1 rad/s — the element spins as fast as it revolves.
2. Shear γ = 1 s⁻¹: $\dot\theta=-\sin^2\theta$: the horizontal thread 0, the vertical −1 rad/s, a thread at 30° −0.25
   rad/s and its partner at 120° −0.75 rad/s; each pair averages −0.5 = ω₃/2.
3. Watch the solid body from a turntable at Ω = 1 rad/s: $\omega'_z=2-2\times1=0$ — the tank looks at rest.

In [ ]:
Gsb = ch03.velocity_gradient_preset("solid_body_rotation", Gamma=1.0)   # G = [[0, −1], [1, 0]]: ω0 = 1 rad/s
Gsh = ch03.velocity_gradient_preset("simple_shear", Gamma=1.0)          # G = [[0, 1], [0, 0]]: γ = 1 1/s
print(ch03.element_rotation_rate(Gsb), ch03.element_rotation_rate(Gsh))   # spin = ½ω: (0, 0, 1) and (0, 0, −0.5)
th = np.deg2rad([0, 30, 90, 120])                                        # four single threads
print(np.round(ch03.material_line_rotation_rate(Gsh, th), 4))            # −γ sin²θ: 0, −0.25, −1, −0.75 rad/s
pairs = np.linspace(0, np.pi, 7)                                         # seven perpendicular pairs
print(0.5*(ch03.material_line_rotation_rate(Gsh, pairs) + ch03.material_line_rotation_rate(Gsh, pairs + np.pi/2)))
print(ch03.vorticity_in_rotating_frame(2.0, 1.0))                        # ω' = ω − 2Ω = 2 − 2 = 0 (D13)
print(ch03.parallel_shear_kinematics(1.0)["omega3"])                     # the shear flow's ω3 = −γ

**What does the code above do?**

1. Spin = ½ω from G (D12).
2. Single threads in the shear flow turn at $-\gamma\sin^2\theta$ (D14 step 3).
3. Every perpendicular pair averages −γ/2 (D14 step 6).
4. The rotating-observer rule $\omega'_z=\omega_z-2\Omega$ (D13).
5. The shear flow's $\omega_3=-\gamma$.

In [ ]:
thd = np.linspace(0, 180, 181)                                 # thread direction [°]

def rates(g):                                                  # turning rates in the shear flow with rate g [1/s]
    Gg = ch03.velocity_gradient_preset("simple_shear", Gamma=g)          # G = [[0, g], [0, 0]]
    one = ch03.material_line_rotation_rate(Gg, np.deg2rad(thd))            # a single thread
    partner = ch03.material_line_rotation_rate(Gg, np.deg2rad(thd) + np.pi/2)   # its perpendicular partner
    return {"one thread θ̇(θ)": (thd, one), "its partner θ̇(θ + 90°)": (thd, partner),
            "pair average = ω₃/2": (thd, 0.5*(one + partner))}            # name → (x, y)

fig = slider_figure(rates, "γ", np.linspace(-2, 2, 21 if not FAST else 11), unit="1/s", xlabel="thread angle θ [°]",
                    ylabel="turning rate [rad/s]", yrange=[-2.2, 2.2],     # fixed y axis
                    title="Single threads disagree, every pair agrees: spin = ω₃/2")
recolor(fig, {"one thread θ̇(θ)": COLORS["teal"], "its partner θ̇(θ + 90°)": COLORS["teal"],   # threads teal
              "pair average = ω₃/2": COLORS["orange"]}, dashes={"its partner θ̇(θ + 90°)": "dash"})   # rotation orange
fig.show()                                                     # draw it

**What does the code above do?**

`material_line_rotation_rate` gives $\dot\theta$ for every direction at once; the slider changes the shear rate γ.

**What you see.** Two teal curves (solid and dashed) and a flat orange line between them.

**How to read it.** The two teal curves mirror each other about −γ/2 and the orange line never bends; at γ = 0 everything is zero.

**What would change if…** …the flow were solid-body rotation (not on this slider): both teal curves would be flat at ω₀ — every thread turns alike.

**What would change if…** …you split the relative velocity du = G·dx of C06 into the part from S and the part from R? The R part is exactly a
rigid rotation at ω/2 — C11.

### 🧩 Deformation plus rigid rotation: du = S·dx + ½ω × dx (3.19) `C11`

*The question:* The arrow from one fluid particle to its neighbour's velocity can be split into two arrows. What are they, and why
does only one of them matter for friction?

*In one line:* $du_i=S_{ij}dx_j+\tfrac12(\boldsymbol\omega\times d\mathbf x)_i$ *(3.19)*

#### The problem in plain words

A fluid that spins like a rigid body (a stirred cup after it settles) feels no internal friction; a fluid that is
sheared does. To write the friction law of Ch. 4 we must separate, near every point, the motion that deforms from the
motion that merely rotates.

#### The idea

```
du   =   S·dx          +     ½ ω × dx                 (3.19)
total    deformation         rigid rotation at angular velocity ω/2 (same as Ω × x with Ω = ω/2)
purple   teal                orange
```

In symbols: $du_i=S_{ij}dx_j+\tfrac12(\boldsymbol\omega\times d\mathbf x)_i$ *(3.19)*.

#### 🧮 Derivation — Relative velocity = deformation + rigid rotation, Eq. (3.19) `D15` (Eq. 3.19)

**What we want to show.** Split the relative velocity of a neighbour into a pure deformation and a rigid rotation, and find the rotation's angular velocity.

**Assumptions.** As in D07 (small dx).

**The plan.**

1. Split the gradient into S and ½R.
2. Replace R by the vorticity.
3. Recognise a cross product — carefully, one index swap flips a sign.
4. Read the rotation's angular velocity.

**Tools we use** (each explained before this point): (3.11) $\partial u_i/\partial x_j=S_{ij}+\tfrac12R_{ij}$ (R01) · (3.15) $R_{ij}=-\varepsilon_{ijk}\omega_k$ (R05) · antisymmetry of ε and $(\mathbf a\times\mathbf b)_i=\varepsilon_{ijk}a_jb_k$, Eq. (2.21) (Ch. 2 §2.7) · rigid-body velocity Ω × x (primer in C08).

**We start from**

$$ \text{(3.10):}\ du_i=(\partial u_i/\partial x_j)\,dx_j $$

*In words:* the neighbour's relative velocity is the velocity gradient times the separation.

---

**Step 1 of 6 — Split the gradient.**

$$ du_i=\big(S_{ij}+\tfrac12R_{ij}\big)dx_j $$

- *Why we can do this:* (3.11), $\frac{\partial u_i}{\partial x_j}=S_{ij}+\tfrac12R_{ij}$: every velocity gradient is its symmetric part plus half the book's rotation tensor (R01).
- *In words:* Two kinds of relative motion.

---

**Step 2 of 6 — Replace R by the vorticity.**

$$ du_i=\big(S_{ij}-\tfrac12\varepsilon_{ijk}\omega_k\big)dx_j $$

- *Why we can do this:* (3.15), $R_{ij}=-\varepsilon_{ijk}\omega_k$ (the book's text cites (3.14), $\frac{1}{\delta V}\frac{D(\delta V)}{Dt}=\frac{\partial u_i}{\partial x_i}$, a slip — it is (3.15)).
- *In words:* The antisymmetric part is carried by the three numbers of ω.

---

**Step 3 of 6 — Distribute over dx_j.**

$$ du_i=S_{ij}dx_j-\tfrac12\varepsilon_{ijk}\omega_kdx_j $$

- *Why we can do this:* Multiply out the bracket; the two terms can now be read separately.
- *In words:* Deformation part and rotation part.

---

**Step 4 of 6 — Swap two indices of ε.**

$$ -\tfrac12\varepsilon_{ijk}\omega_kdx_j=\tfrac12\varepsilon_{ikj}\omega_kdx_j $$

- *Why we can do this:* ε changes sign when two indices are swapped (Ch. 2 §2.7); swapping j and k absorbs the minus sign. The book skips this — the step that fixes the sign.
- *In words:* Rewrite the rotation term with its indices in cross-product order.

---

**Step 5 of 6 — Recognise the cross product.**

$$ \tfrac12\varepsilon_{ikj}\omega_kdx_j=\tfrac12(\boldsymbol\omega\times d\mathbf x)_i $$

- *Why we can do this:* (2.21), $(\mathbf a\times\mathbf b)_i=\varepsilon_{ijk}a_jb_k$, with a = ω carrying the second index (k) and b = dx the third (j).
- *In words:* The rotation part is ½ω × dx.

---

**Step 6 of 6 — Assemble (3.19).**

$$ du_i=S_{ij}dx_j+\tfrac12(\boldsymbol\omega\times d\mathbf x)_i $$

- *Why we can do this:* Put steps 3–5 together. The second term has the form of the rigid-body velocity Ω × x with $\boldsymbol\Omega=\boldsymbol\omega/2$ (primer).
- *In words:* Near any point the fluid deforms by S and turns rigidly at ω/2.

---

**Result**

$$ du_i=\big(S_{ij}-\tfrac12\varepsilon_{ijk}\omega_k\big)dx_j=S_{ij}dx_j+\tfrac12(\boldsymbol\omega\times d\mathbf x)_i\ \text{(3.19)} $$

*In words:* relative velocity = pure deformation + rigid rotation at half the vorticity.

**What it means.** Only the S part changes distances between particles, so only S can produce internal friction — the reason the Newtonian stress law (Ch. 4 §4.5) is built on S. The rotation part is invisible to a co-rotating observer (D13).

**Check it.** Units m/s ✓. The rotation part is perpendicular to dx (½ω × dx · dx = 0): it changes no distance ✓. Numbers (shear γ = 1, dx = (0, 1)): S·dx = (0.5, 0), ½ω × dx = ½(0, 0, −1) × (0, 1, 0) = (0.5, 0, 0), sum (1, 0) = G·dx ✓.

> ⚠️ **Common confusion (traps in `D15`):** Missing the sign flip in step 4 (getting −½ω × dx). Writing ω instead of ω/2 as the angular velocity. Citing (3.14), $\frac{1}{\delta V}\frac{D(\delta V)}{Dt}=\frac{\partial u_i}{\partial x_i}$ instead of (3.15), $R_{ij}=-\varepsilon_{ijk}\omega_k$.

📝 **Note.** **A rigid rotation at half the vorticity** `N30` — The second term has the form of the rigid-body velocity $\mathbf v=\boldsymbol\Omega\times\mathbf x$ (primer P101) with
$\boldsymbol\Omega=\boldsymbol\omega/2$: near any point, the fluid turns rigidly at half its vorticity.

#### ✏️ Tiny example: shear γ = 1 s⁻¹, neighbour dx = (0, 1) m

1. $d\mathbf u=\mathbf G\cdot d\mathbf x=(\gamma\cdot1,0)=(1,0)$ m/s.
2. Strain part $\mathbf S\cdot d\mathbf x$ with $\mathbf S=[[0,\tfrac12],[\tfrac12,0]]$: (0.5, 0).
3. Rotation part with $\boldsymbol\omega=(0,0,-1)$:
   $\tfrac12\boldsymbol\omega\times d\mathbf x=\tfrac12(0\cdot0-(-1)\cdot1,\ (-1)\cdot0-0\cdot0,\ 0)=(0.5,0,0)$.
4. Sum (1, 0) ✓: half of the shear is stretching along 45°, half is a clockwise spin.

In [ ]:
du, du_s, du_r = ch03.relative_velocity_split(Gsh, [0.0, 1.0])   # (3.19) for the shear γ = 1, dx = (0, 1) m
print(du, du_s, du_r)                                            # total, strain part, rotation part [m/s]
dx3 = np.array([0.2, -0.1, 0.3])                                 # a 3-D neighbour [m]
du, du_s, du_r = ch03.relative_velocity_split(Grand, dx3)        # the random G of C09
w = ch03.vorticity_from_gradient(Grand)                          # ω = vector of R
print(np.allclose(du_r, 0.5*np.cross(w, dx3)), np.dot(du_r, dx3))   # rotation part = ½ω × dx, perpendicular to dx

**What does the code above do?**

1. The split for the worked example: $(1,0)=(0.5,0)+(0.5,0)$ m/s.
2. In 3-D the rotation part is exactly ½ω × dx and is perpendicular to dx (dot product ≈ 0) — it never changes the
   distance to the neighbour.

In [ ]:
eps = ch03.levi_civita()                                         # the ε_ijk array (Ch. 2 §2.7; three axes, Ch. 2 P73)
rot = np.zeros(3)                                                # the rotation part, built term by term [m/s]
for i in range(3):                                               # the index form −½ ε_ijk ω_k dx_j of (3.19), as loops
    for j in range(3):                                           # j: summed (repeated index)
        for k in range(3):                                       # k: summed (repeated index)
            rot[i] += -0.5*eps[i, j, k]*w[k]*dx3[j]              # one of the 27 terms
assert np.allclose(rot, du_r)                                    # = the library's rotation part
assert np.allclose(ch03.strain_rate_tensor(Grand) @ dx3 + rot, Grand @ dx3)   # S·dx + rotation = G·dx
print("triple loop:", rot)

The loops implement $du_i=\big(S_{ij}-\tfrac12\varepsilon_{ijk}\omega_k\big)dx_j$ — the first form of
$du_i=S_{ij}dx_j+\tfrac12(\boldsymbol\omega\times d\mathbf x)_i$ *(Eq. 3.19)* in D15.

In [ ]:
from scripts.ch03_drawings import ring_arrows                    # (no physics inside)
fig, axs = plt.subplots(1, 3, figsize=(12, 4))                   # total, strain part, rotation part
for ax, part, ttl in zip(axs, ("total", "strain", "rotation"),
                         ("total du = G·dx", "strain part S·dx", "rotation part ½ω × dx")):   # one panel per part
    ring_arrows(ax, Gsh, radius=0.1, parts=(part,), n=24, scale=0.6)   # 24 neighbours 10 cm away, shear γ = 1
    ax.plot(0, 0, "o", color=COLORS["ink"]); ax.set_aspect("equal")   # the centre point O
    ax.set_xlim(-0.17, 0.17); ax.set_ylim(-0.17, 0.17); ax.set_title(ttl); ax.set_xlabel("$dx_1$ [m]")   # same axes
axs[0].set_ylabel("$dx_2$ [m]")                                  # shared y label
plt.suptitle("Simple shear = pure strain (teal) + clockwise rigid spin at 0.5 rad/s (orange)")   # the message
savefig(fig, "ch03", "ring_split"); plt.show()                   # save to outputs/ch03 and draw

**What you see.** Purple arrows that are all horizontal; teal arrows that fan out along 45° and in along 135°; orange arrows that circulate clockwise.

**How to read it.** The shear's horizontal arrows are, arrow by arrow, the sum of the teal and orange ones: a pure strain that would
turn the ring into an ellipse, plus a rigid clockwise spin at 0.5 rad/s that turns it.

**What would change if…** …G = [[0, −1], [1, 0]] (solid body): the teal arrows would vanish and the purple arrows would be the orange ones.

📝 **Note.** **Summary of §3.4** `N33` — The relative motion near a point = a rigid rotation of the element (at ω/2) + a deformation (S), which itself = pure
stretching along three perpendicular principal axes (next block).

**What would change if…** …you turned the axes to where S has no off-diagonal entries? Then the deformation is three pure stretchings — a small
sphere becomes an ellipsoid (C12).

### 🧩 Principal strain axes: a small sphere becomes an ellipsoid `C12`

*The question:* A round drop of dye is released into a flow. What shape is it a moment later, and which way do its long and short axes
point?

#### The problem in plain words

Satellite images of ocean colour show round patches of plankton pulled into ellipses and then filaments; weather fronts
form where the wind's deformation squeezes temperature contours together (*frontogenesis*, Ch. 13). Both are the
principal axes of the strain-rate tensor at work.

#### The idea

```
in the principal frame (overbar) S is diagonal:  dū₁ = S̄₁₁ dx̄₁,  dū₂ = S̄₂₂ dx̄₂,  dū₃ = S̄₃₃ dx̄₃     (3.21)
each principal direction stretches in proportion to its own length  →  circle → ellipse on those axes (Fig. 3.13)
```

In symbols: $d\bar u_\alpha=\bar S_{\alpha\alpha}\,d\bar x_\alpha$ (no sum) *(3.21)*.

📝 **Note.** **The strain part in the principal frame** `N31` — In the frame of the principal axes of S (Ch. 2 §2.11: a symmetric tensor has three perpendicular eigenvectors, primer
P80; ⚠️ the book's "Section 2.12" reference here means §2.11), the strain part of
$du_i=S_{ij}dx_j+\tfrac12(\boldsymbol\omega\times d\mathbf x)_i$ *(3.19)* is diagonal:

$$ d\bar{\mathbf u}=\bar{\mathbf S}\cdot d\bar{\mathbf x}=\begin{bmatrix}\bar S_{11}&0&0\\0&\bar S_{22}&0\\0&0&\bar S_{33}\end{bmatrix}\begin{bmatrix}d\bar x_1\\d\bar x_2\\d\bar x_3\end{bmatrix} \qquad \text{(3.20)} $$

📝 **Note.** **Its three components** `N32` — with $\bar S_{\alpha\alpha}$ the eigenvalues of S (Greek index: no sum):

$$ d\bar u_1=\bar S_{11}d\bar x_1,\quad d\bar u_2=\bar S_{22}d\bar x_2,\quad d\bar u_3=\bar S_{33}d\bar x_3 \qquad \text{(3.21)} $$

> 📎 **Primer — linear map of a circle is an ellipse.** `P104` · Multiplying every point of a unit circle by a matrix M gives an ellipse. For a symmetric M its axes are M's
eigenvectors and its semi-axes the eigenvalues; for a general M they are the *singular values* and singular vectors
(`np.linalg.svd`) — a gloss we need only for the finite-time caveat (an ellipse with semi-axes $a_\alpha$ along the
coordinate axes obeys $\sum_\alpha X_\alpha^2/a_\alpha^2=1$).

In [ ]:
M = np.array([[1.1, 0.0], [0.0, 0.9]])               # stretch x by 10 %, squeeze y by 10 %
s = np.linspace(0, 2*np.pi, 400); Pc = M @ np.stack([np.cos(s), np.sin(s)])   # the mapped unit circle
print(round(Pc[0].max(), 4), round(Pc[1].max(), 4))  # 1.1 0.9: the semi-axes

#### 🧮 Derivation — A small sphere becomes an ellipsoid on the principal axes of S `D16`

**What we want to show.** Show the book's closing claim of §3.4 (stated without proof): in a short time a small sphere of fluid becomes an ellipsoid whose axes are the principal axes of the strain-rate tensor.

**Assumptions.** First order in dt (step 4); the sphere is small (D07).

**The plan.**

1. Set the rotation aside (it keeps shapes).
2. Go to the principal frame where S is diagonal: (3.20), $d\bar{\mathbf u}=\bar{\mathbf S}\cdot d\bar{\mathbf x}$, (3.21), $d\bar u_\alpha=\bar S_{\alpha\alpha}\,d\bar x_\alpha$.
3. Move each point for dt.
4. Substitute into the sphere's equation and recognise an ellipsoid.
5. Say what happens for finite times.

**Tools we use** (each explained before this point): D15 · rigid-body velocity (primer in C08) · eigenvalues and eigenvectors of a symmetric tensor (ch02 P80) · tensor components in rotated axes (Ch. 2 §2.4) · linear map of a circle is an ellipse (primer in C12).

**We start from**

$$ \text{(3.19):}\ du_i=S_{ij}dx_j+\tfrac12(\boldsymbol\omega\times d\mathbf x)_i\ \text{for points dx on a sphere}\ \lvert d\mathbf x\rvert=\varepsilon $$

*In words:* each surface point moves by a strain part and a rigid rotation.

---

**Step 1 of 8 — Set the rigid rotation aside.**

$$ d\mathbf u_{\rm rot}=\tfrac12\boldsymbol\omega\times d\mathbf x\ \Rightarrow\ \text{shape unchanged} $$

- *Why we can do this:* A rigid rotation (primer) keeps all distances — it turns the finished shape by ½ω dt but cannot change a sphere into anything else. So only S shapes it.
- *In words:* The spin turns the blob; the strain reshapes it.

---

**Step 2 of 8 — Rotate to the principal axes.**

$$ d\bar{\mathbf u}=\bar{\mathbf S}\cdot d\bar{\mathbf x},\qquad\bar{\mathbf S}=\mathrm{diag}(\bar S_{11},\bar S_{22},\bar S_{33}) $$

- *Why we can do this:* S is symmetric, so it has three perpendicular eigenvectors (ch02 P80); in axes along them its components are diagonal (tensor rule, Ch. 2 §2.4). This is (3.20), $d\bar{\mathbf u}=\bar{\mathbf S}\cdot d\bar{\mathbf x}$.
- *In words:* In the right axes the strain is three plain stretchings.

---

**Step 3 of 8 — Write the three components.**

$$ d\bar u_\alpha=\bar S_{\alpha\alpha}\,d\bar x_\alpha\quad(\text{no sum}) $$

- *Why we can do this:* A diagonal matrix times a vector multiplies each component by its own diagonal entry. This is (3.21), $d\bar u_\alpha=\bar S_{\alpha\alpha}\,d\bar x_\alpha$.
- *In words:* Each principal coordinate grows in proportion to itself.

---

**Step 4 of 8 — Move each point for dt.**

$$ d\bar x_\alpha(t+dt)=d\bar x_\alpha\,(1+\bar S_{\alpha\alpha}\,dt) $$

- *Why we can do this:* New position = old + velocity × dt (first order in dt), with step 3's velocity.
- *In words:* Along the stretching axis points move out, along the compressing axis in.

---

**Step 5 of 8 — The sphere's equation before moving.**

$$ \textstyle\sum_\alpha(d\bar x_\alpha)^2=\varepsilon^2 $$

- *Why we can do this:* The points start on a sphere of radius ε; in any orthonormal axes the squared distance is the sum of squares.
- *In words:* The starting shape.

---

**Step 6 of 8 — Substitute the moved positions.**

$$ \textstyle\sum_\alpha\Big[\dfrac{d\bar x_\alpha(t+dt)}{1+\bar S_{\alpha\alpha}dt}\Big]^2=\varepsilon^2 $$

- *Why we can do this:* Invert step 4 for each coordinate (1 + S̄dt ≠ 0 for small dt) and put the old coordinates into step 5.
- *In words:* The equation obeyed by the moved points.

---

**Step 7 of 8 — Recognise an ellipsoid.**

$$ a_\alpha=\varepsilon\,(1+\bar S_{\alpha\alpha}\,dt) $$

- *Why we can do this:* $\sum_\alpha X_\alpha^2/a_\alpha^2=1$ is an ellipsoid with semi-axes $a_\alpha$ along the coordinate axes — here the principal axes of S (gloss).
- *In words:* Longest along the largest eigenvalue, shortest along the smallest.

---

**Step 8 of 8 — Say how long it holds.**

$$ \text{finite }t:\ \text{axes}=\text{singular vectors of }e^{\mathbf Gt} $$

- *Why we can do this:* The book skips this: over finite times rotation and strain act together; the exact shape comes from $e^{\mathbf Gt}$ (primer), whose axes drift from S's when ω ≠ 0 (shear: 43.6° at t = 0.1 s).
- *In words:* The principal-axis picture is exact only at the first instant.

---

**Result**

$$ \begin{array}{l}\text{In a short time dt a sphere of radius ε becomes an ellipsoid with semi-axes} \\ \varepsilon(1+\bar S_{\alpha\alpha}dt)\ \text{along the principal axes of S, turned by ½ω dt}\end{array} $$

*In words:* S decides the shape, ω/2 its orientation.

**What it means.** Blobs of dye, plankton patches and temperature anomalies are pulled out along the stretching axis — filaments in the ocean, fronts in the atmosphere (frontogenesis, Ch. 13). In pure strain the axes stay put; in shear the rotation keeps turning the ellipse toward the flow direction.

**Check it.** Units: S̄dt dimensionless ✓. Volume ratio $\prod(1+\bar S_{\alpha\alpha}dt)\approx1+S_{ii}dt$ = (3.14), $\frac{1}{\delta V}\frac{D(\delta V)}{Dt}=\frac{\partial u_i}{\partial x_i}$ ✓. Numbers (shear γ = 1, ε = 1 mm, dt = 0.1 s): 1.05 and 0.95 mm at ±45° (first order); strain alone e^{±0.05} = 1.0513, 0.9512; exact 1.0512, 0.9512 at 43.6° ✓.

> ⚠️ **Common confusion (traps in `D16`):** Expecting the ellipse to stay at 45° in a shear flow for long times. Forgetting the rotation part turns the shape. Reading S̄ as "the S of the original axes".

#### ✏️ Tiny example: shear γ = 1 s⁻¹, a 1 mm drop after 0.1 s

1. Principal rates ±γ/2 = ±0.5 s⁻¹ at +45° and −45° (Ch. 2 Ex. 2.4, now with γ = 2S₁₂).
2. First order (D16): semi-axes $1\times(1\pm0.5\times0.1)=1.05$ and 0.95 mm.
3. The strain acting alone: $e^{\pm0.05}=1.0513$ and 0.9512 mm.
4. Strain + rotation together (the true shear flow): singular values of $e^{\mathbf Gt}=[[1,0.1],[0,1]]$: 1.0512 and
   0.9512 mm, long axis at 43.6° — turned 1.4° clockwise off 45°. The rotation part alone would turn it by 0.5 rad/s × 0.1 s ≈ 2.9°, but
   the strain keeps pulling the long axis back toward 45°, so the net early turning rate is about γ/4, not γ/2.

In [ ]:
lam, axes = ch03.principal_strain_rates(Gsh)                     # eigenvalues (ascending) and unit eigenvectors of S
print(lam, np.round(axes[:, 1], 4))                              # [−0.5, 0.5]; the stretching axis (1, 1)/√2
for m in ("first_order", "strain_only", "exact"):                # three answers to "what shape after 0.1 s?"
    a, d = ch03.strain_ellipse_axes(Gsh, 0.1, method=m)          # semi-axes (descending) and their directions
    print(f"{m:12s} semi-axes {np.round(a, 4)}, long axis at {np.degrees(np.arctan2(d[1, 0], d[0, 0])):.1f}°")
du_bar, dx_bar = ch03.strain_velocity_principal(Gsh, [0.001, 0.0])   # (3.21) in the eigenframe
print(du_bar/dx_bar)                                             # each component: its own eigenvalue

**What does the code above do?**

1. Eigenvalues and eigenvectors of S (Ch. 2 P80), ordered like λ; `np.arctan2` (Ch. 2 P70) turns a direction into an angle.
2. Three answers to "what shape after 0.1 s": the book's first-order statement (1 ± λt), the strain alone ($e^{\lambda t}$)
   and the exact linear flow (singular values of $e^{\mathbf Gt}$).
3. In the eigenframe each velocity component is its own eigenvalue × its own coordinate,
   $d\bar u_\alpha=\bar S_{\alpha\alpha}\,d\bar x_\alpha$ *(3.21)*.

In [ ]:
nfr = 48 if not FAST else 24                                     # frames
times = np.linspace(0, 3, nfr)                                   # t from 0 to 3 s in the shear γ = 1
fig, ax = plt.subplots(figsize=(5.2, 4.6))                      # one panel, fixed limits (never autoscale)
ax.set_aspect("equal"); ax.set_xlim(-3.2, 3.2); ax.set_ylim(-2.2, 2.2)   # a 1 mm drop, axes in mm
L = 2.0                                                          # half-length of the drawn axes [mm]
ax.plot([-L, L], [-L, L], "--", color=COLORS["blue"], lw=1.2, label="stretching axis of S (45°)")      # eigenvector of +γ/2
ax.plot([-L, L], [L, -L], "--", color=COLORS["rose"], lw=1.2, label="compressing axis of S (−45°)")   # eigenvector of −γ/2
(ring,) = ax.plot([], [], "o", color=COLORS["teal"], ms=3, label="72 tracers")          # updated every frame
(axis_line,) = ax.plot([], [], color=COLORS["accent"], lw=2.2, label="current long axis")   # updated every frame
txt = ax.text(-3.0, 1.9, "", fontsize=9)                         # readout
ax.legend(fontsize=7, loc="lower right"); ax.set_xlabel("$x_1$ [mm]"); ax.set_ylabel("$x_2$ [mm]")
ax.set_title("A circle in simple shear: the ellipse starts on S's axes, then tips")

def update(i):                                                   # draw frame i
    t = times[i]                                                 # time of this frame [s]
    c = ch03.deform_circle(Gsh, t, n=72)                         # tracers of a unit circle carried by e^{Gt}
    ring.set_data(c[0], c[1])                                    # move the dots
    a, d = ch03.strain_ellipse_axes(Gsh, t, method="exact")      # exact semi-axes and directions (SVD)
    axis_line.set_data([-a[0]*d[0, 0], a[0]*d[0, 0]], [-a[0]*d[1, 0], a[0]*d[1, 0]])   # the long axis
    txt.set_text(f"t = {t:.2f} s   a = {a[0]:.2f}, b = {a[1]:.2f} mm   long axis {np.degrees(np.arctan2(d[1, 0], d[0, 0])):.1f}°")
    return ring, axis_line, txt                                  # the artists that changed

show_animation(animate(update, frames=nfr, fig=fig, interval=80), player="video")   # smooth MP4

**What does the code above do?**

1. `deform_circle` carries 72 tracers of a circle with the exact linear map $e^{\mathbf Gt}$.
2. `strain_ellipse_axes(method="exact")` gives the ellipse's semi-axes and long-axis direction at each time.

**What you see.** A ring of teal dots becoming an ever longer ellipse; a purple long axis that starts on the blue 45° line and then tips toward the horizontal.

**How to read it.** At first the ellipse grows along the blue axis, as D16 says; as time goes on the purple axis tips toward the flow
direction (the rotation part keeps turning it) — the principal-axis statement is about the *first instant*.

**What would change if…** …the flow were pure strain (the 'irrotational_strain' preset): the purple and blue axes would stay together for ever.

#### 🎮 Interactive: Can a straight flow make a fluid element spin?

**Why interactive:** the claim is "for every pair of lines" — a figure can show a few pairs, but sweeping the pair angle yourself shows the average never moves while the single lines do.
Two perpendicular threads and a small paddle wheel ride in a flow you choose. Drag the pair's angle: the two threads
turn at different rates but their average stays at ½ω₃. Drag a probe round the ring: its relative velocity splits into
a teal strain arrow and an orange rotation arrow, $du_i=S_{ij}dx_j+\tfrac12(\boldsymbol\omega\times d\mathbf x)_i$
*(Eq. 3.19)*. The circle turns into an ellipse on the principal axes.

**What to try:**
- Preset 'parallel shear γ = 1': sweep the pair angle from 0 to 180° and watch the average line stay flat at −0.5 rad/s.
- Preset 'rotate with the element' (Ω = ω/2): the paddle wheel stops — ω′ = 0.
- Preset 'pure strain': the orange arrows vanish; the ellipse axes never move.
- Derivation tab, D15: at the ε-swap step the orange arrow flips sign on screen.

In [ ]:
show_viz("ch03", "spin_and_principal_axes")   # full-width explainer; ⤢ Full screen for more room

**What would change if…** …the streamlines were circles instead of straight lines? Then "going round" and "spinning" come apart completely —
§3.5's two vortices.

---

## 3.5 Kinematics of Simple Plane Flows

**What is this section about?** Two families of plane flow where one coordinate does all the work: the parallel shear
flow (straight streamlines, spinning elements) and circular flows — solid-body rotation, the irrotational vortex and
the realistic vortices between them.

📝 **Note.** **Parallel shear flow** `N34` — $\mathbf u=(u_1(x_2),0)$ with the shear rate $\gamma(x_2)\equiv du_1/dx_2$ — the flow you used in C10. By
$\omega_3=\frac{\partial u_2}{\partial x_1}-\frac{\partial u_1}{\partial x_2}$ *(Eq. 3.16)*, $\omega_3=-\gamma$. The
vertical thread AB turns at −γ, the horizontal BC at 0, their average −γ/2 = ω₃/2 — and D14 (in C10) showed that
*every* perpendicular pair gives −γ/2. ⚠️ ω₃ = −γ is clockwise for γ > 0. Couette flow (Ch. 8) and boundary layers
(Ch. 9) are this flow.

$$ \gamma(x_2)\equiv du_1/dx_2,\qquad \omega_3=-\gamma  $$

📝 **Note.** **Two elements in the same shear (Fig. 3.14)** `N35` — For the square ABCD with sides along the axes, S has only off-diagonal entries — it shears without stretching its sides.
Turned by 45°, onto the principal axes (⚠️ Ch. 2's Ex. 2.4 had $S_{12}=\Gamma$; here $S_{12}=\gamma/2$), S is diagonal —
the square PQRS stretches along $\bar x_1$ and is compressed along $\bar x_2$ (eigenvalue −γ/2, i.e. compression at rate
γ/2) while its corners stay right angles. Both spin at −γ/2.

$$ S_{ij}=\begin{bmatrix}0&\gamma/2\\\gamma/2&0\end{bmatrix},\qquad \bar S_{ij}=\begin{bmatrix}\gamma/2&0\\0&-\gamma/2\end{bmatrix}  $$

In [ ]:
from scripts.ch03_drawings import shear_elements_frames      # corners of ABCD and PQRS carried by u = (γy, 0)
nfr = 16 if not FAST else 10                                  # frames to step through
frames = shear_elements_frames(1.0, np.linspace(0, 0.75, nfr))   # γ = 1 1/s, t = 0 … 0.75 s
fig, axs = plt.subplots(1, 2, figsize=(9, 3.9))              # ABCD left, PQRS right
arts = []                                                     # the artists each frame updates
for ax, key, col in zip(axs, ("ABCD", "PQRS"), (COLORS["accent"], COLORS["teal"])):
    ax.set_aspect("equal"); ax.set_xlim(-1.0, 1.0); ax.set_ylim(-0.8, 0.8); ax.set_xlabel("$x_1$ [m]")   # fixed limits
    ax.plot(*np.hstack([frames[0][key], frames[0][key][:, :1]]), color=COLORS["grid"], lw=1)   # the start shape
    (ln,) = ax.plot([], [], color=col, lw=2.4)                # the element now
    (pad,) = ax.plot([], [], color=COLORS["ink"], lw=1.5)     # a paddle at its centre, turned by −γt/2
    tx = ax.text(-0.95, 0.62, "", fontsize=8)                 # numbers of this frame
    arts.append((key, ln, pad, tx))
axs[0].set_ylabel("$x_2$ [m]")                                # shared y label
axs[0].set_title("ABCD: sides along the axes (shear)"); axs[1].set_title("PQRS: sides at 45° (stretch + squeeze)")

def update(i):                                                # draw frame i
    f = frames[i]                                             # corners, sides and angles at this time
    spin = -0.5*1.0*f["t"]                                    # element spin −γ/2 times t [rad]
    for key, ln, pad, tx in arts:                             # both elements
        P = f[key]; ln.set_data(*np.hstack([P, P[:, :1]]))    # closed outline through the four corners
        c, s = np.cos(spin), np.sin(spin)                     # paddle: two crossing blades of half-length 0.15 m
        pad.set_data([-0.15*c, 0.15*c, np.nan, 0.15*s, -0.15*s], [-0.15*s, 0.15*s, np.nan, -0.15*c, 0.15*c])
        sides = ", ".join(f"{v:.3f}" for v in f[key + "_sides"]); ang = f[key + "_angles_deg"]   # four sides [m], angles [°]
        tx.set_text(f"t = {f['t']:.2f} s\nsides {sides} m\ncorner {ang[1]:.1f}°")   # corner B (or Q)
    return [a for _, ln, pad, tx in arts for a in (ln, pad, tx)]   # the artists that changed

show_animation(animate(update, frames=nfr, fig=fig, interval=500), player="frames")   # step frame by frame

**What does the code above do?**

1. `shear_elements_frames` carries the corners of both squares with the exact linear map of the shear flow and reports
   side lengths and corner angles.
2. Each frame redraws the two elements, a paddle turned by −γt/2, and the numbers.

**What you see.** Left, a square whose top slides right (a parallelogram); right, a diamond that stretches along 45° and shrinks along −45°. Both paddles turn clockwise together.

**How to read it.** ABCD's vertical sides lengthen only at second order while its corner angle falls at γ per second; PQRS's corner
stays 90.0° to first order while one pair of sides grows and the other shrinks. Step frame by frame: the numbers are the
linear (C07), shear (C08) and spin (C10) rates made visible.

**What would change if…** …you ran longer (t ≫ 1 s): PQRS's angles would drift from 90° too — the finite-time effect of C12's animation.

### 🧩 Going round is not spinning: vorticity in polar coordinates (3.23) `C13`

*The question:* Two tanks of water both go round in circles — one spun up like a merry-go-round, one draining through a plughole. Put a
small paddle wheel in each. Which one turns?

*In one line:* $\omega_z=\frac1r\frac{\partial}{\partial r}(ru_\theta)-\frac1r\frac{\partial u_r}{\partial\theta}$ *(3.23)*

#### The problem in plain words

Hurricanes, bathtub drains, the flow round a stirred cup: circular streamlines everywhere. The vorticity tells us
whether each little parcel spins as it goes round. We need it in polar coordinates, where these flows are simple.

#### The idea

```
solid body  u_θ = ω₀ r :   Γ grows like r²  → ω_z = 2ω₀ everywhere → the wheel spins as it orbits      (Fig. 3.15)
line vortex u_θ = B/r  :   Γ = 2πB for every circle → ω_z = 0 except on the axis → the wheel keeps its heading (Fig. 3.16)
vorticity = circulation per unit area of a small polar sector  →  ω_z = (1/r)∂(r u_θ)/∂r − (1/r)∂u_r/∂θ   (3.23)
```

In symbols: $\omega_z=\frac1r\frac{\partial}{\partial r}(ru_\theta)-\frac1r\frac{\partial u_r}{\partial\theta}$ *(3.23)*.

> 📎 **Primer — polar coordinates as a moving basis.** `P105` · In the plane, $\mathbf e_r=(\cos\theta,\sin\theta)$ and $\mathbf e_\theta=(-\sin\theta,\cos\theta)$ change direction
with θ. A small polar sector has area $r\,dr\,d\theta$ (an arc of length $r\,d\theta$ times a width dr); along a circle
of radius r the line element is $d\mathbf s=r\,d\theta\,\mathbf e_\theta$, along a ray $d\mathbf s=dr\,\mathbf e_r$. Arcs
at r and r + dr have *different* lengths — the source of the extra 1/r terms in polar formulas.

In [ ]:
r, dr, dth = 2.0, 0.01, 0.02                          # a sector at r = 2 m, 1 cm deep, 0.02 rad wide
inner, outer = r*dth, (r + dr)*dth                    # arc lengths [m]
print(round(outer - inner, 6), dr*dth)                # 0.0002 0.0002: the arcs differ by dr·dθ
print(r*dr*dth)                                       # 0.0004 m²: the sector's area

#### 🧮 Derivation — Vorticity in polar coordinates, Eq. (3.23), from the circulation round a small sector `D17` (Eq. 3.23)

**What we want to show.** Derive the vorticity of a plane flow in polar coordinates, which the book takes from Appendix B (and leaves to Exercise 3.20), and apply it to solid-body rotation.

**Assumptions.** u smooth inside the sector (step 9: the sector must not contain a singular point such as the axis of a line vortex).

**The plan.**

1. The sector's area.
2. The two arcs (their lengths differ).
3. The two radial legs.
4. Add, divide, take the limit.
5. Apply to (3.22), $u_r=0,\ u_\theta=\omega_0r$.

**Tools we use** (each explained before this point): Circulation and Stokes, (3.18), $\Gamma=\oint_C\mathbf u\cdot d\mathbf s=\int_A\boldsymbol\omega\cdot\mathbf n\,dA$ (R07) · polar coordinates as a moving basis (primer in C13) · line integral round a loop (ch02 P86) · first-order Taylor expansion (ch01 P26).

**We start from**

$$ \begin{array}{l}\text{Vorticity is circulation per unit area (R07, from (3.18)} \\ \Gamma=\oint_C\mathbf u\cdot d\mathbf s=\int_A\boldsymbol\omega\cdot\mathbf n\,dA\ \text{):}\ \omega_z=\lim_{A\to0}\frac{1}{A}\oint_C\mathbf u\cdot d\mathbf s\ \text{, taken round the small} \\ \text{polar sector r…r + dr, θ…θ + dθ, counterclockwise}\end{array} $$

*In words:* add u·ds round a tiny sector, divide by its area.

---

**Step 1 of 10 — The sector's area.**

$$ A=r\,dr\,d\theta $$

- *Why we can do this:* Polar area element (primer): an arc of length r dθ times a width dr, to leading order.
- *In words:* The small area we divide by.

---

**Step 2 of 10 — The outer arc, counterclockwise.**

$$ \int_{\rm outer}\mathbf u\cdot d\mathbf s=\big[u_\theta\,r\big]_{r+dr}\,d\theta $$

- *Why we can do this:* On the arc at radius r + dr, $d\mathbf s=(r+dr)\,d\theta\,\mathbf e_\theta$ (primer), so u·ds = u_θ(r + dr)dθ, taken at the arc's midpoint.
- *In words:* The outer arc collects the tangential velocity times its length.

---

**Step 3 of 10 — The inner arc, run backwards.**

$$ \int_{\rm inner}\mathbf u\cdot d\mathbf s=-\big[u_\theta\,r\big]_{r}\,d\theta $$

- *Why we can do this:* Going counterclockwise round the sector the inner arc is traversed clockwise, $d\mathbf s=-r\,d\theta\,\mathbf e_\theta$.
- *In words:* The inner arc subtracts.

---

**Step 4 of 10 — Combine the arcs by Taylor.**

$$ \big[u_\theta r\big]_{r+dr}d\theta-\big[u_\theta r\big]_rd\theta=\dfrac{\partial(ru_\theta)}{\partial r}\,dr\,d\theta $$

- *Why we can do this:* First-order Taylor in r of the product $ru_\theta$. The book skips this: the arcs have different lengths, which is why r stays inside the derivative.
- *In words:* The arcs nearly cancel; what is left is how r u_θ changes outward.

---

**Step 5 of 10 — The radial leg at θ, outward.**

$$ \int_{\theta}\mathbf u\cdot d\mathbf s=\big[u_r\big]_{\theta}\,dr $$

- *Why we can do this:* Along a ray $d\mathbf s=dr\,\mathbf e_r$, so only the radial component contributes.
- *In words:* The outward leg collects u_r.

---

**Step 6 of 10 — The radial leg at θ + dθ, inward.**

$$ \int_{\theta+d\theta}\mathbf u\cdot d\mathbf s=-\big[u_r\big]_{\theta+d\theta}\,dr $$

- *Why we can do this:* Counterclockwise round the sector this leg runs inward, $d\mathbf s=-dr\,\mathbf e_r$.
- *In words:* The inward leg subtracts.

---

**Step 7 of 10 — Combine the radial legs by Taylor.**

$$ \big[u_r\big]_\theta dr-\big[u_r\big]_{\theta+d\theta}dr=-\dfrac{\partial u_r}{\partial\theta}\,d\theta\,dr $$

- *Why we can do this:* First-order Taylor expansion of u_r in θ (P26), exactly as the two arcs were combined in r in step 4.
- *In words:* A radial velocity that changes with angle adds circulation.

---

**Step 8 of 10 — Add all four legs.**

$$ d\Gamma=\Big[\dfrac{\partial(ru_\theta)}{\partial r}-\dfrac{\partial u_r}{\partial\theta}\Big]dr\,d\theta $$

- *Why we can do this:* The circulation is the sum over the closed loop: steps 4 and 7.
- *In words:* The circulation of the small sector.

---

**Step 9 of 10 — Divide by the area, shrink.**

$$ \omega_z=\dfrac1r\dfrac{\partial}{\partial r}(ru_\theta)-\dfrac1r\dfrac{\partial u_r}{\partial\theta} $$

- *Why we can do this:* Vorticity = circulation per unit area (R07): divide step 8 by $r\,dr\,d\theta$ and let dr, dθ → 0; the neglected Taylor terms vanish. This is (3.23), $\omega_z=\frac1r\frac{\partial}{\partial r}(ru_\theta)-\frac1r\frac{\partial u_r}{\partial\theta}$.
- *In words:* The polar formula.

---

**Step 10 of 10 — Apply it to solid-body rotation.**

$$ \omega_z=\dfrac1r\dfrac{d}{dr}(r\cdot\omega_0r)=2\omega_0 $$

- *Why we can do this:* (3.22), $u_r=0$ and $u_\theta=\omega_0r$: the second term vanishes and $d(\omega_0r^2)/dr=2\omega_0r$.
- *In words:* Every element of a solid-body rotation spins at ω₀ (half of 2ω₀) — as fast as it revolves.

---

**Result**

$$ \omega_z=\frac1r\frac{\partial}{\partial r}(ru_\theta)-\frac1r\frac{\partial u_r}{\partial\theta}\ \text{(3.23),}\ =2\omega_0\ \text{for (3.22)} $$

*In words:* in polar coordinates the vorticity is the outward change of r u_θ minus the angular change of u_r, per unit r.

**What it means.** "Going round" (u_θ ≠ 0) and "spinning" (ω_z ≠ 0) are different: what matters is how r u_θ changes with r. If r u_θ is constant (the line vortex) the fluid circles without spinning. Every vortex formula in Ch. 5, 6 and 13 uses (3.23), $\omega_z=\frac1r\frac{\partial}{\partial r}(ru_\theta)-\frac1r\frac{\partial u_r}{\partial\theta}$.

**Check it.** Units 1/s ✓. Line vortex (3.25) $u_\theta=B/r$: $\frac1r\frac{d}{dr}(B)=0$ ✓. Cartesian curl of (−ω₀y, ω₀x) = 2ω₀ ✓. Numbers: the notebook's four-leg sum for a Rankine core (Γ = 2π, σ = 2 m) gives 0.500 s⁻¹ = Γ/πσ² ✓. The sympy cell below compares $\omega_z=\frac1r\frac{\partial}{\partial r}(ru_\theta)-\frac1r\frac{\partial u_r}{\partial\theta}$ (3.23) with the Cartesian curl for the non-axisymmetric pair $u_r=r^2\sin\theta$, $u_\theta=r\cos\theta$.

In [ ]:
r, th = sp.symbols('r theta', positive=True)          # polar coordinates
x, y = sp.symbols('x y', real=True)                   # Cartesian coordinates
ur, uth = r**2*sp.sin(th), r*sp.cos(th)               # a non-axisymmetric test field (u_r, u_θ)
polar = ch03.polar_vorticity_z_sym(ur, uth, r, th)    # (3.23): (1/r)∂(r u_θ)/∂r − (1/r)∂u_r/∂θ
ux = ur*sp.cos(th) - uth*sp.sin(th)                   # Cartesian components u = u_r e_r + u_θ e_θ
uy = ur*sp.sin(th) + uth*sp.cos(th)
to_xy = {r: sp.sqrt(x**2 + y**2), th: sp.atan2(y, x)} # write them as functions of (x, y)
curl = sp.diff(uy.subs(to_xy), x) - sp.diff(ux.subs(to_xy), y)   # (3.16): ω_z = ∂v/∂x − ∂u/∂y
back = {x: r*sp.cos(th), y: r*sp.sin(th)}             # return to polar coordinates to compare
print(sp.simplify(sp.expand_trig(curl.subs(back)) - polar))   # 0: the sector derivation agrees with the curl
print(sp.simplify(polar))                             # the vorticity of this test field

> ⚠️ **Common confusion (traps in `D17`):** Writing $\partial u_\theta/\partial r$ instead of $\frac1r\partial(ru_\theta)/\partial r$ (forgetting the arcs differ). Orientation of the legs. Applying it to a sector that contains the axis of a line vortex (u is singular there).

📝 **Note.** **Solid-body rotation** `N36` — a tank spun steadily until the fluid turns with it. By
$\omega_z=\frac1r\frac{\partial}{\partial r}(ru_\theta)-\frac1r\frac{\partial u_r}{\partial\theta}$ *(3.23)*,
$\omega_z=\frac1r\frac{d}{dr}(\omega_0r^2)=2\omega_0$ everywhere: each element spins about its own centre at the rate
it revolves round the axis; S = 0, nothing deforms (Fig. 3.15).

$$ u_r=0\quad\text{and}\quad u_\theta=\omega_0r \qquad \text{(3.22)} $$

📝 **Note.** **Its circulation round a centred circle** `N37` — = vorticity 2ω₀ × area πr². It holds for *any* circuit, centred or not: with uniform ω, Stokes'
$\Gamma=\oint_C\mathbf u\cdot d\mathbf s=\int_A\boldsymbol\omega\cdot\mathbf n\,dA$ *(3.18)* gives Γ = ω × enclosed area.
Number: ω₀ = 1 s⁻¹, r = 1 m → Γ = 2π = 6.283 m²/s; an off-centre circle of radius 1 m centred at (2, 0) m also gives
6.283 m²/s (checked below).

$$ \Gamma=\oint_C\mathbf u\cdot d\mathbf s=\int_0^{2\pi}u_\theta r\,d\theta=2\pi ru_\theta=2\pi r^2\omega_0 \qquad \text{(3.24)} $$

📝 **Note.** **The irrotational (line) vortex** `N38` — the ideal limit of a drain or a tornado far from its core. (⚠️ Fig. 3.16 writes C for B.) By (3.23),
$\omega_z=\frac1r\frac{d}{dr}(r\cdot B/r)=\frac1r\frac{dB}{dr}=0$ for every r > 0.

$$ u_r=0\quad\text{and}\quad u_\theta=B/r \qquad \text{(3.25)} $$

📝 **Note.** **Yet the circulation round any centred circle is the same nonzero number** `N39` — independent of r ($u_\theta r=B$ is constant).

$$ \Gamma=\int_0^{2\pi}u_\theta r\,d\theta=2\pi ru_\theta=2\pi B \qquad \text{(3.26)} $$

📝 **Note.** **All the vorticity sits on the axis** `N40` — Taking vorticity as circulation per unit area in a shrinking disc gives a value that is infinite at r = 0 with a finite
area integral 2πB: a *delta function* (Ch. 2 gloss: an infinitely concentrated amount whose total is finite). The point
vortex of Ch. 6 is exactly this.

$$ [\omega_z]_{r\to0}=\lim_{r\to0}\frac1A\int_A\omega_z\,dA=\lim_{r\to0}\frac{1}{\pi r^2}\oint_C\mathbf u\cdot d\mathbf s=\lim_{r\to0}\frac{2B}{r^2} \qquad \text{(3.27)} $$

📝 **Note.** **A loop that does not enclose the axis has zero circulation** `N41` — the sector ABCD of Fig. 3.16: the radial legs BC and DA contribute nothing ($\mathbf u\perp d\mathbf s$), and the arcs
cancel because $u_\theta r=B$. Elements in this flow deform but do not spin.

$$ \Gamma_{ABCD}=-[u_\theta r]_r\Delta\theta+[u_\theta r]_{r+\Delta r}\Delta\theta=0  $$

#### ✏️ Tiny example: the two tanks with easy numbers

1. Solid body ω₀ = 1 s⁻¹: $u_\theta=r$; $\omega_z=\frac1r\frac{d(r\cdot r)}{dr}=\frac{2r}{r}=2$ s⁻¹; spin ½ω_z = 1 rad/s =
   orbit rate $u_\theta/r$ = 1 rad/s.
2. Line vortex B = 1 m²/s: $u_\theta=1/r$; $\omega_z=\frac1r\frac{d(r\cdot1/r)}{dr}=\frac1r\frac{d(1)}{dr}=0$; spin 0, orbit
   rate $1/r^2$ rad/s.
3. Circulations: circle r = 1 m: solid body $2\pi(1)^2(1)=6.283$ m²/s; line vortex $2\pi(1)=6.283$ m²/s — equal at
   r = 1, but at r = 2 m: 25.13 vs 6.283 m²/s.
4. Mean vorticity in the disc r = 0.1 m for the line vortex: $2B/r^2=200$ s⁻¹; at r = 0.01 m: 20 000 s⁻¹.

In [ ]:
r, th, w0, B = sp.symbols('r theta omega_0 B', positive=True)          # symbols for (3.23)
print(ch03.polar_vorticity_z_sym(0, w0*r, r, th), ch03.polar_vorticity_z_sym(0, B/r, r, th))   # 2ω0 and 0
print(ch03.polar_vorticity_z(lambda r, t: 0.0, lambda r, t: 1.0*r, 1.3, 0.4),        # stencils in r and θ: 2
      ch03.polar_vorticity_z(lambda r, t: 0.0, lambda r, t: 1.0/r, 1.3, 0.4))        # … and ≈ 0
kw = dict(Gamma=2*np.pi, sigma=1.0)                  # "solid": ω0 = Γ/(2πσ²) = 1 1/s; "line": B = Γ/2π = 1 m²/s
print(ch03.circulation_circle("solid", 1.0, **kw), ch03.circulation_circle("solid", 1.0, center=(2.0, 0.0), **kw))
print([round(ch03.circulation_circle("line", r_, **kw), 4) for r_ in (0.5, 1.0, 2.0)])   # 2πB at every radius
print(ch03.annular_sector_circulation("line", 1.0, 0.1, 0.2, **kw))   # a sector away from the axis: 0
print([ch03.mean_vorticity_in_disc("line", r_, **kw) for r_ in (0.1, 0.01)])   # 2B/r²: grows without bound

**What does the code above do?**

1. sympy applies $\omega_z=\frac1r\frac{\partial}{\partial r}(ru_\theta)-\frac1r\frac{\partial u_r}{\partial\theta}$
   *(3.23)* to both profiles: $2\omega_0$ and 0.
2. Stencils in r and θ agree (2.0 and ≈ 10⁻¹²).
3. Γ = ω × area even off-centre (N37): 6.283 m²/s for both circles.
4. 2πB at every radius (N39).
5. Zero for a sector away from the axis (N41).
6. The delta-function core: the mean vorticity in a disc grows like $1/r^2$ (N40).

In [ ]:
r0, dr, dth = 1.3, 0.01, 0.02                         # a small polar sector inside a Rankine core
kwR = dict(Gamma=2*np.pi, sigma=2.0)                  # Γ = 2π m²/s, core radius σ = 2 m
uth = lambda r: ch03.rankine_vortex(r, **kwR)[0]      # u_θ(r) of the Rankine vortex [m/s]
outer = uth(r0 + dr)*(r0 + dr)*dth                    # outer arc, counterclockwise: u_θ (r + dr) dθ
inner = -uth(r0)*r0*dth                               # inner arc, run backwards
legs = 0.0                                            # radial legs: u_r = 0, so u·ds = 0 there
area = (r0 + dr/2)*dr*dth                             # exact sector area ((r+dr)² − r²)dθ/2 [m²]
mine = (outer + inner + legs)/area                    # circulation / area = ω_z (D17 with numbers)
lib = ch03.polar_vorticity_z(None, "rankine", r0, 0.0, **kwR)     # (3.23) by stencils
cart = ch03.vorticity(ch03.vortex_velocity_field("rankine", **kwR), np.array([r0*np.cos(0.4), r0*np.sin(0.4)]), 0.0)[2]
assert np.allclose(mine, lib, rtol=1e-3) and np.allclose(lib, cart, rtol=1e-6)   # all = Γ/πσ² = 0.5 1/s
print(f"four legs: {mine:.5f}   polar formula: {lib:.5f}   Cartesian curl: {cart:.5f}   (Γ/πσ² = {2*np.pi/(np.pi*4):.5f})")

D17 with numbers: four line integrals and one division.

In [ ]:
rr = np.geomspace(1e-3, 10, 60)                        # disc radii from 1 mm to 10 m
kw = dict(Gamma=2*np.pi, sigma=1.0)
fig, ax = plt.subplots(figsize=(7, 4))                  # log–log axes (Ch. 1 P13)
ax.loglog(rr, ch03.mean_vorticity_in_disc("line", rr, **kw), color=COLORS["orange"], lw=2.3, label="line vortex: 2B/r² (slope −2)")
ax.loglog(rr, ch03.mean_vorticity_in_disc("solid", rr, **kw), color=COLORS["teal"], lw=2.3, label="solid body: 2ω₀")   # flat
ax.loglog(rr, ch03.mean_vorticity_in_disc("rankine", rr, **kw), "--", color=COLORS["accent"], lw=2.3, label="Rankine, σ = 1 m")
ax.axvline(1.0, color=COLORS["muted"], ls=":", lw=1)    # the core radius σ
ax.set_xlabel("disc radius $r$ [m]"); ax.set_ylabel(r"mean vorticity $\Gamma(r)/\pi r^2$ [1/s]"); ax.legend(fontsize=8)
ax.set_title("An irrotational vortex keeps all its vorticity on the axis")
savefig(fig, "ch03", "mean_vorticity_disc"); plt.show() # save to outputs/ch03 and draw

**What you see.** A straight orange line of slope −2, a flat teal line, and a dashed purple curve that switches from one to the other at r = σ = 1 m.

**How to read it.** The mean vorticity in a disc is its circulation divided by its area. For the line vortex the circulation is the
same for every disc, so the smaller the disc, the larger the average — all of it sits on the axis. A real vortex has a
finite core, inside which it behaves like a solid body.

**What would change if…** …σ shrank toward zero: the purple curve's flat part would climb (Γ/πσ²) and the Rankine vortex would approach the line vortex.

In [ ]:
nfr = 60 if not FAST else 30                           # frames
times = np.linspace(0, 2*np.pi, nfr)                   # one revolution of the solid body [s]
radii = np.array([0.5, 1.0, 1.5, 2.0])                 # four paddle wheels [m]
cases = {"solid body, ω₀ = 1 1/s": (lambda r: 1.0*np.ones_like(r), 2.0),      # (orbit rate u_θ/r [rad/s], ω_z [1/s])
         "line vortex, B = 1 m²/s": (lambda r: 1.0/r**2, 0.0)}
fig, axs = plt.subplots(1, 2, figsize=(9.6, 4.6))       # solid body left, line vortex right
arts = []                                              # the artists each frame updates
for ax, (name, (Om, wz)) in zip(axs, cases.items()):
    ax.set_aspect("equal"); ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5); ax.set_title(name)   # fixed limits
    for R_ in radii: ax.add_patch(plt.Circle((0, 0), R_, fill=False, color=COLORS["grid"]))    # the streamlines (circles)
    (wheels,) = ax.plot([], [], color=COLORS["ink"], lw=1.6)   # all four paddle wheels in one line
    (el,) = ax.plot([], [], color=COLORS["teal"], lw=2)        # the polar element's outline
    arts.append((Om, wz, wheels, el))
axs[0].set_xlabel("$x$ [m]"); axs[1].set_xlabel("$x$ [m]"); axs[0].set_ylabel("$y$ [m]")
sq = np.linspace(0, 1, 15)                             # points along each edge of the element

def update(i):                                         # draw frame i
    t = times[i]                                       # time of this frame [s]
    out = []                                           # artists that changed
    for Om, wz, wheels, el in arts:                    # both vortices
        ang = Om(radii)*t                              # each wheel's position angle: it orbits at u_θ/r
        spin = 0.5*wz*t                                # each wheel turns at ½ω_z (D12, D17)
        xs, ys = [], []                                # blade end points, NaN-separated
        for R_, a in zip(radii, ang):                  # one wheel per radius
            cx, cy = R_*np.cos(a), R_*np.sin(a)        # its hub
            for b in (spin, spin + np.pi/2):           # two crossing blades
                xs += [cx - 0.22*np.cos(b), cx + 0.22*np.cos(b), np.nan]; ys += [cy - 0.22*np.sin(b), cy + 0.22*np.sin(b), np.nan]
        wheels.set_data(xs, ys)                        # move the wheels
        rr_ = np.r_[1 + 0.4*sq, np.full(15, 1.4), 1.4 - 0.4*sq, np.full(15, 1.0)]   # element outline: r from 1 to 1.4 m
        tt_ = np.r_[np.zeros(15), 0.35*sq, np.full(15, 0.35), 0.35 - 0.35*sq]      # … and θ from 0 to 0.35 rad
        th_now = tt_ + Om(rr_)*t                       # every boundary point orbits at its own rate
        el.set_data(rr_*np.cos(th_now), rr_*np.sin(th_now))   # back to x, y
        out += [wheels, el]
    return out

show_animation(animate(update, frames=nfr, fig=fig, interval=70), player="video")   # smooth MP4

**What does the code above do?**

1. Every point moves on its circle at the orbit rate $u_\theta/r$ (1 rad/s for the solid body, $1/r^2$ for the line vortex).
2. Each paddle wheel is turned by $\tfrac12\omega_z t$ — the element spin of D12, with $\omega_z$ from (3.23).
3. The teal polar element is drawn by carrying 60 points of its outline.

**What you see.** Left: four wheels going round together, each turning once per revolution, and a teal element that keeps its shape. Right: wheels orbiting at very different speeds but never turning, and an element sheared into a thin wedge.

**How to read it.** On the left, going round and spinning go together (ω_z = 2ω₀). On the right the wheels keep pointing the same way
while they circle — ω_z = 0 — although the fluid clearly deforms: fast inside, slow outside.

**What would change if…** …a wheel were added at r = 0.2 m on the right: it would race round (25 rad/s) — and still not turn.

**What would change if…** …a real drain vortex were measured? Near the axis the speed does not blow up like 1/r; the core turns like a solid
body and the outside like a line vortex — the Rankine and Gaussian models of C14.

### 🧩 Real vortices have a core: the Rankine and Gaussian models (3.28)–(3.29) `C14`

*The question:* A tornado's wind is calm at its centre, fastest a little way out and weaker far away. Which simple formula reproduces
that profile, and where exactly is the fastest wind?

*In one line:* $u_\theta=\frac{\Gamma r}{2\pi\sigma^2}\ (r\le\sigma),\ \frac{\Gamma}{2\pi r}\ (r>\sigma)$ *(3.28)* · $u_\theta=\frac{\Gamma}{2\pi r}\big(1-e^{-r^2/\sigma^2}\big)$ *(3.29)*

#### The problem in plain words

Tornado chasers, hurricane forecasters and aircraft-wake engineers all quote two numbers: the peak wind and the radius
where it occurs. Pure solid-body rotation grows for ever; the line vortex blows up at the axis. Joining them gives a
vortex with a spinning core and an irrotational skirt — the model behind every cyclone schematic.

#### The idea

```
              core (r < σ): solid body, ω_z = Γ/πσ²       outside: irrotational, u_θ = Γ/2πr
Rankine:   sharp join at r = σ (speed continuous, vorticity jumps)     → peak at r = σ exactly
Gaussian:  smooth join, ω_z = (Γ/πσ²) e^{−r²/σ²}                        → peak at r ≈ 1.1209 σ
```

📝 **Note.** **Real vortices** `N42` — bathtub drains, wing-tip vortices (Ch. 14), tornadoes, tropical cyclones (Ch. 13) — combine a nearly solid-body core
with a nearly irrotational outer flow and have bounded speeds.

#### The maths — the Rankine vortex

Uniform vorticity inside a core of radius σ, none outside; Γ is the total circulation:

$$\omega_z(r)=\begin{Bmatrix}\Gamma/\pi\sigma^2=\text{const.}&r\le\sigma\\0&r>\sigma\end{Bmatrix}\quad\text{and}\quad u_\theta(r)=\begin{Bmatrix}(\Gamma/2\pi\sigma^2)r&r\le\sigma\\\Gamma/2\pi r&r>\sigma\end{Bmatrix}\qquad(3.28)$$

A piecewise formula is evaluated with `np.where(r <= sigma, inside, outside)` (Ch. 1 P46).

#### 🧮 Derivation — The Rankine vortex is self-consistent: core vorticity, continuity at σ, peak at σ `D18` (Eq. 3.28)

**What we want to show.** Check that the two halves of (3.28), $u_\theta=\frac{\Gamma r}{2\pi\sigma^2}\ (r\le\sigma),\ \frac{\Gamma}{2\pi r}\ (r>\sigma)$ fit together — uniform vorticity inside, none outside, speed continuous, total circulation Γ — and find the peak speed; the book states all of this without working.

**Assumptions.** Axisymmetric (u_r = 0, no θ-dependence).

**The plan.**

1. Vorticity in each region by (3.23), $\omega_z=\frac1r\frac{\partial}{\partial r}(ru_\theta)-\frac1r\frac{\partial u_r}{\partial\theta}$.
2. Continuity of the speed at σ.
3. Total circulation.
4. Where the speed peaks.

**Tools we use** (each explained before this point): (3.23), $\omega_z=\frac1r\frac{\partial}{\partial r}(ru_\theta)-\frac1r\frac{\partial u_r}{\partial\theta}$ (D17) · derivative of a product · the circulation of a centred circle, (3.24), $\Gamma=2\pi ru_\theta=2\pi r^2\omega_0$ (solid body) and (3.26), $\Gamma=2\pi ru_\theta=2\pi B$ (line vortex) (N37, N39) · continuity at a join and a kink as a maximum (gloss).

**We start from**

$$ \text{(3.28):}\ u_\theta(r)=\frac{\Gamma}{2\pi\sigma^2}r\ \text{for}\ r\le\sigma\ \text{,}\ u_\theta(r)=\frac{\Gamma}{2\pi r}\ \text{for}\ r>\sigma\ \text{,}\ u_r=0 $$

*In words:* solid-body rotation inside a core of radius σ, a line vortex outside.

---

**Step 1 of 5 — Vorticity inside by (3.23).**

$$ \omega_z=\dfrac1r\dfrac{d}{dr}\Big(\dfrac{\Gamma r^2}{2\pi\sigma^2}\Big)=\dfrac{\Gamma}{\pi\sigma^2} $$

- *Why we can do this:* u_r = 0 leaves the first term of (3.23), $\omega_z=\frac1r\frac{\partial}{\partial r}(ru_\theta)-\frac1r\frac{\partial u_r}{\partial\theta}$; $d(r^2)/dr=2r$ and the r cancels.
- *In words:* The core has uniform vorticity — a solid-body rotation with $\omega_0=\Gamma/2\pi\sigma^2$.

---

**Step 2 of 5 — Vorticity outside by (3.23).**

$$ \omega_z=\dfrac1r\dfrac{d}{dr}\Big(\dfrac{\Gamma}{2\pi}\Big)=0 $$

- *Why we can do this:* Outside, r u_θ = Γ/2π is constant, so its derivative is zero.
- *In words:* No vorticity outside the core.

---

**Step 3 of 5 — Compare the two speeds at σ.**

$$ \dfrac{\Gamma}{2\pi\sigma^2}\sigma=\dfrac{\Gamma}{2\pi\sigma} $$

- *Why we can do this:* Evaluate each branch at r = σ (gloss: a join is continuous when the two sides agree there). The speed is continuous; the vorticity jumps from Γ/πσ² to 0.
- *In words:* No jump in speed, a jump in spin.

---

**Step 4 of 5 — Total circulation.**

$$ \Gamma(r\ge\sigma)=2\pi r\,u_\theta=\Gamma=\dfrac{\Gamma}{\pi\sigma^2}\cdot\pi\sigma^2 $$

- *Why we can do this:* The centred-circle circulation, (3.24), $\Gamma=2\pi ru_\theta=2\pi r^2\omega_0$ (solid body) and (3.26), $\Gamma=2\pi ru_\theta=2\pi B$ (line vortex): outside the core $ru_\theta=\Gamma/2\pi$, so the circle gives Γ; by Stokes it equals core vorticity × core area.
- *In words:* All the circulation lives in the core.

---

**Step 5 of 5 — Locate the peak speed.**

$$ u_{\max}=\dfrac{\Gamma}{2\pi\sigma}\ \text{at}\ r=\sigma $$

- *Why we can do this:* $du_\theta/dr>0$ inside (linear rise) and $<0$ outside (1/r fall): the derivative changes sign at σ — a maximum at a kink, not at a zero of the derivative (gloss).
- *In words:* The fastest wind is at the edge of the core.

---

**Result**

$$ \omega_z=\Gamma/\pi\sigma^2\ \text{(}\ r\le\sigma\ \text{), 0 (}\ r>\sigma\ \text{); u\_θ continuous at σ with}\ u_{\max}=\Gamma/2\pi\sigma $$

*In words:* a spinning core wrapped in an irrotational skirt, fastest at the core's edge.

**What it means.** The simplest model of a real vortex: bounded speed, finite core. The jump in vorticity is unphysical for a viscous fluid (viscosity smooths it — the Gaussian of D19).

**Check it.** Units: Γ/πσ² [1/s], Γ/2πσ [m/s] ✓. σ → 0 at fixed Γ: the core shrinks to the line vortex ✓. Numbers (Γ = 2π m²/s, σ = 1 m): inside u = r, ω = 2 s⁻¹; peak 1 m/s at 1 m ✓.

> ⚠️ **Common confusion (traps in `D18`):** Setting du_θ/dr = 0 to find the maximum (there is no such point — the maximum is a kink). Thinking the vorticity is continuous because the speed is.

> 📎 **Primer — substitution in an integral.** `P106` · Replace a messy variable by a simpler one and convert dx too: with $s=r^2/\sigma^2$, $ds=2r\,dr/\sigma^2$, so
$2\pi r\,dr=\pi\sigma^2\,ds$ and the limits r = 0 … R become s = 0 … R²/σ². The value of the integral does not change.

In [ ]:
from scipy.integrate import quad                     # adaptive quadrature (P87)
sig, R = 1.0, 1.5                                    # core radius and outer radius [m]
lhs = quad(lambda r: np.exp(-r**2/sig**2)*2*np.pi*r, 0, R)[0]      # in r
rhs = quad(lambda s: np.exp(-s)*np.pi*sig**2, 0, R**2/sig**2)[0]   # in s = r²/σ²
print(round(lhs, 4), round(rhs, 4))                  # 2.8105 2.8105 (= π(1 − e^−2.25))

#### 🧮 Derivation — The Gaussian vortex's speed from its vorticity, Eq. (3.29) `D19` (Eq. 3.29)

**What we want to show.** Show that the speed profile of the Gaussian vortex (3.29), $u_\theta=\frac{\Gamma}{2\pi r}\big(1-e^{-r^2/\sigma^2}\big)$ follows from its vorticity by Stokes' theorem — the book states both halves without connecting them — and find its two limits.

**Assumptions.** Axisymmetric flow, u_r = 0 (step 4: u_θ is the same all round the circle).

**The plan.**

1. Circulation inside radius r as an area integral of vorticity.
2. Substitute to do the integral.
3. Circulation as u_θ times the circle's length.
4. Solve for u_θ.
5. Limits near and far.

**Tools we use** (each explained before this point): Stokes/circulation (3.18), $\Gamma=\oint_C\mathbf u\cdot d\mathbf s=\int_A\boldsymbol\omega\cdot\mathbf n\,dA$ (R07) · substitution in an integral (primer in C14) · ∫e^{−s}ds (ch01 P36) · the circle integral $\Gamma=2\pi ru_\theta$ of (3.24), $\Gamma=2\pi ru_\theta=2\pi r^2\omega_0$ (N37) · first-order Taylor of e^{−x} (ch01 P26).

**We start from**

$$ \omega_z(r)=\frac{\Gamma}{\pi\sigma^2}\exp(-r^2/\sigma^2) $$

*In words:* the vorticity is largest on the axis and falls off smoothly over a core radius σ.

---

**Step 1 of 7 — Circulation inside radius r.**

$$ \Gamma(r)=\displaystyle\int_0^r\omega_z(r')\,2\pi r'\,dr' $$

- *Why we can do this:* (3.18), $\Gamma=\oint_C\mathbf u\cdot d\mathbf s=\int_A\boldsymbol\omega\cdot\mathbf n\,dA$: circulation round the circle = vorticity flux through the disc; for axisymmetric ω split the disc into rings of area 2πr′dr′.
- *In words:* Add up the vorticity inside the circle.

---

**Step 2 of 7 — Substitute s = r′²/σ².**

$$ \Gamma(r)=\dfrac{\Gamma}{\pi\sigma^2}\,\pi\sigma^2\displaystyle\int_0^{r^2/\sigma^2}e^{-s}\,ds $$

- *Why we can do this:* With $ds=2r'dr'/\sigma^2$, $2\pi r'dr'=\pi\sigma^2ds$ and the limits become 0 … r²/σ² (substitution primer).
- *In words:* A plain exponential integral.

---

**Step 3 of 7 — Integrate the exponential.**

$$ \Gamma(r)=\Gamma\big(1-e^{-r^2/\sigma^2}\big) $$

- *Why we can do this:* $\int_0^Xe^{-s}ds=1-e^{-X}$ (P36); the π σ² cancel.
- *In words:* The fraction of the total circulation inside r.

---

**Step 4 of 7 — Circulation of the circle directly.**

$$ \Gamma(r)=\oint\mathbf u\cdot d\mathbf s=2\pi r\,u_\theta(r) $$

- *Why we can do this:* On a centred circle $d\mathbf s=r\,d\theta\,\mathbf e_\theta$ and u_θ is constant round it (axisymmetric) — as in (3.24), $\Gamma=\int_0^{2\pi}u_\theta r\,d\theta=2\pi ru_\theta$.
- *In words:* The same circulation measured on the loop itself.

---

**Step 5 of 7 — Equate and solve for u_θ.**

$$ u_\theta(r)=\dfrac{\Gamma}{2\pi r}\Big(1-e^{-r^2/\sigma^2}\Big) $$

- *Why we can do this:* Steps 3 and 4 describe the same number; divide by 2πr (r > 0). This is (3.29), $u_\theta=\frac{\Gamma}{2\pi r}\big(1-e^{-r^2/\sigma^2}\big)$.
- *In words:* The speed profile of the Gaussian vortex.

---

**Step 6 of 7 — Near the axis, r ≪ σ.**

$$ u_\theta\approx\dfrac{\Gamma}{2\pi\sigma^2}\,r $$

- *Why we can do this:* First-order Taylor $e^{-x}\approx1-x$ for small $x=r^2/\sigma^2$, so $1-e^{-x}\approx r^2/\sigma^2$.
- *In words:* A solid-body core with $\omega_0=\Gamma/2\pi\sigma^2$ — like the Rankine core.

---

**Step 7 of 7 — Far away, r ≫ σ.**

$$ u_\theta\to\dfrac{\Gamma}{2\pi r} $$

- *Why we can do this:* $e^{-r^2/\sigma^2}\to0$ much faster than any power of r.
- *In words:* A line vortex with B = Γ/2π — irrotational outside the core.

---

**Result**

$$ u_\theta(r)=\frac{\Gamma}{2\pi r}\big(1-\exp(-r^2/\sigma^2)\big)\ \text{(3.29), with}\ \Gamma(r)=\Gamma(1-e^{-r^2/\sigma^2}) $$

*In words:* the speed is the circulation enclosed divided by the circle's length.

**What it means.** Any axisymmetric vorticity profile gives its speed the same way: $u_\theta=\Gamma(r)/2\pi r$. The Gaussian is the Lamb–Oseen vortex of viscous flow at one instant (σ² = 4νt, Ch. 5/8). Near r = 0 the formula subtracts nearly equal numbers — compute it with `-np.expm1(-x)` (primer).

**Check it.** Units m/s ✓. Γ(∞) = Γ ✓. Numbers (Γ = 2π, σ = 1): u_θ(1) = 0.632 m/s; u_θ(5) = (1 − e^{−25})/5 = 0.2 to 11 digits ✓. (3.23), $\omega_z=\frac1r\frac{\partial}{\partial r}(ru_\theta)-\frac1r\frac{\partial u_r}{\partial\theta}$ applied to (3.29), $u_\theta=\frac{\Gamma}{2\pi r}\big(1-e^{-r^2/\sigma^2}\big)$ gives back ω_z (the sympy cell below) ✓.

In [ ]:
r, sig, Gam = sp.symbols('r sigma Gamma', positive=True)            # radius, core radius, circulation
uth = Gam/(2*sp.pi*r)*(1 - sp.exp(-r**2/sig**2))                   # the Gaussian speed profile (3.29)
wz = sp.diff(r*uth, r)/r                                            # (3.23) with u_r = 0: (1/r) d(r u_θ)/dr
print(sp.simplify(wz - Gam/(sp.pi*sig**2)*sp.exp(-r**2/sig**2)))    # 0: it gives back ω_z of (3.29)
print(sp.limit(2*sp.pi*r*uth, r, sp.oo))                            # Γ: all the circulation, far away (step 3)

> ⚠️ **Common confusion (traps in `D19`):** Forgetting the ring weight 2πr′ in step 1. Thinking the core has zero vorticity because the speed vanishes at the axis (the vorticity is largest there). Evaluating 1 − exp(−x) naively for tiny x.

📝 **Note.** **The Gaussian vortex** `N43` — the smooth version. It is the Lamb–Oseen vortex of viscous flow at one instant, with $\sigma^2=4\nu t$ growing in time
(Ch. 5, Ch. 8) — our pointer, not the book's.

$$ \omega_z(r)=\frac{\Gamma}{\pi\sigma^2}\exp\!\big(-r^2/\sigma^2\big)\quad\text{and}\quad u_\theta(r)=\frac{\Gamma}{2\pi r}\Big(1-\exp\!\big(-r^2/\sigma^2\big)\Big) \qquad \text{(3.29)} $$

> 📎 **Primer — np.expm1 and cancellation near zero.** `P107` · Near r = 0, $1-e^{-x}$ subtracts two numbers that are almost equal and loses digits (*catastrophic cancellation*).
`np.expm1(y)` computes $e^y-1$ accurately for tiny y, so $1-e^{-x}=$ `-np.expm1(-x)`. `gaussian_vortex` uses it and
returns exactly 0 at r = 0.

In [ ]:
x = 1e-12                                            # a tiny argument
print(1 - np.exp(-x), -np.expm1(-x))                 # 9.99977878e-13 (wrong from the 5th digit on) vs 9.999999999995e-13

> 📎 **Primer — scipy.optimize.brentq.** `P108` · Finds a root of f(x) = 0 inside a bracket [a, b] where f changes sign; guaranteed to converge, and fast. Choose the
bracket so it excludes roots you do not want.

In [ ]:
from scipy.optimize import brentq                    # bracketing root finder
g = lambda x: 1 + 2*x - np.exp(x)                    # the equation of D20
print(round(g(0.5), 4), round(g(3.0), 4))            # 0.3513 > 0, −13.0855 < 0: a sign change in (0.5, 3)
print(brentq(g, 0.5, 3.0))                           # 1.2564312086…

#### 🧮 Derivation — Where the Gaussian vortex's speed peaks: 1 + 2r²/σ² = e^{r²/σ²}, r ≈ 1.1209σ `D20`

**What we want to show.** Find the radius of maximum speed of the Gaussian vortex — the book quotes r ≈ 1.12091σ and leaves the work to Exercise 3.26.

**Assumptions.** Γ, σ > 0 (steps 1, 3).

**The plan.**

1. Write u_θ with one dimensionless variable.
2. Differentiate and set to zero.
3. Simplify to the book's equation.
4. Solve numerically, excluding the trivial root.

**Tools we use** (each explained before this point): Product rule (ch01 P38) and chain rule · maximum of a function (gloss) · scipy.optimize.brentq (primer in C14) · Lambert W (gloss in the check).

**We start from**

$$ \text{(3.29):}\ u_\theta(r)=\frac{\Gamma}{2\pi r}\big(1-e^{-r^2/\sigma^2}\big) $$

*In words:* zero on the axis and far away, so a maximum lies between.

---

**Step 1 of 7 — Write u_θ with x = r²/σ².**

$$ u_\theta=\dfrac{\Gamma}{2\pi\sigma}\,f(x),\qquad f(x)=\dfrac{1-e^{-x}}{\sqrt x} $$

- *Why we can do this:* Substitute r = σ√x; the constant Γ/2πσ does not affect where the maximum is, and r ↦ x is increasing for r > 0.
- *In words:* One curve f(x) for every vortex; σ only rescales it.

---

**Step 2 of 7 — Differentiate f.**

$$ f'(x)=\dfrac{e^{-x}}{\sqrt x}-\dfrac{1-e^{-x}}{2x^{3/2}} $$

- *Why we can do this:* Product rule (P38) on $(1-e^{-x})\,x^{-1/2}$: $(e^{-x})x^{-1/2}+(1-e^{-x})(-\tfrac12x^{-3/2})$; the peak is where this slope vanishes.
- *In words:* The slope of the speed curve.

---

**Step 3 of 7 — Set f′ = 0, multiply by 2x^{3/2}.**

$$ 2x\,e^{-x}-(1-e^{-x})=0 $$

- *Why we can do this:* A smooth maximum has zero slope (gloss); x > 0, so multiplying by $2x^{3/2}$ keeps the roots.
- *In words:* The condition for the peak.

---

**Step 4 of 7 — Multiply by e^{x}.**

$$ 1+2x=e^{x} $$

- *Why we can do this:* $e^x\neq0$; rearrange $2x-e^x+1=0$. With x = r²/σ² this is the book's $1+2r^2/\sigma^2=\exp(r^2/\sigma^2)$.
- *In words:* A straight line meets an exponential.

---

**Step 5 of 7 — Exclude the trivial root.**

$$ x=0\ \text{solves it but is the axis} $$

- *Why we can do this:* 1 + 0 = e⁰, but r = 0 is where u_θ = 0 — a minimum. The line and the exponential cross once more for x > 0 (the exponential starts slower, then overtakes).
- *In words:* We want the second crossing.

---

**Step 6 of 7 — Solve numerically in a bracket.**

$$ x^*=1.25643\ \ (\text{brentq on }(0.5,3)) $$

- *Why we can do this:* The equation has no elementary solution; $g(x)=1+2x-e^x$ is +0.351 at 0.5 and −13.1 at 3, so brentq (primer) finds the one root between.
- *In words:* The peak is at x ≈ 1.2564.

---

**Step 7 of 7 — Return to r and the peak speed.**

$$ r^*=\sigma\sqrt{x^*}=1.12091\,\sigma,\qquad u_{\max}=0.6382\,\dfrac{\Gamma}{2\pi\sigma} $$

- *Why we can do this:* Undo x = r²/σ²; evaluate f(x*) = (1 − e^{−1.2564})/1.1209 = 0.6382.
- *In words:* The strongest wind is about 12 % outside σ, 36 % weaker than the Rankine peak.

---

**Result**

$$ 1+2\frac{r^2}{\sigma^2}=\exp\!\big(r^2/\sigma^2\big)\ \text{,}\ r^*\approx1.12091\,\sigma\ \text{,}\ u_{\max}\approx0.6382\,\Gamma/(2\pi\sigma) $$

*In words:* the Gaussian vortex's peak sits just outside its core radius.

**What it means.** Measuring the radius of maximum wind of a vortex that looks Gaussian gives its core size σ = r*/1.121 — how tornado and wake-vortex radars are read. For the Rankine model the same radius is σ itself.

**Check it.** Units: x dimensionless ✓. Residual $1+2x^*-e^{x^*}$ < 1e-14 ✓. Lambert W cross-check (gloss: W is the inverse of $we^w$): $x^*=-W_{-1}(-e^{-1/2}/2)-\tfrac12=1.2564312086$ ✓ (`scipy.special.lambertw(z, -1)`). Numbers (Γ = 2π, σ = 1): u_max = 0.638 m/s at 1.121 m ✓.

> ⚠️ **Common confusion (traps in `D20`):** Accepting x = 0. Differentiating with respect to r in one term and x in another. Taking r* = 1.2564σ (that is x*, not √x*).

📝 **Note.** **Where the wind peaks** `N44` — Rankine: at r = σ exactly, $u_{\max}=\Gamma/2\pi\sigma$. Gaussian: where the equation below holds, i.e.
$r\approx1.1209\,\sigma$, with $u_{\max}\approx0.6382\,\Gamma/(2\pi\sigma)$ — 36 % below the Rankine peak for the same Γ
and σ.

$$ 1+2\,\frac{r^2}{\sigma^2}=\exp\!\big(r^2/\sigma^2\big)  $$

#### ✏️ Tiny example: Γ = 2π m²/s, σ = 1 m

1. Rankine inside: $u_\theta=(2\pi/2\pi\cdot1)r=r$ m/s; $\omega_z=2\pi/\pi=2$ s⁻¹.
2. Rankine peak at r = 1 m: 1 m/s; at r = 2 m: $2\pi/(2\pi\cdot2)=0.5$ m/s.
3. Gaussian at r = 1 m: $u_\theta=(1/1)(1-e^{-1})=0.632$ m/s.
4. Gaussian peak: r = 1.1209 m, $x=r^2=1.2564$, $u_\theta=(1-e^{-1.2564})/1.1209=0.7153/1.1209=0.638$ m/s.
5. Far away both → 1/r: at 5 m, 0.200 m/s (Gaussian $(1-e^{-25})/5=0.2$ to 11 digits).

In [ ]:
Gam, sig = 2*np.pi, 1.0                                      # circulation [m²/s] and core radius [m]
r = np.array([0.5, 1.0, 1.1209, 2.0, 5.0])                   # five radii [m]
print("Rankine  u, w:", [np.round(v, 4) for v in ch03.rankine_vortex(r, Gam, sig)])    # (3.28): u_θ [m/s], ω_z [1/s]
print("Gaussian u, w:", [np.round(v, 4) for v in ch03.gaussian_vortex(r, Gam, sig)])   # (3.29)
rstar = ch03.gaussian_vortex_max_radius(sig)                 # the root of 1 + 2x = e^x by brentq (D20)
print(rstar, ch03.gaussian_vortex(rstar, Gam, sig)[0])       # 1.1209064 m and the peak speed 0.63817 m/s
print(ch03.gaussian_vortex_max_radius(sig, method="lambertw"))   # the same radius from the Lambert-W formula
print(ch03.circulation_circle("gaussian", 3.0, Gamma=Gam, sigma=sig)/Gam)   # Γ(3σ)/Γ = 1 − e^−9
for name, s_, umax in (("bathtub", 5e-3, 0.3), ("tornado", 100.0, 60.0), ("tropical cyclone", 36e3, 50.0)):
    G_ = 2*np.pi*s_*umax/0.6382                              # Γ from the peak wind of a Gaussian vortex [m²/s]
    print(f"{name:17s} sigma = {s_:8.3g} m  u_max = {umax:4.1f} m/s  ->  Gamma = {G_:.2g} m^2/s, r_max = {1.1209*s_:.3g} m")

**What does the code above do?**

1. Both models at five radii: Rankine (0.5, 1, 0.892, 0.5, 0.2) m/s with a vorticity step 2 → 0; Gaussian (0.442, 0.632,
   0.638, 0.491, 0.200) m/s with a smooth bell.
2. The maximum by `brentq` (D20) and its Lambert-W twin: 1.1209064 m.
3. $\Gamma(r)/\Gamma\to1$ outside the core (D19 step 3): 0.99988 at 3σ.
4. Our order-of-magnitude real vortices span nine orders of magnitude in Γ with the same shape.

In [ ]:
x = np.linspace(1e-3, 4, 400)                                # r/σ [–]
uR, wR = ch03.rankine_vortex(x, 2*np.pi, 1.0)                # Γ = 2π, σ = 1: u is already in units of Γ/2πσ = 1 m/s; ω/2 in units of Γ/πσ² = 2 1/s
uG, wG = ch03.gaussian_vortex(x, 2*np.pi, 1.0)
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.9))           # (a) speed, (b) vorticity
a.plot(x, uR, color=COLORS["teal"], lw=2.3, label="Rankine (3.28)")          # kink at r = σ
a.plot(x, uG, "--", color=COLORS["teal"], lw=2.3, label="Gaussian (3.29)")   # smooth
a.plot(x, x, ":", color=COLORS["muted"], label="solid body"); a.plot(x, 1/x, ":", color=COLORS["grid"], label="line vortex")   # ghosts
a.plot([1.0], [1.0], "o", color=COLORS["accent"]); a.plot([1.1209], [0.6382], "o", color=COLORS["accent"], label="peaks: σ and 1.1209σ")
a.set_ylim(0, 1.3); a.set_xlabel(r"$r/\sigma$ [–]"); a.set_ylabel(r"$u_\theta\,/\,(\Gamma/2\pi\sigma)$ [–]"); a.legend(fontsize=8)
a.set_title("Speed: a spinning core and an irrotational skirt")
b.plot(x, wR/2, color=COLORS["orange"], lw=2.3, label="Rankine: a step")     # uniform core, nothing outside
b.plot(x, wG/2, "--", color=COLORS["orange"], lw=2.3, label="Gaussian: a bell")   # e^{−r²/σ²}
b.set_xlabel(r"$r/\sigma$ [–]"); b.set_ylabel(r"$\omega_z\,/\,(\Gamma/\pi\sigma^2)$ [–]"); b.legend(fontsize=8)   # dimensionless
b.set_title("Vorticity: where the core is")
savefig(fig, "ch03", "vortex_profiles"); plt.show()           # save to outputs/ch03 and draw

**What you see.** Two humps with peaks at 1 and ≈ 1.12 (left); a step and a bell (right).

**How to read it.** The core is where the vorticity is; outside it both speed profiles follow the line vortex $\Gamma/2\pi r$. The
smooth Gaussian vorticity moves the peak outward and lowers it (0.638 instead of 1 in units of Γ/2πσ).

**What would change if…** …σ were doubled at the same Γ: in physical units the peak would halve and move twice as far out ($u_{\max}\propto\Gamma/\sigma$).

In [ ]:
rr = np.linspace(0.005, 5, 300)                              # radius [m]

def profiles(s):                                             # Γ = 2π m²/s, core radius s [m]
    rs = ch03.gaussian_vortex_max_radius(s)                  # Gaussian peak radius (D20)
    return {"Rankine u_θ": (rr, ch03.rankine_vortex(rr, 2*np.pi, s)[0]),        # (3.28)
            "Gaussian u_θ": (rr, ch03.gaussian_vortex(rr, 2*np.pi, s)[0]),      # (3.29)
            "peaks (σ and 1.1209σ)": ([s, rs], [1.0/s, ch03.gaussian_vortex(rs, 2*np.pi, s)[0]])}   # Rankine peak Γ/2πσ = 1/s

fig = slider_figure(profiles, "σ", np.linspace(0.25, 2, 15 if not FAST else 8), unit="m", xlabel="r [m]",
                    ylabel="u_θ [m/s]", xrange=[0, 5], yrange=[0, 4.2], modes={"peaks (σ and 1.1209σ)": "markers"},
                    title="A smaller core spins faster: u_max ∝ Γ/σ")    # fixed axes
recolor(fig, {"Rankine u_θ": COLORS["teal"], "Gaussian u_θ": COLORS["teal"], "peaks (σ and 1.1209σ)": COLORS["accent"]},
        dashes={"Gaussian u_θ": "dash"})                     # Gaussian dashed, as in the static figure
fig.show()                                                   # draw it

**What does the code above do?**

The slider changes the core radius σ at fixed Γ = 2π m²/s; the markers sit at the two peaks.

**What you see.** A teal kinked curve (Rankine), a dashed smooth one (Gaussian) and two purple markers.

**How to read it.** The Rankine peak sits on the kink at σ, the Gaussian peak at 1.12σ, both moving outward and down as σ grows; beyond about 2σ the two curves coincide with Γ/2πr.

**What would change if…** …Γ doubled: every curve would double in height; the peak radii would not move.

#### 🎮 Interactive: Going round in circles ≠ spinning

**Why interactive:** "going round" and "spinning" can only be told apart in motion; watching wheels orbit and turn (or not) on one clock, and dragging a loop on and off the axis, makes the difference visible.
Four vortices on one stage: paddle wheels ride with the flow and turn at half the local vorticity, a loop you can drag
measures the circulation, and the profiles $u_\theta(r)$ and $\omega_z(r)$ show where the core is. The same clock runs
the tracers, the wheels and the circulation graph.

**What to try:**
- Mode 'line vortex': the wheels orbit but always point the same way; drag the loop off the axis — Γ drops to 0.
- Mode 'solid body': every wheel turns once per revolution.
- Mode 'Gaussian': drag σ and watch the peak marker sit at 1.1209σ.
- Preset 'tropical cyclone': read the radius of maximum wind and the core vorticity in Explain.

In [ ]:
show_viz("ch03", "vortex_paddle_wheels")   # full-width explainer; ⤢ Full screen for more room

**What would change if…** …the fluid were viscous? The Gaussian core would spread as $\sigma^2=4\nu t$ while Γ stays fixed (Ch. 5). And all of
this has been about *points* — §3.6 asks how a quantity inside a whole moving volume changes.

---

## 3.6 Reynolds Transport Theorem

**What is this section about?** How fast does the amount of something inside a volume change when the volume itself
moves and deforms? In one dimension this is Leibniz's rule; in three it is the Reynolds transport theorem — the bridge
from "what happens to a moving lump of fluid" to equations at fixed points, used for every conservation law in Ch. 4.

### 🧩 The Reynolds transport theorem: change inside a moving volume (3.35) `C15`

*The question:* A balloon is being inflated in a room that is warming up. How fast does the heat content inside the balloon change —
and which part of that comes from the warming, which from the balloon's growing skin?

*In one line:* $\frac{d}{dt}\int_{V^*}F\,dV=\int_{V^*}\frac{\partial F}{\partial t}dV+\int_{A^*}F\,\mathbf b\cdot\mathbf n\,dA$ *(3.35)*

#### The problem in plain words

Conservation laws are about *things* — a lump of fluid keeps its mass, Newton's law acts on it — but the lump moves and
deforms. Engineers draw a *control volume* (a pipe section, a jet engine, a layer of ocean) whose walls may move. In
both cases we need d/dt of an integral whose region moves: the answer has an "inside" part and a "swept by the walls"
part.

#### The idea

```
d/dt ∫_{V*(t)} F dV  =  ∫_{V*} ∂F/∂t dV   +   ∮_{A*} F b·n dA           (3.35)
   change of the total    change in place       what the moving wall sweeps in (b·n > 0) or out (b·n < 0)
1-D:  d/dt ∫_a^b F dx = ∫_a^b ∂F/∂t dx + ḃ F(b) − ȧ F(a)                  (3.30)
```

In symbols: $\frac{d}{dt}\int_{V^*}F\,dV=\int_{V^*}\frac{\partial F}{\partial t}dV+\int_{A^*}F\,\mathbf b\cdot\mathbf n\,dA$ *(3.35)*, and in 1-D
$\frac{d}{dt}\int_{a}^{b}F\,dx=\int_a^b\frac{\partial F}{\partial t}dx+\frac{db}{dt}F(b,t)-\frac{da}{dt}F(a,t)$ *(3.30)*.

📝 **Note.** **Why we need it** `N45` — every integral conservation law of Ch. 4 (mass, momentum, energy) is a time derivative of an integral over a moving,
deforming volume.

> 📎 **Primer — differentiation under the integral sign.** `P109` · If the limits are fixed, the time derivative may pass inside:
$\frac{d}{dt}\int_a^bF(x,t)\,dx=\int_a^b\frac{\partial F}{\partial t}dx$ (F and ∂F/∂t continuous). It is the special case
of Leibniz's rule with ȧ = ḃ = 0.

In [ ]:
t = sp.symbols('t'); x = sp.symbols('x')             # time and position
F = sp.sin(t)*x**2 + t**2*x                          # any smooth F(x, t)
print(sp.simplify(sp.diff(sp.integrate(F, (x, 0, 1)), t) - sp.integrate(sp.diff(F, t), (x, 0, 1))))   # 0

📝 **Note.** **Leibniz's theorem** `N46` — d/dt of an integral with moving limits. The book cites a proof it does not give; here it is (D21).

$$ \frac{d}{dt}\int_{x=a(t)}^{x=b(t)}F(x,t)\,dx=\int_a^b\frac{\partial F}{\partial t}dx+\frac{db}{dt}F(b,t)-\frac{da}{dt}F(a,t) \qquad \text{(3.30)} $$

#### 🧮 Derivation — Leibniz's theorem, Eq. (3.30) `D21` (Eq. 3.30)

**What we want to show.** Prove the rule for differentiating an integral whose limits move — the book cites it without proof and builds the Reynolds transport theorem on it.

**Assumptions.** F and ∂F/∂t continuous (steps 1, 5); a(t), b(t) differentiable (step 3).

**The plan.**

1. Write I with an antiderivative.
2. Differentiate each end with the chain rule.
3. Turn the leftover time derivatives into an integral of ∂F/∂t.

**Tools we use** (each explained before this point): Fundamental theorem of calculus (ch02 P84) · chain rule (ch01 P49) · differentiation under the integral sign (primer in C15).

**We start from**

$$ I(t)=\int_{a(t)}^{b(t)}F(x,t)\,dx $$

*In words:* the amount of F between two moving end points.

---

**Step 1 of 7 — Introduce an antiderivative in x.**

$$ \Phi(x,t)\equiv\displaystyle\int_c^xF(x',t)\,dx',\qquad\dfrac{\partial\Phi}{\partial x}=F $$

- *Why we can do this:* With a fixed lower limit c, Φ exists for continuous F and its x-derivative is F (fundamental theorem of calculus, P84).
- *In words:* The running total of F from a fixed point.

---

**Step 2 of 7 — Write I with Φ.**

$$ I(t)=\Phi(b(t),t)-\Phi(a(t),t) $$

- *Why we can do this:* Fundamental theorem of calculus: an integral is the antiderivative at the top minus at the bottom.
- *In words:* The amount between the ends = total up to b minus total up to a.

---

**Step 3 of 7 — Chain rule at the upper end.**

$$ \dfrac{d}{dt}\Phi(b(t),t)=\dfrac{\partial\Phi}{\partial x}\Big|_b\dfrac{db}{dt}+\dfrac{\partial\Phi}{\partial t}\Big|_b $$

- *Why we can do this:* Φ depends on t through its first argument b(t) and directly — the chain rule (P49), as in D02 step 3.
- *In words:* The total up to b changes because b moves and because F changes.

---

**Step 4 of 7 — Both ends, with ∂Φ/∂x = F.**

$$ \dfrac{dI}{dt}=F(b,t)\dfrac{db}{dt}-F(a,t)\dfrac{da}{dt}+\dfrac{\partial\Phi}{\partial t}\Big|_b-\dfrac{\partial\Phi}{\partial t}\Big|_a $$

- *Why we can do this:* Step 3 for b minus the same for a, using step 1 for ∂Φ/∂x.
- *In words:* The end terms have appeared.

---

**Step 5 of 7 — Differentiate Φ under the integral sign.**

$$ \dfrac{\partial\Phi}{\partial t}(x,t)=\displaystyle\int_c^x\dfrac{\partial F}{\partial t}(x',t)\,dx' $$

- *Why we can do this:* In Φ the limits c and x are held fixed while t varies, so the derivative may pass inside (primer; F and ∂F/∂t continuous). The book skips this.
- *In words:* Φ's time change is the sum of F's local changes.

---

**Step 6 of 7 — Subtract at the two ends.**

$$ \dfrac{\partial\Phi}{\partial t}\Big|_b-\dfrac{\partial\Phi}{\partial t}\Big|_a=\displaystyle\int_a^b\dfrac{\partial F}{\partial t}\,dx $$

- *Why we can do this:* The part from c to a is common to both and cancels.
- *In words:* The change of F in place, summed between the ends.

---

**Step 7 of 7 — Assemble (3.30).**

$$ \dfrac{d}{dt}\displaystyle\int_{a(t)}^{b(t)}F\,dx=\int_a^b\dfrac{\partial F}{\partial t}dx+\dfrac{db}{dt}F(b,t)-\dfrac{da}{dt}F(a,t) $$

- *Why we can do this:* Put step 6 into step 4 and reorder.
- *In words:* Change in place + what the upper end sweeps in − what the lower end sweeps out.

---

**Result**

$$ \frac{d}{dt}\int_{x=a(t)}^{x=b(t)}F(x,t)\,dx=\int_a^b\frac{\partial F}{\partial t}dx+\frac{db}{dt}F(b,t)-\frac{da}{dt}F(a,t)\ \text{(3.30)} $$

*In words:* interior change plus the gain at the moving upper limit minus the loss at the moving lower limit (Fig. 3.17's three strips).

**What it means.** The 1-D Reynolds transport theorem: (3.35), $\frac{d}{dt}\int_{V^*}F\,dV=\int_{V^*}\frac{\partial F}{\partial t}dV+\int_{A^*}F\,\mathbf b\cdot\mathbf n\,dA$ with the "surface" reduced to two end points with normals ±1 (D22 step 12). Layer budgets (Ch. 13) and the boundary-layer momentum integral (Ch. 9) are Leibniz in action.

**Check it.** Units: [F]·m/s on every term ✓. Fixed limits: only the integral of ∂F/∂t ✓. F = 1: dI/dt = ḃ − ȧ, the rate the length changes ✓. Numbers: F = x²t, a = t, b = t² at t = 2: 18.667 + 128 − 8 = 138.667 = d/dt[(t⁷ − t⁴)/3] = (7·64 − 4·8)/3 ✓ (`ch03.leibniz_example(2.0)`); the transport explainer's default 0.6 + 1.0 + 0.3 = 1.9 ✓. The sympy cell below runs the case $F=x^2t$, $a=t$, $b=t^2$.

In [ ]:
x, t = sp.symbols('x t', positive=True)             # position and time
F, a, b = x**2*t, t, t**2                           # the check case: F = x²t between a = t and b = t²
lhs = sp.diff(sp.integrate(F, (x, a, b)), t)        # left side: differentiate the integral itself
rhs = sp.integrate(sp.diff(F, t), (x, a, b)) + sp.diff(b, t)*F.subs(x, b) - sp.diff(a, t)*F.subs(x, a)   # right side of (3.30)
print(sp.simplify(lhs - rhs))                       # 0: Leibniz holds
print(lhs.subs(t, 2))                               # 416/3 = 138.667 at t = 2 s

> ⚠️ **Common confusion (traps in `D21`):** The sign of the lower-limit term (−ȧF(a): an end moving left *adds* F). Treating ∂Φ/∂t as ∂F/∂t (it is an integral of it). Passing d/dt inside an integral whose limits move.

📝 **Note.** **Fig. 3.17, the picture of (3.30)** `N47` — three thin strips — a band of height $\frac{\partial F}{\partial t}dt$ over [a, b] (the interior change), a strip of
width db and height F(b) gained at the upper end, a strip of width da and height F(a) lost at the lower end; the corner
pieces are dt² small. Below: $F=1+0.5x+0.3t$ between $a(t)=1-0.2t$ and $b(t)=3+0.4t$, so
$\frac{d}{dt}\int_a^bF\,dx=\int_a^b\frac{\partial F}{\partial t}dx+\frac{db}{dt}F(b,t)-\frac{da}{dt}F(a,t)$.

In [ ]:
Fw, dFw = ch03.rtt_field("warming", dim=1)                    # F = 1 + 0.5 x + 0.3 t and ∂F/∂t = 0.3
a_ = lambda t: 1.0 - 0.2*t; b_ = lambda t: 3.0 + 0.4*t         # the moving limits [m]
nfr = 12 if not FAST else 8                                   # frames to step through
times = np.linspace(0, 2.2, nfr); dt = 0.4                    # frame times and a (visible) strip time Δt [s]
xx = np.linspace(0, 4.5, 300)
fig, ax = plt.subplots(figsize=(7.5, 4))                     # one panel

def update(i):                                                # redraw frame i from scratch (few artists)
    ax.clear(); t = times[i]; a, b = a_(t), b_(t)             # this frame's time and limits
    ax.plot(xx, Fw(xx, t), color=COLORS["ink"], lw=2, label="$F(x, t)$")    # the field now
    xi = np.linspace(a, b, 100)                               # points between the limits
    ax.fill_between(xi, 0, Fw(xi, t), color=COLORS["grid"], alpha=0.8)                     # ∫_a^b F dx now
    ax.fill_between(xi, Fw(xi, t), Fw(xi, t + dt), color=COLORS["blue"], alpha=0.5, label="interior: Δt ∂F/∂t")   # F rises in place
    xb = np.linspace(b, b_(t + dt), 20); ax.fill_between(xb, 0, Fw(xb, t), color=COLORS["orange"], alpha=0.6, label="gained at b: ḃΔt F(b)")
    xa = np.linspace(a_(t + dt), a, 20); ax.fill_between(xa, 0, Fw(xa, t), color=COLORS["rose"], alpha=0.6, label="a moves left: −ȧΔt F(a) > 0")
    L = ch03.leibniz_terms(Fw, dFw, a, b, -0.2, 0.4, t)       # the three rates of (3.30) at this t
    ax.set_title(f"t = {t:.2f} s: interior {L.interior:.2f} + upper {L.upper:.2f} − lower ({L.lower:.2f}) = {L.total:.2f} [F·m/s]", fontsize=9)
    ax.set_xlim(0, 4.5); ax.set_ylim(0, 4.0); ax.set_xlabel("$x$ [m]"); ax.set_ylabel("$F$"); ax.legend(fontsize=7, loc="upper left")
    return []                                                 # everything was redrawn

show_animation(animate(update, frames=nfr, fig=fig, interval=600), player="frames")   # step through the strips

**What does the code above do?**

1. `rtt_field("warming", dim=1)` is the shared test field of the notebook and the explainer.
2. Each frame shades the three strips of
   $\frac{d}{dt}\int_a^bF\,dx=\int_a^b\frac{\partial F}{\partial t}dx+\frac{db}{dt}F(b,t)-\frac{da}{dt}F(a,t)$ *(3.30)*
   for a visible step Δt = 0.4 s, and prints the exact rates from
   `leibniz_terms` (the lower term is $\dot aF(a)$, which is *subtracted*).

**What you see.** A grey area under a rising line, a blue band on top of it, an orange strip on the right and a rose strip on the left.

**How to read it.** The orange strip grows on the right as b advances; the rose strip is *added* on the left here because a moves left
(ȧ < 0 makes $-\dot aF(a)$ positive). At t = 0 the three rates are 0.6, 1.0 and 0.3, total 1.9 (checked in the code
below).

**What would change if…** …a moved right instead (ȧ > 0): the rose strip would be taken away and the total would drop.

📝 **Note.** **Control volume** `N48` — $V^*(t)$ with **control surface** $A^*(t)$, outward unit normal $\mathbf n$ and surface velocity $\mathbf b$ (Fig.
3.18). The surface need not follow the fluid: $\mathbf b=\mathbf u$ for a *material* volume, $\mathbf b=0$ for a volume
fixed in space, anything else for a piston, a balloon or a moving layer. `ch03.GrowingSphere`, `GrowingCylinder`,
`MovingBox`, `GrowingCone` are concrete examples (Python objects that know their volume, surface and b at any t).

> 📎 **Primer — signed swept volume of a moving surface.** `P110` · In a short time Δt a surface patch dA moving with velocity $\mathbf b$ sweeps a thin prism of height
$(\mathbf b\Delta t)\cdot\mathbf n$ — only the normal component counts (sliding along the surface sweeps nothing). With
the outward $\mathbf n$ the volume is **signed**: positive where the wall advances outward (the region gains volume),
negative where it retreats.

In [ ]:
n = np.array([1.0, 0.0]); dA, dt = 0.01, 0.1         # outward normal, patch area [m²], time step [s]
for b in ([2.0, 0.0], [0.0, 5.0], [-1.0, 3.0]):      # advancing, sliding, retreating wall velocities [m/s]
    print(b, np.dot(b, n)*dt*dA)                     # 0.002, 0.0 (sliding), -0.001 (retreating) [m³]

📝 **Note.** **The start of the derivation** `N49` — the definition of a time derivative (Fig. 3.18: the volume solid at t, dashed at t + Δt).

$$ \frac{d}{dt}\int_{V^*(t)}F(\mathbf x,t)dV=\lim_{\Delta t\to0}\frac{1}{\Delta t}\Big\{\int_{V^*(t+\Delta t)}F(\mathbf x,t+\Delta t)dV-\int_{V^*(t)}F(\mathbf x,t)dV\Big\} \qquad \text{(3.31)} $$

📝 **Note.** **Four terms** `N50` — Splitting the new volume, $\Delta V\equiv V^*(t+\Delta t)-V^*(t)$, and Taylor-expanding F in time gives:

$$ \int_{V^*(t+\Delta t)}F(\mathbf x,t+\Delta t)dV\cong\int_{V^*(t)}F\,dV+\int_{V^*(t)}\Delta t\frac{\partial F}{\partial t}dV+\int_{\Delta V}F\,dV+\int_{\Delta V}\Delta t\frac{\partial F}{\partial t}dV \qquad \text{(3.32)} $$

📝 **Note.** **The first cancels, the last is second order** `N51` —

$$ \frac{d}{dt}\int_{V^*(t)}F\,dV=\lim_{\Delta t\to0}\frac{1}{\Delta t}\Big\{\int_{V^*(t)}\Delta t\frac{\partial F}{\partial t}dV+\int_{\Delta V}F\,dV\Big\} \qquad \text{(3.33)} $$

📝 **Note.** **…and the sliver is a surface integral** `N52` — Each of these is a step of D22 below, where the moves between them are filled in.

$$ \int_{\Delta V}F(\mathbf x,t)dV\cong\int_{A^*(t)}F(\mathbf x,t)(\mathbf b\Delta t\cdot\mathbf n)dA\quad\text{as}\quad\Delta t\to0 \qquad \text{(3.34)} $$

#### 🧮 Derivation — The Reynolds transport theorem: (3.31) → (3.32) → (3.33) → (3.34) → (3.35) `D22` (Eq. 3.35)

**What we want to show.** Find the rate of change of the amount of F inside a volume V*(t) whose surface moves with any velocity b — the tool that turns every conservation law of Ch. 4 into equations. The book gives the steps; we fill the gaps (the sign of the sliver, the orders of smallness, the value of F in the sliver, the 1-D reduction).

**Assumptions.** F and ∂F/∂t continuous (steps 2, 9); b continuous and A* piecewise smooth, so the sliver has thickness O(Δt) everywhere (steps 5, 8); Δt → 0 at the end.

**The plan.**

1. Split the new region into the old one plus a thin sliver ΔV, and Taylor-expand F in time → (3.32), $\int_{V^*(t+\Delta t)}F(t+\Delta t)dV\cong\int_{V^*}F\,dV+\int_{V^*}\Delta t\frac{\partial F}{\partial t}dV+\int_{\Delta V}F\,dV+\int_{\Delta V}\Delta t\frac{\partial F}{\partial t}dV$.
2. Cancel and drop what is second order → (3.33), $\frac{d}{dt}\int_{V^*}F\,dV=\lim_{\Delta t\to0}\frac{1}{\Delta t}\Big\{\int_{V^*}\Delta t\frac{\partial F}{\partial t}dV+\int_{\Delta V}F\,dV\Big\}$.
3. Turn the sliver into a surface integral → (3.34), $\int_{\Delta V}F\,dV\cong\int_{A^*}F\,(\mathbf b\Delta t\cdot\mathbf n)\,dA$.
4. Take the limit → (3.35), $\frac{d}{dt}\int_{V^*}F\,dV=\int_{V^*}\frac{\partial F}{\partial t}dV+\int_{A^*}F\,\mathbf b\cdot\mathbf n\,dA$; check the 1-D case.

**Tools we use** (each explained before this point): (3.31), $\frac{d}{dt}\int_{V^*}F\,dV=\lim_{\Delta t\to0}\frac{1}{\Delta t}\Big\{\int_{V^*(t+\Delta t)}F(t+\Delta t)dV-\int_{V^*(t)}F\,dV\Big\}$ (N49) · signed swept volume of a moving surface (primer in C15) · first-order Taylor expansion in time (ch01 P26; gloss in step 2) · limits and orders of smallness (ch02 P68) · mean-value theorem for integrals (ch02 P85) · Leibniz (3.30), $\frac{d}{dt}\int_{a}^{b}F\,dx=\int_a^b\frac{\partial F}{\partial t}dx+\frac{db}{dt}F(b,t)-\frac{da}{dt}F(a,t)$ (D21).

**We start from**

$$ \text{(3.31):}\ \dfrac{d}{dt}\displaystyle\int_{V^*(t)}F(\mathbf x,t)dV=\lim_{\Delta t\to0}\dfrac{1}{\Delta t}\Big\{\int_{V^*(t+\Delta t)}F(\mathbf x,t+\Delta t)dV-\int_{V^*(t)}F(\mathbf x,t)dV\Big\} $$

*In words:* the definition of a time derivative, applied to an integral whose region moves.

---

**Step 1 of 12 — Split the new region.**

$$ \displaystyle\int_{V^*(t+\Delta t)}=\int_{V^*(t)}+\int_{\Delta V},\qquad\Delta V\equiv V^*(t+\Delta t)-V^*(t) $$

- *Why we can do this:* Integrals add over regions. ΔV is **signed**: where the surface retreats (b·n < 0) the sliver is subtracted (primer) — one formula covers growth and shrinking; the book leaves this implicit.
- *In words:* New region = old region + the band the walls swept.

---

**Step 2 of 12 — Taylor-expand F in time.**

$$ F(\mathbf x,t+\Delta t)=F(\mathbf x,t)+\Delta t\,\dfrac{\partial F}{\partial t}(\mathbf x,t)+O(\Delta t^2) $$

- *Why we can do this:* First-order Taylor expansion in t at fixed x (gloss; P26); F is continuously differentiable. We want everything at time t.
- *In words:* F a moment later ≈ F now + its local rate × Δt.

---

**Step 3 of 12 — Insert both into the first integral.**

$$ \displaystyle\int_{V^*(t+\Delta t)}\!F(t+\Delta t)\,dV\cong\int_{V^*}\!F\,dV+\int_{V^*}\!\Delta t\,\dfrac{\partial F}{\partial t}dV+\int_{\Delta V}\!F\,dV+\int_{\Delta V}\!\Delta t\,\dfrac{\partial F}{\partial t}dV $$

- *Why we can do this:* Apply step 2 inside both pieces of step 1 and distribute; this is (3.32), $\int_{V^*(t+\Delta t)}F(t+\Delta t)dV\cong\int_{V^*}F\,dV+\int_{V^*}\Delta t\frac{\partial F}{\partial t}dV+\int_{\Delta V}F\,dV+\int_{\Delta V}\Delta t\frac{\partial F}{\partial t}dV$.
- *In words:* Four terms: old amount, local change, amount in the sliver, local change in the sliver.

---

**Step 4 of 12 — Cancel against the old amount.**

$$ \text{bracket of (3.31)}=\displaystyle\int_{V^*}\Delta t\,\dfrac{\partial F}{\partial t}dV+\int_{\Delta V}F\,dV+\int_{\Delta V}\Delta t\,\dfrac{\partial F}{\partial t}dV $$

- *Why we can do this:* The first term of (3.32), $\int_{V^*(t+\Delta t)}F(t+\Delta t)dV\cong\int_{V^*}F\,dV+\int_{V^*}\Delta t\frac{\partial F}{\partial t}dV+\int_{\Delta V}F\,dV+\int_{\Delta V}\Delta t\frac{\partial F}{\partial t}dV$ is exactly the $\int_{V^*(t)}F\,dV$ subtracted in (3.31), $\frac{d}{dt}\int_{V^*}F\,dV=\lim_{\Delta t\to0}\frac{1}{\Delta t}\Big\{\int_{V^*(t+\Delta t)}F(t+\Delta t)dV-\int_{V^*(t)}F\,dV\Big\}$.
- *In words:* Only the changes are left.

---

**Step 5 of 12 — Size the sliver.**

$$ \Delta V=O(\Delta t)\ \Rightarrow\ \displaystyle\int_{\Delta V}\Delta t\,\dfrac{\partial F}{\partial t}dV=O(\Delta t^2) $$

- *Why we can do this:* The sliver's thickness is (b·n)Δt, so its volume is O(Δt); a bounded integrand times O(Δt) volume times Δt is O(Δt²). The book says "second order"; this is why.
- *In words:* The last term is much smaller than the others.

---

**Step 6 of 12 — Divide by Δt and drop it.**

$$ \dfrac{d}{dt}\displaystyle\int_{V^*}F\,dV=\lim_{\Delta t\to0}\dfrac{1}{\Delta t}\Big\{\int_{V^*}\Delta t\,\dfrac{\partial F}{\partial t}dV+\int_{\Delta V}F\,dV\Big\} $$

- *Why we can do this:* After dividing by Δt the dropped term is O(Δt) → 0 (orders of smallness, P68). This is (3.33), $\frac{d}{dt}\int_{V^*}F\,dV=\lim_{\Delta t\to0}\frac{1}{\Delta t}\Big\{\int_{V^*}\Delta t\frac{\partial F}{\partial t}dV+\int_{\Delta V}F\,dV\Big\}$.
- *In words:* Two contributions survive.

---

**Step 7 of 12 — Take the volume term's limit.**

$$ \dfrac{1}{\Delta t}\displaystyle\int_{V^*}\Delta t\,\dfrac{\partial F}{\partial t}dV=\int_{V^*}\dfrac{\partial F}{\partial t}dV $$

- *Why we can do this:* Δt is a constant for the integral, so it comes out and cancels exactly — no limit needed.
- *In words:* The change of F in place, summed over the volume.

---

**Step 8 of 12 — The volume swept by one patch.**

$$ dV_{\rm swept}=(\mathbf b\,\Delta t)\cdot\mathbf n\,dA $$

- *Why we can do this:* A patch dA moving at b for Δt sweeps a thin prism whose height is the normal part of its displacement; sliding along the surface sweeps nothing (primer). Positive where it advances.
- *In words:* Each piece of wall adds (or removes) a thin slab.

---

**Step 9 of 12 — F in the sliver ≈ its surface value.**

$$ \displaystyle\int_{\Delta V}F\,dV\cong\int_{A^*(t)}F\,(\mathbf b\,\Delta t\cdot\mathbf n)\,dA $$

- *Why we can do this:* Add the slabs of step 8. Within a slab of thickness O(Δt), F differs from its surface value by O(Δt) (mean-value theorem, P85), an O(Δt²) error. This is (3.34), $\int_{\Delta V}F\,dV\cong\int_{A^*}F\,(\mathbf b\Delta t\cdot\mathbf n)\,dA$.
- *In words:* The sliver's content is F at the wall times the swept volume.

---

**Step 10 of 12 — Divide by Δt.**

$$ \dfrac{1}{\Delta t}\displaystyle\int_{\Delta V}F\,dV\to\int_{A^*}F\,\mathbf b\cdot\mathbf n\,dA $$

- *Why we can do this:* Δt comes out of the surface integral and cancels; the O(Δt²) error of step 9 becomes O(Δt) and vanishes in the limit.
- *In words:* The rate at which the moving walls sweep F in or out.

---

**Step 11 of 12 — Assemble the theorem.**

$$ \dfrac{d}{dt}\displaystyle\int_{V^*(t)}F\,dV=\int_{V^*(t)}\dfrac{\partial F}{\partial t}dV+\int_{A^*(t)}F\,\mathbf b\cdot\mathbf n\,dA $$

- *Why we can do this:* Put steps 7 and 10 into (3.33), $\frac{d}{dt}\int_{V^*}F\,dV=\lim_{\Delta t\to0}\frac{1}{\Delta t}\Big\{\int_{V^*}\Delta t\frac{\partial F}{\partial t}dV+\int_{\Delta V}F\,dV\Big\}$. This is (3.35), $\frac{d}{dt}\int_{V^*}F\,dV=\int_{V^*}\frac{\partial F}{\partial t}dV+\int_{A^*}F\,\mathbf b\cdot\mathbf n\,dA$.
- *In words:* Change inside = change in place + what the walls sweep.

---

**Step 12 of 12 — Check the 1-D case.**

$$ A^*=\{a,b\},\ \mathbf n=\mp1\ \Rightarrow\ \dfrac{d}{dt}\displaystyle\int_a^bF\,dx=\int_a^b\dfrac{\partial F}{\partial t}dx+\dot bF(b)-\dot aF(a) $$

- *Why we can do this:* In one dimension the 'surface' is the two end points with outward normals −1 at a and +1 at b; the surface integral becomes a two-term sum, and (3.30), $\frac{d}{dt}\int_{a}^{b}F\,dx=\int_a^b\frac{\partial F}{\partial t}dx+\frac{db}{dt}F(b,t)-\frac{da}{dt}F(a,t)$ returns.
- *In words:* Leibniz is the Reynolds transport theorem in 1-D.

---

**Result**

$$ \frac{d}{dt}\int_{V^*(t)}F(\mathbf x,t)dV=\int_{V^*(t)}\frac{\partial F(\mathbf x,t)}{\partial t}dV+\int_{A^*(t)}F(\mathbf x,t)\,\mathbf b\cdot\mathbf n\,dA\ \text{(3.35)} $$

*In words:* the rate of change of what is inside a moving volume = what changes in place + what the moving surface sweeps in (b·n > 0) or out (b·n < 0).

**What it means.** The theorem is pure kinematics — no physics law is used. With b = u (a material volume) it gives the rate of change of a lump of fluid's mass, momentum or energy, which Ch. 4 sets equal to zero, the net force, or the heat and work; with b = 0 it gives the fixed-control-volume forms engineers use. It fails only if F or the surface is not smooth enough (a shock inside V*, a surface that tears).

**Check it.** Units: [F]·m³/s on every term ✓. b = 0: d/dt passes inside the integral ✓. F = 1: dV*/dt = ∮b·n dA ✓ (D23). Numbers: growing sphere R = 1 m, Ṙ = 0.1 m/s, F = t at t = 1 s: 4.189 + 1.257 = 5.445 = d/dt(tV) ✓; Ex. 3.2: 0.1047 m³/s both ways ✓; the (3.32), $\int_{V^*(t+\Delta t)}F(t+\Delta t)dV\cong\int_{V^*}F\,dV+\int_{V^*}\Delta t\frac{\partial F}{\partial t}dV+\int_{\Delta V}F\,dV+\int_{\Delta V}\Delta t\frac{\partial F}{\partial t}dV$ terms for the growing sphere: T4 ∝ Δt² (slope 2.00) ✓.

In [ ]:
import sympy as sp                                             # symbolic algebra
t, dt, R0, Rd = sp.symbols('t Delta_t R_0 Rdot', positive=True) # time, step, radius at t = 0, growth rate
r, th, ph = sp.symbols('r theta phi', nonnegative=True)         # spherical coordinates of a point
R = R0 + Rd*t                                                  # radius of V*(t); on its surface b·n = Rdot
F = t*(r*sp.sin(th)*sp.cos(ph))**2                             # F = t x1^2 in spherical coordinates
dV = r**2*sp.sin(th)                                           # volume element r^2 sin(theta) dr dtheta dphi
def vol(expr, r0, r1):                                         # integral of expr over the shell r0 < r < r1
    return sp.integrate(expr*dV, (r, r0, r1), (th, 0, sp.pi), (ph, 0, 2*sp.pi))
lhs = sp.diff(vol(F, 0, R), t)                                 # d/dt of the integral over V*(t): left of (3.31)
vterm = vol(sp.diff(F, t), 0, R)                               # step 7: integral of dF/dt over V*
sterm = sp.integrate((F*Rd*dV).subs(r, R), (th, 0, sp.pi), (ph, 0, 2*sp.pi))  # step 10: surface F b.n dA, dA = R^2 sin
print(sp.simplify(lhs - (vterm + sterm)))                      # 0: (3.35) holds (step 11)
T3 = vol(F, R, R.subs(t, t + dt))                              # step 1: F over the swept shell (third term of (3.32))
T4 = vol(dt*sp.diff(F, t), R, R.subs(t, t + dt))               # fourth term of (3.32)
print(sp.series(T4, dt, 0, 3).removeO())                       # starts at dt^2: second order (step 5)
print(sp.simplify(sp.limit(T3/dt, dt, 0) - sterm))             # 0: (3.34), the sliver becomes the surface term (steps 9-10)

> ⚠️ **Common confusion (traps in `D22`):** Using an unsigned swept volume (and then adding a separate "outflow" term). Keeping the ΔV·Δt term. Evaluating F in the sliver at t + Δt. Moving d/dt inside ∫_{V*(t)} when b ≠ 0.

📝 **Note.** **Two readings of (3.35)** `N53` — of $\frac{d}{dt}\int_{V^*}F\,dV=\int_{V^*}\frac{\partial F}{\partial t}dV+\int_{A^*}F\,\mathbf b\cdot\mathbf n\,dA$.
(1) With F = 1 it says the volume changes at the rate the surface sweeps: $dV^*/dt=\int_{A^*}\mathbf b\cdot\mathbf n\,dA$;
for a small material volume (b = u) this is $\frac{1}{\delta V}\frac{D}{Dt}(\delta V)=\frac{\partial u_i}{\partial x_i}$
*(3.14)* — D23. (2) For a small material volume it contains the material derivative
$\frac{DF}{Dt}=\frac{\partial F}{\partial t}+\mathbf u\cdot\nabla F$ *(3.5)*, plus a term $F\nabla\cdot\mathbf u$ from the
volume change — D24. Where b·n > 0 the surface advances, where b·n < 0 it retreats; one signed term covers both. Only for
a fixed volume (b = 0) may d/dt pass inside the integral.

#### 🧮 Derivation — With F = 1 and b = u the theorem gives the volumetric strain rate, Eq. (3.14) `D23` (Eq. 3.14)

**What we want to show.** Show that the Reynolds transport theorem, applied to a small material volume with F = 1, reproduces (3.14), $\frac{1}{\delta V}\frac{D(\delta V)}{Dt}=\frac{\partial u_i}{\partial x_i}$ — the book's first "physical interpretation", left to Exercise 3.28.

**Assumptions.** Material volume: its surface moves with the fluid, b = u (start); u smooth (step 2).

**The plan.**

1. Simplify the left side and the volume term.
2. Turn the surface integral into a volume integral (Gauss).
3. Shrink the volume (mean value).

**Tools we use** (each explained before this point): (3.35), $\frac{d}{dt}\int_{V^*}F\,dV=\int_{V^*}\frac{\partial F}{\partial t}dV+\int_{A^*}F\,\mathbf b\cdot\mathbf n\,dA$ (D22) · Gauss' divergence theorem (2.30), $\int_V\nabla\cdot\mathbf Q\,dV=\oint_A\mathbf Q\cdot\mathbf n\,dA$ (Ch. 2 §2.12) · mean-value theorem for integrals (ch02 P85).

**We start from**

$$ \text{(3.35) with F = 1 and b = u:}\ \dfrac{d}{dt}\displaystyle\int_{\delta V}dV=\int_{\delta V}\dfrac{\partial 1}{\partial t}dV+\int_{A^*}\mathbf u\cdot\mathbf n\,dA $$

*In words:* the volume of a lump of fluid changes by what its surface sweeps.

---

**Step 1 of 5 — Simplify both sides.**

$$ \dfrac{D(\delta V)}{Dt}=\displaystyle\oint_{A^*}\mathbf u\cdot\mathbf n\,dA $$

- *Why we can do this:* ∫dV is the volume, and its rate following the fluid is D/Dt; ∂1/∂t = 0 removes the volume term.
- *In words:* A lump's volume changes by the net outward flow through its skin.

---

**Step 2 of 5 — Apply Gauss' theorem.**

$$ \displaystyle\oint_{A^*}\mathbf u\cdot\mathbf n\,dA=\int_{\delta V}\nabla\cdot\mathbf u\,dV $$

- *Why we can do this:* Divergence theorem (2.30), $\int_V\nabla\cdot\mathbf Q\,dV=\oint_A\mathbf Q\cdot\mathbf n\,dA$ (Ch. 2 §2.12) with Q = u.
- *In words:* Outflow through the skin = divergence summed inside.

---

**Step 3 of 5 — Mean-value theorem.**

$$ \displaystyle\int_{\delta V}\nabla\cdot\mathbf u\,dV=(\nabla\cdot\mathbf u)(\mathbf x^*)\,\delta V $$

- *Why we can do this:* For a continuous integrand the integral equals its value at some point x* inside times the volume (P85).
- *In words:* A small volume sees an average divergence.

---

**Step 4 of 5 — Divide by δV.**

$$ \dfrac{1}{\delta V}\dfrac{D(\delta V)}{Dt}=(\nabla\cdot\mathbf u)(\mathbf x^*) $$

- *Why we can do this:* δV ≠ 0; combine steps 1–3.
- *In words:* The fractional growth rate is the divergence somewhere inside.

---

**Step 5 of 5 — Shrink the volume to a point.**

$$ \dfrac{1}{\delta V}\dfrac{D(\delta V)}{Dt}=\dfrac{\partial u_i}{\partial x_i} $$

- *Why we can do this:* As δV → 0 around x, x* → x and continuity gives ∇·u(x); in index form this is (3.14), $\frac{1}{\delta V}\frac{D}{Dt}(\delta V)=\frac{\partial u_i}{\partial x_i}=S_{ii}$.
- *In words:* The same result as D11, now from the transport theorem.

---

**Result**

$$ \frac{1}{\delta V}\frac{D}{Dt}(\delta V)=\frac{\partial u_i}{\partial x_i}\ \text{(3.14)} $$

*In words:* divergence is the fractional growth rate of a small material volume, whichever route you take.

**What it means.** D11 (edge by edge) and D23 (whole surface) agree — the kinematic half of the continuity equation of Ch. 4 holds for blobs of any shape.

**Check it.** Units 1/s ✓. u = (x, y, z): 3 s⁻¹ ✓. The transport explainer's ellipse (u = (0.1x, 0.1y)) with a = 2 m, b = 1 m (area 2π m²): dA/dt = 0.2 × 6.283 = 1.257 m²/s ✓ (default a = 1, b = 0.6 m: 0.2 × 1.885 = 0.377 m²/s). `ch03.material_volume_rate` returns equal surface flux and ∫∇·u (to 1e-10, run in the C15 code cell below) ✓.

> ⚠️ **Common confusion (traps in `D23`):** Using b = 0 (then the surface term vanishes and the volume "does not change"). Dividing by δV after the limit instead of before.

#### 🧮 Derivation — For a small material volume the theorem gives back the material derivative, Eq. (3.5) `D24` (Eq. 3.5)

**What we want to show.** Show the book's second interpretation (Exercise 3.30) — that (3.35), $\frac{d}{dt}\int_{V^*}F\,dV=\int_{V^*}\frac{\partial F}{\partial t}dV+\int_{A^*}F\,\mathbf b\cdot\mathbf n\,dA$ "extends (3.5), $\frac{DF}{Dt}=\frac{\partial F}{\partial t}+\mathbf u\cdot\nabla F$ to finite volumes" — by deriving the material derivative *from* the transport theorem, and expose the extra term F∇·u that the book's wording hides (the step Ch. 4 uses to derive the continuity and momentum equations).

**Assumptions.** Material volume, b = u (start); F and u smooth (steps 1, 4–5); δV → 0 around a point x (steps 4–5).

**The plan.**

1. Turn the surface term into a volume integral and expand the divergence of a product.
2. Shrink the lump so the integrals become values times δV.
3. Split the left side with the product rule and remove the lump's swelling with (3.14), $\frac{1}{\delta V}\frac{D(\delta V)}{Dt}=\frac{\partial u_i}{\partial x_i}$.
4. What is left is the rate following the lump.

**Tools we use** (each explained before this point): (3.35), $\frac{d}{dt}\int_{V^*}F\,dV=\int_{V^*}\frac{\partial F}{\partial t}dV+\int_{A^*}F\,\mathbf b\cdot\mathbf n\,dA$ (D22) · Gauss' divergence theorem (Ch. 2 §2.12) · product rule for ∇·(Fu) (gloss in step 3) · mean-value theorem (ch02 P85) · product rule (ch01 P38) · (3.14), $\frac{1}{\delta V}\frac{D(\delta V)}{Dt}=\frac{\partial u_i}{\partial x_i}$ (C09, D23).

**We start from**

$$ \begin{array}{l}\text{(3.35) with b = u on a small material volume δV:} \\ \dfrac{d}{dt}\displaystyle\int_{\delta V}F\,dV=\int_{\delta V}\dfrac{\partial F}{\partial t}dV+\oint_{A^*}F\,\mathbf u\cdot\mathbf n\,dA\end{array} $$

*In words:* the amount of F carried by a lump of fluid changes in place and by what its moving skin sweeps.

---

**Step 1 of 8 — Gauss on the surface term.**

$$ \displaystyle\oint_{A^*}F\,\mathbf u\cdot\mathbf n\,dA=\int_{\delta V}\nabla\cdot(F\mathbf u)\,dV $$

- *Why we can do this:* Divergence theorem with the vector field Fu (Ch. 2 §2.12); we want the whole budget as one volume integral.
- *In words:* What the skin sweeps = the divergence of the flux of F inside.

---

**Step 2 of 8 — Combine the two volume integrals.**

$$ \dfrac{d}{dt}\displaystyle\int_{\delta V}F\,dV=\int_{\delta V}\Big[\dfrac{\partial F}{\partial t}+\nabla\cdot(F\mathbf u)\Big]dV $$

- *Why we can do this:* Both terms of (3.35), $\frac{d}{dt}\int_{V^*}F\,dV=\int_{V^*}\frac{\partial F}{\partial t}dV+\int_{A^*}F\,\mathbf b\cdot\mathbf n\,dA$ are now integrals over the same volume; add the integrands.
- *In words:* One integrand for the whole budget.

---

**Step 3 of 8 — Expand the divergence of a product.**

$$ \nabla\cdot(F\mathbf u)=\mathbf u\cdot\nabla F+F\,\nabla\cdot\mathbf u $$

- *Why we can do this:* Product rule in index form: $\partial(Fu_i)/\partial x_i=u_i\,\partial F/\partial x_i+F\,\partial u_i/\partial x_i$ (gloss).
- *In words:* F carried around + F spread out by the swelling flow.

---

**Step 4 of 8 — Shrink the right side.**

$$ \displaystyle\int_{\delta V}\Big[\dfrac{\partial F}{\partial t}+\mathbf u\cdot\nabla F+F\,\nabla\cdot\mathbf u\Big]dV\to\Big(\dfrac{\partial F}{\partial t}+\mathbf u\cdot\nabla F+F\,\nabla\cdot\mathbf u\Big)\delta V $$

- *Why we can do this:* Mean-value theorem for integrals (P85): the integral of a continuous integrand is its value at some point x* inside times δV, and x* → x as the lump shrinks round x. We want values at one point.
- *In words:* For a tiny lump the budget's right side is the integrand at x times the lump's volume.

---

**Step 5 of 8 — Shrink the left side.**

$$ \dfrac{D}{Dt}(F\,\delta V)=\Big(\dfrac{\partial F}{\partial t}+\mathbf u\cdot\nabla F+F\,\nabla\cdot\mathbf u\Big)\delta V $$

- *Why we can do this:* By the same theorem ∫F dV → FδV; because the volume is material (b = u), its time derivative follows the lump, which is what D/Dt means. Equate with the right side of step 2.
- *In words:* The budget of one tiny lump.

---

**Step 6 of 8 — Product rule on the left.**

$$ \dfrac{D}{Dt}(F\,\delta V)=\delta V\,\dfrac{DF}{Dt}+F\,\dfrac{D(\delta V)}{Dt} $$

- *Why we can do this:* Product rule (P38) for the two factors F (the lump's value) and δV (its volume), both followed with the lump.
- *In words:* The content changes because F changes and because the lump swells.

---

**Step 7 of 8 — Insert the swelling rate (3.14).**

$$ \dfrac{D}{Dt}(F\,\delta V)=\delta V\,\dfrac{DF}{Dt}+F\,(\nabla\cdot\mathbf u)\,\delta V $$

- *Why we can do this:* (3.14), $\frac{1}{\delta V}\frac{D(\delta V)}{Dt}=\frac{\partial u_i}{\partial x_i}$ (D23 derived it from this same theorem with F = 1). The book skips this.
- *In words:* The swelling part is F times the divergence.

---

**Step 8 of 8 — Equate steps 5 and 7, cancel.**

$$ \dfrac{DF}{Dt}=\dfrac{\partial F}{\partial t}+\mathbf u\cdot\nabla F $$

- *Why we can do this:* Both lines describe the same quantity; the term F∇·u δV appears in each and cancels; divide by δV ≠ 0. This is (3.5), $\frac{DF}{Dt}=\frac{\partial F}{\partial t}+\mathbf u\cdot\nabla F$.
- *In words:* The rate following a lump is the local rate plus the advective rate — the material derivative, recovered.

---

**Result**

$$ \begin{array}{l}\frac{d}{dt}\int_{\delta V}F\,dV=\int_{\delta V}\big[\frac{\partial F}{\partial t}+\mathbf u\cdot\nabla F+F\nabla\cdot\mathbf u\big]dV\ \text{and, for δV → 0,} \\ \frac{DF}{Dt}=\frac{\partial F}{\partial t}+\mathbf u\cdot\nabla F\ \text{(3.5)}\end{array} $$

*In words:* the transport theorem for a small lump = the material derivative + the effect of the lump's volume changing.

**What it means.** "(3.35), $\frac{d}{dt}\int_{V^*}F\,dV=\int_{V^*}\frac{\partial F}{\partial t}dV+\int_{A^*}F\,\mathbf b\cdot\mathbf n\,dA$ extends (3.5), $\frac{DF}{Dt}=\frac{\partial F}{\partial t}+\mathbf u\cdot\nabla F$ to finite volumes" hides a term: per unit volume the content of a lump changes at DF/Dt + F∇·u. Setting F = ρ, ρu, ρe gives the three conservation laws of Ch. 4; with F = ρ × (something per unit mass) the F∇·u term is exactly what the continuity equation absorbs.

**Check it.** Units [F]/s ✓. F = 1: steps 5 and 7 both give D(δV)/Dt = (∇·u)δV, i.e. D23 ✓. F = ρ with the lump's mass conserved: the left side of step 5 is 0, so $\frac{D\rho}{Dt}+\rho\nabla\cdot\mathbf u=0$ — Ch. 4's continuity equation (preview) ✓. Same result as D02, which reached (3.5), $\frac{DF}{Dt}=\frac{\partial F}{\partial t}+\mathbf u\cdot\nabla F$ from the chain rule instead ✓.

> ⚠️ **Common confusion (traps in `D24`):** Dropping F∇·u and concluding d/dt∫F dV = ∫DF/Dt dV (true only for incompressible flow). Using b = 0 instead of b = u. Treating F in D(FδV)/Dt as the field at a fixed point rather than the value carried by the lump.

#### ✏️ Tiny example: a balloon growing in a warming room

Sphere of radius R = 1 m growing at Ṙ = 0.1 m/s; F = t (a "heat content per volume" rising uniformly, ∂F/∂t = 1 per
second) evaluated at t = 1 s.

1. Volume term: $\int_{V^*}\partial F/\partial t\,dV=1\times\tfrac43\pi R^3=4.189$.
2. Surface term: on the sphere $\mathbf b\cdot\mathbf n=\dot R=0.1$ m/s and F = 1: $\oint F\,\mathbf b\cdot\mathbf n\,dA=1\times0.1\times4\pi R^2=1.257$.
3. Total 5.445.
4. Direct: $\int F\,dV=t\cdot\tfrac43\pi R(t)^3$, whose derivative is $\tfrac43\pi R^3+t\cdot4\pi R^2\dot R=4.189+1.257=5.445$ ✓.

In [ ]:
cv = ch03.GrowingSphere(0.9, 0.1)                     # R(t) = 0.9 + 0.1 t [m], so R = 1.0 m at t = 1 s (b·n = Ṙ on the surface)
F = lambda x, t: t + 0*x[0]                           # F(x, t) = t: uniform, rising at 1 per second
dF = lambda x, t: 1.0 + 0*x[0]                        # ∂F/∂t
print(ch03.reynolds_transport(F, dF, cv, 1.0))        # (3.35): volume term, surface term, total
print(ch03.rtt_check(F, dF, cv, 1.0))                 # finite difference of ∫F dV (left of (3.31)) vs the RTT total
Fw, dFw = ch03.rtt_field("warming", dim=1)            # 1-D Leibniz with the explainer's default numbers
print(ch03.leibniz_terms(Fw, dFw, 1.0, 3.0, -0.2, 0.4, 0.0))   # a = 1, b = 3, ȧ = −0.2, ḃ = 0.4 at t = 0
d = ch03.leibniz_example(2.0)                         # the D21 check case F = x²t, a = t, b = t² at t = 2 s
print({k: round(d[k], 3) for k in ("interior", "upper", "lower", "total", "exact")})
F2, dF2 = ch03.rtt_field("ramp", dim=2)               # F = 1 + 0.5 x1 in the plane
print(ch03.rtt_ellipse_2d(F2, dF2, 2.0, 1.0, 0.2, 0.1, (0.0, 0.0), (0.3, 0.0), 0.0))   # a translating, growing ellipse
print(ch03.material_volume_rate(lambda x, t: 0.1*x, ch03.GrowingSphere(1.0, 0.1), 0.0))   # D23: ∮u·n dA = ∫∇·u dV for u = 0.1 x

**What does the code above do?**

1. `GrowingSphere` is a control volume that knows its nodes, its surface normal and its surface velocity b at any t.
2. `reynolds_transport` evaluates the two terms of
   $\frac{d}{dt}\int_{V^*}F\,dV=\int_{V^*}\frac{\partial F}{\partial t}dV+\int_{A^*}F\,\mathbf b\cdot\mathbf n\,dA$ *(3.35)*
   by quadrature: (4.1888, 1.2566, 5.4454).
3. `rtt_check` gets the same total from a finite difference of the volume integral itself.
4. Leibniz with the transport explainer's default numbers: 0.6 + 1.0 − (−0.3) = 1.9 (the lower term is $\dot aF(a)$ and is subtracted).
5. The D21 check case: 18.667 + 128 − 8 = 138.667 = the exact derivative.
6. A translating, growing ellipse in a ramp field (the explainer's 2-D mode): $\pi(\dot ab+a\dot b)(1+0.5c_x)+\pi ab\cdot0.5\dot c_x=1.2566+0.9425$.
7. D23 on a sphere: for $\mathbf u=0.1\,\mathbf x$ the outward flux $\oint\mathbf u\cdot\mathbf n\,dA$ and $\int\nabla\cdot\mathbf u\,dV$ are both $0.3\times\tfrac43\pi=1.2566$ m³/s.

In [ ]:
nr, nth, nph = (16, 16, 32) if not FAST else (12, 12, 24)    # midpoint grid in r, θ, φ (Ch. 2 P83)

def sphere_nodes(R):                                  # midpoint nodes and weights of a ball of radius R
    r = (np.arange(nr) + 0.5)*R/nr; th = (np.arange(nth) + 0.5)*np.pi/nth; ph = (np.arange(nph) + 0.5)*2*np.pi/nph
    Rr, Th, Ph = np.meshgrid(r, th, ph, indexing="ij")
    w = (Rr**2*np.sin(Th)*(R/nr)*(np.pi/nth)*(2*np.pi/nph)).ravel()          # r² sin θ Δr Δθ Δφ
    X = np.stack([Rr*np.sin(Th)*np.cos(Ph), Rr*np.sin(Th)*np.sin(Ph), Rr*np.cos(Th)]).reshape(3, -1)
    return X, w

R, Rd, t0 = 1.0, 0.1, 1.0                             # radius at t0, growth rate, time
X, w = sphere_nodes(R)
vol = np.sum(dF(X, t0)*w)                             # ∫ ∂F/∂t dV as a midpoint sum
Th, Ph = np.meshgrid((np.arange(nth) + 0.5)*np.pi/nth, (np.arange(nph) + 0.5)*2*np.pi/nph, indexing="ij")
wA = (R**2*np.sin(Th)*(np.pi/nth)*(2*np.pi/nph)).ravel()                       # R² sin θ Δθ Δφ
Xs = R*np.stack([np.sin(Th)*np.cos(Ph), np.sin(Th)*np.sin(Ph), np.cos(Th)]).reshape(3, -1)
surf = np.sum(F(Xs, t0)*Rd*wA)                        # ∮ F b·n dA with b·n = Ṙ
assert np.allclose([vol, surf], ch03.reynolds_transport(F, dF, cv, 1.0)[:2], rtol=5e-3)   # midpoint ≈ library
I = lambda tt: np.sum(F(sphere_nodes(0.9 + 0.1*tt)[0], tt)*sphere_nodes(0.9 + 0.1*tt)[1])  # ∫_{V*(t)} F dV
dtt = 1e-3
fd = (I(t0 + dtt) - I(t0 - dtt))/(2*dtt)              # d/dt of the integral, by a central difference
assert np.allclose(fd, vol + surf, rtol=5e-3)         # = volume term + surface term
print(f"midpoint sums: volume {vol:.4f} + surface {surf:.4f} = {vol + surf:.4f};  d/dt of the sum {fd:.4f}\n", end="")   # one write: no stray empty output

Midpoint sums on a spherical grid converge like $h^2$; with 16 × 16 × 32 cells they agree with the library to about 10⁻³.

In [ ]:
F3 = lambda x, t: x[0]**2*t                           # a field that varies in space: F = x1² t
dF3 = lambda x, t: x[0]**2                            # ∂F/∂t
dts = np.logspace(-4, -1, 16 if not FAST else 10)     # time steps Δt [s]
rows = {d: ch03.swept_terms_sphere(1.0, 0.1, F3, dF3, 1.0, d) for d in dts}   # the four terms of (3.32), computed once
fig = slider_figure(lambda d: {"terms T1…T4": ([1, 2, 3, 4], [abs(rows[d][k]) for k in ("T1", "T2", "T3", "T4")])},
                    "Δt", dts, unit="s", xlabel="term of (3.32)", ylabel="size [F·m³]", xrange=[0.5, 4.5],
                    modes={"terms T1…T4": "lines+markers"}, height=420,
                    title="Eq. (3.32) for a growing sphere: T4 falls twice as fast as T2 and T3")   # one trace per step
for tr, d in zip(fig.data, dts):                      # one trace per slider step: put T4/T3 into its name
    tr.name = f"T4/T3 = {rows[d]['T4']/rows[d]['T3']:.1e}"; tr.marker.size = 12; tr.line.color = COLORS["blue"]
fig.update_yaxes(type="log", range=[-11, 1])          # log scale, fixed (10⁻¹¹ … 10)
fig.update_xaxes(tickvals=[1, 2, 3, 4], ticktext=["T1 ∫F", "T2 ∫Δt ∂F/∂t", "T3 ∫_ΔV F", "T4 ∫_ΔV Δt ∂F/∂t"])   # term names
fig.update_layout(showlegend=True)                    # the legend carries T4/T3
fig.show()                                            # draw it

**What does the code above do?**

1. `swept_terms_sphere` evaluates the four integrals of
   $\int_{V^*(t+\Delta t)}F(t+\Delta t)\,dV\cong\int_{V^*}F\,dV+\int_{V^*}\Delta t\frac{\partial F}{\partial t}dV+\int_{\Delta V}F\,dV+\int_{\Delta V}\Delta t\frac{\partial F}{\partial t}dV$ *(3.32)* for a sphere of radius 1.1 m at t = 1 s growing at
   0.1 m/s, for 16 time steps between 10⁻⁴ and 10⁻¹ s.
2. The slider moves Δt; the legend shows the ratio T4/T3, which shrinks in proportion to Δt.

In [ ]:
T2 = np.array([rows[d]["T2"] for d in dts]); T3 = np.array([rows[d]["T3"] for d in dts])   # the kept terms [F·m³]
T4 = np.array([rows[d]["T4"] for d in dts]); sl = np.abs([rows[d]["sliver_error"] for d in dts])   # the dropped pieces
fig, ax = plt.subplots(figsize=(7, 4))                # log–log axes
for y, col, lab in ((T2, COLORS["blue"], "T2 (kept)"), (T3, COLORS["orange"], "T3 (kept)"),
                    (T4, COLORS["rose"], "T4 (dropped)"), (sl, COLORS["accent"], "|T3 − surface sliver| (3.34)")):
    ax.loglog(dts, y, "o-", color=col, ms=4, label=f"{lab}: slope {observed_order(dts, y):.2f}")   # slope by least squares
ax.set_xlabel(r"$\Delta t$ [s]"); ax.set_ylabel("size [F·m³]"); ax.legend(fontsize=8)            # axes with units
ax.set_title("Orders of smallness: first-order terms survive division by Δt, second-order ones do not")
savefig(fig, "ch03", "rtt_orders"); plt.show()        # save to outputs/ch03 and draw

**What you see.** Four straight lines on log–log axes: two of slope 1 (blue, orange), two of slope 2 (rose, purple).

**How to read it.** After dividing by Δt, the slope-1 terms tend to finite rates (the volume and surface terms), while the slope-2
terms — T4 and the error of replacing the sliver by a surface integral — tend to zero. That is exactly why (3.33),
$\frac{d}{dt}\int_{V^*}F\,dV=\lim\frac{1}{\Delta t}\{\int_{V^*}\Delta t\,\partial F/\partial t\,dV+\int_{\Delta V}F\,dV\}$,
keeps two terms and drops the rest.

**What would change if…** …the sphere grew ten times faster: T3 and T4 would both grow ×10 (both ∝ ΔV), but T4/T3 would still be ∝ Δt.

📝 **Note.** **Ex. 3.2 — a growing cone** `N54` — A right circular cone of fixed height h has base radius r(t) growing at ṙ. Directly, $V=\tfrac13\pi hr^2$ gives
$dV/dt=\tfrac23\pi hr_o\dot r$. By (3.35) with F = 1 and the cone as V*, only the sloping side contributes: a side point
at height z moves outward at $(z/h)\dot r\,\mathbf e_R$, the outward normal is $\mathbf n=\mathbf e_R\cos\theta-\mathbf e_z\sin\theta$
(θ = the cone's half-angle, $\tan\theta=r_o/h$ — a fourth θ in this chapter), and the slanted area element is
$dA=z\tan\theta\,d\varphi\,dz/\cos\theta$. ⚠️ Two print slips taught corrected: the base's points move radially in its own
plane, so it is **b·n = 0** there (not b = 0), which is all the theorem needs; the "[?]" in the printed integrand is just
the factor z.

$$ \frac{dV}{dt}=\int_{z=0}^{h}\int_{\varphi=0}^{2\pi}\frac zh\dot r\,\mathbf e_R\cdot(\mathbf e_R\cos\theta-\mathbf e_z\sin\theta)\,z\tan\theta\,d\varphi\frac{dz}{\cos\theta}=\frac{2\pi\dot r\tan\theta}{h}\int_0^hz^2dz=\tfrac23\pi h^2\dot r\tan\theta=\tfrac23\pi hr_o\dot r  $$

In [ ]:
print(ch03.example_3_2(1.0, 0.5, 0.1))               # h = 1 m, r_o = 0.5 m, ṙ = 0.1 m/s: dV/dt three ways [m³/s]

**What does the code above do?**

The direct derivative, the book's surface integral by `dblquad` (P87), the general quadrature on `GrowingCone` and the closed form all give 0.10472 m³/s.

📝 **Note.** **Fig. 3.18, our 2-D drawing** `N55` — a deforming, translating ellipse over the ramp field F = 1 + 0.5x₁.

In [ ]:
from scripts.ch03_drawings import rtt_blob_figure             # drawing helper; the numbers come from ch03.rtt_ellipse_2d
fig, terms, fd = rtt_blob_figure(a=2.0, b=1.0, adot=0.2, bdot=-0.1, cdot=(0.3, 0.0), F_name="ramp", dt=0.3)   # ellipse 2 × 1 m
fig.axes[0].set_title("swept band: blue where b·n > 0, rose where b·n < 0")   # a shorter title that fits the panel
print(f"volume term {terms[0]:.4f} + surface term {terms[1]:.4f} = {terms[2]:.4f};  measured d/dt of the integral {fd:.4f}")   # [F·m²/s]
savefig(fig, "ch03", "fig3_18_rtt"); plt.show()               # save to outputs/ch03 and draw

**What you see.** Left: an ellipse (solid) and its position a moment later (dashed) over a grey ramp, orange b arrows on the
boundary, blue slivers on the right where the boundary advances and rose slivers at the top and bottom where it
retreats. Right: an orange surface bar and a purple total bar, both reaching the black diamond; the blue volume bar
has zero height.

**How to read it.** The rose slivers are *negative* volume — the same formula counts them with the right sign. Here F does not
depend on time, so the volume term is zero and the whole change comes from the moving surface; the sum of the bars is
the measured rate.

**What would change if…** …b = 0 (a fixed box): the slivers and the orange bar would vanish, and d/dt could pass inside the integral.

#### 🎮 Interactive: What changes inside a moving box?

**Why interactive:** the theorem is a budget whose terms change sign with the direction the walls move; moving the walls yourself and watching the slivers turn blue or rose shows why one signed surface term covers all cases.
A moving boundary sweeps a band whose colour shows the sign of b·n; bars for the volume and surface terms of
$\frac{d}{dt}\int_{V^*}F\,dV=\int_{V^*}\frac{\partial F}{\partial t}dV+\int_{A^*}F\,\mathbf b\cdot\mathbf n\,dA$ *(Eq. 3.35)*
add up to the measured rate of change of ∫F, and a Δt slider shows the dropped second-order term vanish. Modes: Leibniz
in 1-D, a deforming ellipse in 2-D, and the growing cone of Ex. 3.2.

**What to try:**
- Preset 'fixed volume (b = 0)': the orange bar disappears.
- Preset 'rigid translation in a uniform F': the two terms cancel exactly — a uniform F carried along does not change.
- Make one side retreat: its sliver turns rose and subtracts.
- Derivation tab, D22: at the 'drop second order' step, scrub Δt on the log–log view.

In [ ]:
show_viz("ch03", "reynolds_transport_cv")   # full-width explainer; ⤢ Full screen for more room

**What would change if…** …F were the density ρ and V* a material volume (b = u)? The left side is the rate of change of the lump's mass, which
is zero; D24's steps then give $\frac{D\rho}{Dt}+\rho\nabla\cdot\mathbf u=0$ — the continuity equation that opens
Chapter 4. With F = ρu you get Newton's law for the lump; with F = ρ(e + u²/2), the energy equation.

*↪ S01 · Practise on the book's Exercises 3.1–3.30 (curvilinear operators, flow lines, the Galilean proof, strain and
rotation of simple flows, circulation, transport-theorem checks). The derivations the text leaves to Exercises 3.3,
3.12, 3.17–3.20, 3.23, 3.26, 3.28 and 3.30 are written out above in our own words (D03, D06, D10, D11, D13, D17, N37,
D20, D23, D24).*

*↪ S02 · Further reading: Van Dyke's *An Album of Fluid Motion* and Samimy et al.'s *A Gallery of Fluid Motion* for
photographs of streak and path lines; Riley, Hobson & Bence for Leibniz's rule; Thompson (1972) for the geometric
transport-theorem argument (all cited in the book's literature list).*

---

## ✅ What should have clicked

- **C01** A particle is a label; a field is a function of (x, t); $F[\mathbf r(t;\mathbf r_o,t_o),t]=F(\mathbf x,t)$ at $\mathbf x=\mathbf r$ *(3.2)* translates between them.
- **C02** $\frac{DF}{Dt}=\frac{\partial F}{\partial t}+\mathbf u\cdot\nabla F$ *(3.5)*: what a moving parcel feels = what a fixed probe sees + what it meets by moving.
- **C03** A streamline is tangent to $\mathbf u$ at one frozen instant: $dx/u=dy/v=dz/w$ *(3.7)*.
- **C04** A path line follows one particle, $d\mathbf r/dt=\mathbf u(\mathbf r,t)$ *(3.8)*; a streak line collects everything from one port; in unsteady flow the three lines differ.
- **C05** The acceleration $\frac{\partial\mathbf u}{\partial t}+(\mathbf u\cdot\nabla)\mathbf u$ is the same for every observer moving at constant velocity *(3.9)*; only its split is not.
- **C06** $du_i=(\partial u_i/\partial x_j)\,dx_j$ *(3.10)*: the neighbourhood of a point moves linearly.
- **C07** The diagonal of S is the stretching rate per length; along any direction $\mathbf n$ it is $\mathbf n\cdot\mathbf S\cdot\mathbf n$.
- **C08** The off-diagonal $S_{12}$ is half the closing rate of a right angle; rigid motion has S = 0.
- **C09** $\frac1{\delta V}\frac{D(\delta V)}{Dt}=\frac{\partial u_i}{\partial x_i}=S_{ii}$ *(3.14)*: divergence is the swelling rate.
- **C10** A fluid element spins at ½ω — the average of any perpendicular pair of threads — and the spin depends on the observer's rotation ($\omega'_z=\omega_z-2\Omega$).
- **C11** $du_i=S_{ij}dx_j+\tfrac12(\boldsymbol\omega\times d\mathbf x)_i$ *(3.19)*: deformation plus rigid rotation.
- **C12** S stretches a small sphere into an ellipsoid on its principal axes, $d\bar u_\alpha=\bar S_{\alpha\alpha}d\bar x_\alpha$ *(3.21)*, at the first instant.
- **C13** $\omega_z=\frac1r\frac{\partial}{\partial r}(ru_\theta)-\frac1r\frac{\partial u_r}{\partial\theta}$ *(3.23)*: solid-body rotation spins every element (2ω₀); the line vortex spins none (except its axis).
- **C14** Real vortices have a solid-body core and an irrotational outside; the peak wind is at σ (Rankine, $u_\theta=\Gamma r/2\pi\sigma^2$ inside, *(3.28)*) or 1.1209σ (Gaussian, $u_\theta=\frac{\Gamma}{2\pi r}(1-e^{-r^2/\sigma^2})$ *(3.29)*).
- **C15** $\frac{d}{dt}\int_{V^*}F\,dV=\int_{V^*}\frac{\partial F}{\partial t}dV+\int_{A^*}F\,\mathbf b\cdot\mathbf n\,dA$ *(3.35)*: change in place + what the moving walls sweep (signed).

**What later chapters build on**

- Ch. 4: the transport theorem *(3.35)* + $\frac{DF}{Dt}=\frac{\partial F}{\partial t}+\mathbf u\cdot\nabla F$ *(3.5)* → continuity, Cauchy's equation, energy; S → the Newtonian stress law; rotating frames ($\omega'=\omega-2\Omega$) → Coriolis.
- Ch. 5: vorticity, circulation $\Gamma=\oint\mathbf u\cdot d\mathbf s$ *(3.18)*, vortex stretching (S along ω), the Lamb–Oseen vortex.
- Ch. 6: irrotational flow $\boldsymbol\omega=0$ *(3.17)*, the point vortex $u_\theta=B/r$ *(3.25)*, the cylinder of Fig. 3.2.
- Ch. 7: particle orbits via $d\mathbf r/dt=\mathbf u(\mathbf r,t)$ *(3.8)*; wave frames via Galilean invariance, $\frac{\partial\mathbf u}{\partial t}+(\mathbf u\cdot\nabla)\mathbf u=\frac{\partial\mathbf u'}{\partial t'}+(\mathbf u'\cdot\nabla')\mathbf u'$ *(3.9)*.
- Ch. 13: absolute vs relative vorticity, frontogenesis on principal strain axes, cyclone profiles.

**Not taught here (and where to find it)**
- Curvilinear forms of ∇, ∇², (u·∇)u (Appendix B; used from Ch. 4 on).
- Solutions for the velocity potential (Ch. 6).
- Viscous spreading of the Gaussian vortex (Ch. 5, 8).